In [ ]:
# =================================
# Rotational Momentum Functions
# =================================

import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime
from typing import Dict, Tuple, List, Optional
from tqdm.auto import tqdm
from datetime import datetime, timedelta
import os
from typing import Union, List, Dict, Tuple, Any
from itertools import product, combinations
import ace_tools_open as tools
from __future__ import annotations

In [ ]:
"""
rotational_engine.py
====================
Refactoring del motore rotazionale.

ARCHITETTURA A 3 LAYER:
  Layer 1 – Pure functions   : compute_rebal_dates, compute_scores, select_tickers, build_weight_matrix
  Layer 2 – Orchestrator     : run_rotational_engine  →  RotationalResult (dataclass)
  Layer 3 – VBT Bridge       : build_portfolio, build_portfolio_from_wfo_summary

PRINCIPI:
  - Arità STABILE: niente più tuple a lunghezza variabile.
  - Carry-forward ESPLICITO: colonna `carried: bool` nel DataFrame selections.
  - Single source of truth per rebal_dates: get_rebalance_dates() (già esistente, importata).
  - Nessun doppio carry-forward.
  - Weight-shift difeso da asserzione su trading-only index.
  - bottom_tickers sperimentale → rimosso dal core, disponibile come utility separata.

COMPATIBILITÀ COI NOTEBOOK:
  - build_rotational_portfolios_from_wfo_result  → alias di build_portfolio_from_wfo_summary
  - collect_selections_from_summary              → alias di collect_wfo_selections
  - build_rotational_portfolios_from_selections  → alias di build_portfolio_from_selections
"""

from __future__ import annotations

import warnings
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd


# ─────────────────────────────────────────────────────────────────────────────
# UTILITIES  (normalizzazione indice, già presenti nel progetto)
# ─────────────────────────────────────────────────────────────────────────────

def _norm_dt_index(idx) -> pd.DatetimeIndex:
    return pd.to_datetime(idx).normalize()


# ─────────────────────────────────────────────────────────────────────────────
# LAYER 1 – PURE FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def compute_rebal_dates(
    trading_index: pd.DatetimeIndex,
    freq: str,
    grace_days_map: dict | None = None,
) -> pd.DatetimeIndex:
    """
    Single source of truth per le date di ribilanciamento.

    Restituisce l'ultimo trading-day per ciascun periodo completo.
    Un periodo è "incompleto" (e quindi escluso) se la distanza tra l'ultima
    data disponibile e la fine calendario del periodo supera `grace_days`.

    Parameters
    ----------
    trading_index : pd.DatetimeIndex
        Indice dei giorni di trading (solo trading days).
    freq : str
        Frequenza: 'ME'/'M', 'QE'/'Q', 'YE'/'Y', 'W'/'W-FRI', 'D'.
    grace_days_map : dict, optional
        Override dei grace-days per frequenza.
        Default: {'ME': 3, 'QE': 5, 'YE': 5, 'W': 0}

    Returns
    -------
    pd.DatetimeIndex  (sorted, unique, tz-naive, normalized)
    """
    idx = pd.DatetimeIndex(trading_index).normalize().sort_values().unique()
    if len(idx) == 0:
        return pd.DatetimeIndex([])

    f = str(freq).upper().strip()

    if grace_days_map is None:
        grace_days_map = {"ME": 3, "QE": 5, "YE": 5, "W": 0}

    last_date = idx.max()

    # ── mapping frequenza → (period_key, grace) ──────────────────────────────
    if f in {"ME", "M", "MONTH", "MONTHLY"}:
        period_key, grace = "M", grace_days_map.get("ME", 3)
    elif f in {"QE", "Q", "QUARTER", "QUARTERLY"}:
        period_key, grace = "Q", grace_days_map.get("QE", 5)
    elif f in {"YE", "Y", "A", "ANNUAL", "YEARLY"}:
        period_key, grace = "Y", grace_days_map.get("YE", 5)
    elif f == "W":
        period_key, grace = "W-FRI", grace_days_map.get("W", 0)
    elif f.startswith("W-"):
        period_key, grace = f, grace_days_map.get("W", 0)
    elif f in {"D", "DAY", "DAILY"}:
        return idx
    else:
        raise ValueError(f"compute_rebal_dates: frequenza non supportata '{freq}'")

    periods = idx.to_period(period_key)
    dates = (
        pd.Series(idx, index=periods)
        .groupby(level=0)
        .max()
        .values
    )
    dates = pd.DatetimeIndex(dates).normalize().sort_values().unique()

    if len(dates) == 0:
        return dates

    # ── rimuovi ultimo periodo se incompleto ──────────────────────────────────
    last_rebal = dates.max()
    if last_rebal == last_date:
        last_period = last_date.to_period(period_key)
        period_end = pd.Timestamp(last_period.end_time).normalize()
        gap = (period_end - last_date).days
        if gap > int(grace):
            dates = dates[:-1]

    return pd.DatetimeIndex(dates).unique().sort_values()


# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class ScoreParams:
    """Parametri per il calcolo degli score di selezione."""
    momentum_lookback_days: int = 126
    riskparity_lookback_days: int = 20
    momentum_weight: float = 0.7
    use_acceleration: bool = False
    ema_span: int = 200          # usato solo se filter_ema=True (in SelectionParams)

    def __post_init__(self):
        if not 0.0 <= self.momentum_weight <= 1.0:
            raise ValueError("momentum_weight deve essere in [0, 1]")


@dataclass
class SelectionParams:
    """Parametri per i filtri di selezione dei ticker."""
    n_top: int = 5
    filter_ema: bool = False
    filter_volatility: bool = False
    filter_min_momentum: bool = False
    volatility_quantile: float = 0.75
    min_momentum_threshold: float = 1.0


@dataclass
class EngineParams:
    """Tutti i parametri del motore rotazionale."""
    score: ScoreParams = field(default_factory=ScoreParams)
    selection: SelectionParams = field(default_factory=SelectionParams)
    rebalance_frequency: str = "ME"
    init_cash: float = 100_000

    @classmethod
    def from_dict(cls, d: dict) -> "EngineParams":
        """Costruisce EngineParams da un dizionario (es. riga di summary_df)."""
        sp = ScoreParams(
            momentum_lookback_days=int(d.get("momentum_lookback_days", 126)),
            riskparity_lookback_days=int(d.get("riskparity_lookback_days", 20)),
            momentum_weight=float(d.get("momentum_weight", 0.7)),
            use_acceleration=bool(d.get("use_acceleration", False)),
            ema_span=int(d.get("ema_span", 200)),
        )
        selp = SelectionParams(
            n_top=int(d.get("n_top", 5)),
            filter_ema=bool(d.get("filter_ema", False)),
            filter_volatility=bool(d.get("filter_volatility", False)),
            filter_min_momentum=bool(d.get("filter_min_momentum", False)),
            volatility_quantile=float(d.get("volatility_quantile", 0.75)),
            min_momentum_threshold=float(d.get("min_momentum_threshold", 1.0)),
        )
        return cls(
            score=sp,
            selection=selp,
            rebalance_frequency=str(d.get("rebalance_frequency", "ME")),
        )


# ─────────────────────────────────────────────────────────────────────────────

def _precompute_indicators(
    prices: pd.DataFrame,
    params: "EngineParams",
    vol_cache: dict | None = None,
) -> dict:
    """
    Precomputa tutti gli indicatori necessari al loop di selezione.
    Riceve l'intero EngineParams per accedere sia a ScoreParams che a SelectionParams.
    Ritorna un dizionario di DataFrame allineati all'indice di prices.

    Non ha side effects; è testabile indipendentemente.
    """
    sp   = params.score
    selp = params.selection

    rets = prices.pct_change().fillna(0.0)

    mom_df = prices / prices.shift(sp.momentum_lookback_days)

    lb = sp.riskparity_lookback_days
    if vol_cache is not None and lb in vol_cache:
        vol_df = vol_cache[lb].reindex(rets.index)
    else:
        vol_df = rets.rolling(lb, min_periods=1).std(ddof=0)

    inv_vol_df = 1.0 / vol_df.replace(0.0, np.nan)

    rank_mom  = mom_df.rank(pct=True, axis=1, na_option="bottom")
    rank_ivol = inv_vol_df.rank(pct=True, axis=1, na_option="bottom")

    result = {
        "mom":       mom_df,
        "vol":       vol_df,
        "inv_vol":   inv_vol_df,
        "rank_mom":  rank_mom,
        "rank_ivol": rank_ivol,
    }

    if selp.filter_ema:
        result["ema"] = prices.ewm(
            span=sp.ema_span, adjust=False, min_periods=1
        ).mean()

    if sp.use_acceleration:
        _SMOOTH, _SHIFT, _WEIGHT = 5, 10, 0.20
        mom_sm = mom_df.ewm(span=_SMOOTH, adjust=False, min_periods=1).mean()
        accel  = mom_sm - mom_sm.shift(_SHIFT)
        result["accel"]         = accel
        result["rank_accel"]    = accel.rank(pct=True, axis=1, na_option="bottom")
        result["_accel_weight"] = _WEIGHT

    return result


# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class _SelectionResult:
    """Risultato della selezione per una singola data di ribilanciamento."""
    date: pd.Timestamp
    tickers: list[str]          # lista dei ticker selezionati (può essere vuota)
    carried: bool               # True se è un carry-forward di una selezione precedente
    score: pd.Series            # score completo (tutti i ticker, NaN se filtrati)
    n_passed_filters: int       # quanti ticker hanno passato tutti i filtri


def _select_at_date(
    date: pd.Timestamp,
    prices: pd.DataFrame,
    indicators: dict,
    score_params: ScoreParams,
    sel_params: SelectionParams,
    prev_selection: list[str] | None,
) -> _SelectionResult:
    """
    Calcola la selezione per una singola rebal_date.

    - Applica filtri (EMA, volatilità, min-momentum, accelerazione).
    - Calcola combo score.
    - Seleziona top-N.
    - Se selezione vuota E prev_selection disponibile → carry-forward (carried=True).
    - Se selezione vuota E nessun precedente → tickers=[], carried=False.

    Questa funzione è PURA rispetto allo stato: prev_selection è input esplicito.
    """
    d = pd.Timestamp(date).normalize()

    mom       = indicators["mom"].loc[d]
    inv_vol   = indicators["inv_vol"].loc[d]
    rank_mom  = indicators["rank_mom"].loc[d]
    rank_ivol = indicators["rank_ivol"].loc[d]

    # mask base: entrambi gli indicatori presenti
    mask = mom.notna() & inv_vol.notna()

    if sel_params.filter_ema:
        ema = indicators["ema"].loc[d]
        mask &= prices.loc[d] > ema

    if sel_params.filter_volatility:
        vol = indicators["vol"].loc[d]
        q   = vol.quantile(sel_params.volatility_quantile)
        mask &= vol < q

    if sel_params.filter_min_momentum:
        mask &= mom > sel_params.min_momentum_threshold

    # combo score
    w = score_params.momentum_weight
    combo = w * rank_mom + (1.0 - w) * rank_ivol

    if score_params.use_acceleration:
        accel_today = indicators["accel"].loc[d]
        mask &= accel_today > 0
        combo = combo + indicators.get("_accel_weight", 0.20) * indicators["rank_accel"].loc[d]

    combo_masked = combo.where(mask)
    n_passed = int(mask.sum())

    # selezione top-N
    valid = combo_masked.dropna()
    if not valid.empty:
        top_tickers = list(valid.nlargest(sel_params.n_top).index)
        carried = False
    elif prev_selection:
        top_tickers = list(prev_selection)
        carried = True
    else:
        top_tickers = []
        carried = False

    return _SelectionResult(
        date=d,
        tickers=top_tickers,
        carried=carried,
        score=combo_masked.sort_values(ascending=False),
        n_passed_filters=n_passed,
    )


# ─────────────────────────────────────────────────────────────────────────────

def build_weight_matrix(
    selections: pd.DataFrame,
    prices_idx: pd.DatetimeIndex,
) -> pd.DataFrame:
    """
    Costruisce la matrice dei pesi equal-weight da un DataFrame di selezioni,
    allineata all'intero indice dei prezzi e con shift(1) per evitare look-ahead.

    Parameters
    ----------
    selections : pd.DataFrame
        Index  : rebal_dates (pd.DatetimeIndex)
        Columns: 'tickers' (list[str]), + altre colonne ignorate
        Deve contenere almeno la colonna 'tickers'.
    prices_idx : pd.DatetimeIndex
        Indice completo dei prezzi (trading days only).

    Returns
    -------
    pd.DataFrame  shape=(len(prices_idx), n_assets)
        Pesi equal-weight, shiftati di 1 giorno (ordine eseguito il giorno dopo).

    Raises
    ------
    ValueError
        Se prices_idx ha buchi > 4 giorni consecutivi (segnale che non è trading-only).
    """
    prices_idx = pd.DatetimeIndex(prices_idx).normalize().sort_values()

    # ── guardrail: verifica che sia un indice trading-only ────────────────────
    if len(prices_idx) > 1:
        gaps = pd.Series(prices_idx).diff().dt.days.dropna()
        max_gap = gaps.max()
        if max_gap > 7:
            warnings.warn(
                f"build_weight_matrix: gap massimo di {max_gap} giorni in prices_idx. "
                "Gap > 7 giorni potrebbe indicare dati non trading-only (es. calendario errato).",
                stacklevel=2,
            )

    # ── ricava l'universo di asset da tutte le selezioni ─────────────────────
    all_tickers: set[str] = set()
    for tlist in selections["tickers"]:
        if isinstance(tlist, (list, tuple, set)):
            all_tickers.update(tlist)
    cols = sorted(all_tickers)

    if not cols:
        return pd.DataFrame(0.0, index=prices_idx, columns=[])

    # ── costruzione pesi sulle rebal_dates ───────────────────────────────────
    w_sparse = pd.DataFrame(0.0, index=selections.index, columns=cols)
    for d, row in selections.iterrows():
        tlist = row.get("tickers", [])
        if not isinstance(tlist, (list, tuple, set)) or len(tlist) == 0:
            continue
        valid = [t for t in tlist if t in cols]
        if valid:
            w_sparse.loc[d, valid] = 1.0 / len(valid)

    # ── espandi all'intero indice, poi shift(1) ───────────────────────────────
    w_full = w_sparse.reindex(prices_idx).ffill().fillna(0.0)
    w_shifted = w_full.shift(1).ffill().fillna(0.0)

    return w_shifted


# ─────────────────────────────────────────────────────────────────────────────
# LAYER 2 – ORCHESTRATOR
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class RotationalResult:
    """
    Output strutturato di run_rotational_engine.

    Attributes
    ----------
    selections : pd.DataFrame
        Una riga per ogni rebal_date.
        Colonne: 'tickers' (list[str]), 'carried' (bool), 'n_passed_filters' (int).
        Index name: 'rebal_date'.
    weights : pd.DataFrame
        Pesi equal-weight sull'intero indice dei prezzi, shiftati di 1 giorno.
    rankings : pd.DataFrame
        Combo-score per tutti i ticker, una riga per rebal_date.
    rebal_dates : pd.DatetimeIndex
        Date di ribilanciamento effettive usate.
    params : EngineParams
        Parametri usati per questa run.
    """
    selections: pd.DataFrame
    weights: pd.DataFrame
    rankings: pd.DataFrame
    rebal_dates: pd.DatetimeIndex
    params: EngineParams


def run_rotational_engine(
    prices: pd.DataFrame,
    params: EngineParams,
    vol_cache: dict | None = None,
    start_date: str | pd.Timestamp | None = None,
    debug: bool = False,
) -> RotationalResult:
    """
    Cuore del motore rotazionale. Pura orchestrazione del Layer 1.

    Il carry-forward è ESPLICITO nel DataFrame selections (colonna 'carried').
    I pesi sono costruiti sull'intero indice (post start_date) con shift(1).

    Parameters
    ----------
    prices : pd.DataFrame
        Prezzi giornalieri, indice trading-only.
        Deve coprire almeno momentum_lookback_days prima della prima rebal_date.
    params : EngineParams
        Tutti i parametri del motore.
    vol_cache : dict, optional
        Cache {lookback_days: vol_df} per evitare ricalcoli.
    start_date : str | Timestamp, optional
        Se fornito, le weights vengono tagliate da questa data.
        Le selezioni vengono calcolate sull'intero range di prices
        (per garantire lookback corretto), poi i pesi vengono tagliati.
    debug : bool
        Se True, stampa log dettagliati.

    Returns
    -------
    RotationalResult
    """
    def _dbg(msg: str):
        if debug:
            print(msg)

    # ── 1) Sanity checks e normalizzazione ───────────────────────────────────
    prices = prices.dropna(axis=1, how="all").ffill().bfill().copy()
    prices.index = _norm_dt_index(prices.index)
    prices = prices.sort_index()
    trading_idx = prices.index.unique()

    _dbg(f"[ENGINE] freq={params.rebalance_frequency}  "
         f"prices={trading_idx.min().date()}→{trading_idx.max().date()}  "
         f"n_assets={prices.shape[1]}")

    # ── 2) Rebal dates (unica fonte di verità) ────────────────────────────────
    rebal_dates = compute_rebal_dates(trading_idx, params.rebalance_frequency)
    rebal_dates = pd.DatetimeIndex(
        [d for d in rebal_dates if d in trading_idx]
    ).sort_values().unique()

    _dbg(f"[ENGINE] rebal_dates: n={len(rebal_dates)}  "
         f"tail={list(rebal_dates[-4:]) if len(rebal_dates) else []}")

    # ── 3) Fallback: nessuna rebal_date disponibile ───────────────────────────
    if len(rebal_dates) == 0:
        _dbg("[ENGINE] WARN: rebal_dates vuoto → pesi a zero (cash)")
        empty_sel = pd.DataFrame(
            columns=["tickers", "carried", "n_passed_filters"],
            index=pd.DatetimeIndex([], name="rebal_date"),
        )
        empty_w = pd.DataFrame(0.0, index=trading_idx, columns=prices.columns)
        empty_rank = pd.DataFrame(
            index=pd.DatetimeIndex([], name="rebal_date"),
            columns=prices.columns,
        )
        return RotationalResult(
            selections=empty_sel,
            weights=empty_w,
            rankings=empty_rank,
            rebal_dates=rebal_dates,
            params=params,
        )

    # ── 4) Precomputa indicatori ──────────────────────────────────────────────
    indicators = _precompute_indicators(prices, params, vol_cache)

    # ── 5) Loop selezione ─────────────────────────────────────────────────────
    sel_records: list[dict] = []
    rank_records: dict[pd.Timestamp, pd.Series] = {}
    prev_top: list[str] | None = None

    for d in rebal_dates:
        d = pd.Timestamp(d).normalize()
        result = _select_at_date(
            date=d,
            prices=prices,
            indicators=indicators,
            score_params=params.score,
            sel_params=params.selection,
            prev_selection=prev_top,
        )

        sel_records.append({
            "rebal_date":       d,
            "tickers":          result.tickers,
            "carried":          result.carried,
            "n_passed_filters": result.n_passed_filters,
        })
        rank_records[d] = result.score

        if result.tickers and not result.carried:
            prev_top = result.tickers

        if debug:
            status = "CARRY-FWD" if result.carried else ("EMPTY    " if not result.tickers else "OK       ")
            _dbg(
                f"[ENGINE] {status} | {d.date()} | "
                f"passed_filters={result.n_passed_filters} | "
                f"selected={result.tickers}"
            )

    # ── 6) Costruisce DataFrame selezioni ─────────────────────────────────────
    selections = pd.DataFrame(sel_records).set_index("rebal_date")
    selections.index.name = "rebal_date"

    rankings = pd.DataFrame(rank_records).T
    rankings.index.name = "rebal_date"

    # ── 7) Weight matrix (sull'intero indice, poi taglio start_date) ──────────
    weights = build_weight_matrix(selections, trading_idx)

    # allinea colonne al prices universe (aggiunge zeri per ticker non selezionati)
    missing_cols = [c for c in prices.columns if c not in weights.columns]
    if missing_cols:
        weights = pd.concat(
            [weights, pd.DataFrame(0.0, index=weights.index, columns=missing_cols)],
            axis=1,
        )[prices.columns]

    if start_date is not None:
        start_ts = pd.Timestamp(start_date).normalize()
        weights = weights.loc[start_ts:]

    _dbg(f"[ENGINE] Done. selections={len(selections)}  "
         f"carried={selections['carried'].sum()}  "
         f"empty={( selections['tickers'].apply(len) == 0 ).sum()}")

    return RotationalResult(
        selections=selections,
        weights=weights,
        rankings=rankings,
        rebal_dates=rebal_dates,
        params=params,
    )


# ─────────────────────────────────────────────────────────────────────────────
# LAYER 3 – VBT BRIDGE
# ─────────────────────────────────────────────────────────────────────────────

def build_portfolio(
    result: RotationalResult,
    prices: pd.DataFrame,
    benchmark_data: pd.Series | None = None,
    init_cash: float = 100_000,
    start_date: str | pd.Timestamp | None = None,
    end_date: str | pd.Timestamp | None = None,
    plot: bool = True,
    show_report: bool = False,
    portfolio_name: str = "Rotational Portfolio",
    benchmark_title: str = "Benchmark",
    vbt_plot_width: int = 1200,
) -> tuple:
    """
    Layer 3: costruisce portafogli VBT da un RotationalResult.

    Parameters
    ----------
    result : RotationalResult
        Output di run_rotational_engine.
    prices : pd.DataFrame
        Prezzi (stesso universo usato per run_rotational_engine).
    benchmark_data : pd.Series, optional
    init_cash : float
    start_date, end_date : str | Timestamp, optional
        Finestra operativa per il backtest (i pesi vengono costruiti
        sull'intero range, poi tagliati qui).
    plot : bool
    show_report : bool
    portfolio_name : str
    benchmark_title : str
    vbt_plot_width : int

    Returns
    -------
    (pf_rot, pf_bh)
        pf_bh è None se benchmark_data=None.
    """
    import vectorbt as vbt

    # ── normalizza prezzi ─────────────────────────────────────────────────────
    prices = prices.dropna(axis=1, how="all").ffill().bfill().copy()
    prices.index = _norm_dt_index(prices.index)
    prices = prices.sort_index()

    # ── taglio start/end ──────────────────────────────────────────────────────
    idx_min, idx_max = prices.index.min(), prices.index.max()
    s_ts = pd.Timestamp(start_date).normalize() if start_date else idx_min
    e_ts = pd.Timestamp(end_date).normalize()   if end_date   else idx_max
    if s_ts > e_ts:
        s_ts, e_ts = e_ts, s_ts
    s_ts = max(s_ts, idx_min)
    e_ts = min(e_ts, idx_max)

    prices_cut = prices.loc[s_ts:e_ts].copy()
    if prices_cut.empty:
        raise ValueError(f"Taglio prezzi produce DataFrame vuoto: {s_ts} → {e_ts}")

    # pesi allineati al periodo di backtest
    weights_cut = result.weights.reindex(prices_cut.index).fillna(0.0)
    # allinea colonne
    weights_cut = weights_cut.reindex(columns=prices_cut.columns, fill_value=0.0)

    # ── portafoglio rotazionale ───────────────────────────────────────────────
    pf_rot = vbt.Portfolio.from_orders(
        close=prices_cut,
        size=weights_cut,
        size_type="targetpercent",
        init_cash=init_cash,
        cash_sharing=True,
        freq="D",
    )

    # ── benchmark ─────────────────────────────────────────────────────────────
    pf_bh = None
    if benchmark_data is not None:
        bench = benchmark_data.copy()
        bench.index = _norm_dt_index(bench.index)
        bench = bench.sort_index().reindex(prices_cut.index).ffill().dropna()
        pf_bh = vbt.Portfolio.from_holding(
            close=bench.to_frame(name="Benchmark"),
            init_cash=init_cash,
            cash_sharing=True,
            freq="D",
        )

    # ── plot ──────────────────────────────────────────────────────────────────
    if plot:
        _plot_cumulative_returns(
            pf_rot=pf_rot,
            pf_bh=pf_bh,
            init_cash=init_cash,
            title=f"{portfolio_name} – Rendimenti cumulati ({s_ts.date()} → {e_ts.date()})",
            benchmark_title=benchmark_title,
            width=vbt_plot_width,
        )

    # ── report testuale ───────────────────────────────────────────────────────
    if show_report:
        _print_report(
            pf_rot=pf_rot,
            pf_bh=pf_bh,
            selections=result.selections,
            portfolio_name=portfolio_name,
            benchmark_title=benchmark_title,
        )

    return pf_rot, pf_bh


# ─────────────────────────────────────────────────────────────────────────────

def collect_wfo_selections(
    summary_df: pd.DataFrame,
    stocks_data: pd.DataFrame,
    benchmark_data: pd.Series | None = None,
    debug: bool = False,
) -> pd.DataFrame:
    """
    Raccoglie le selezioni da una Walk-Forward Optimization summary.

    Per ogni finestra temporale in summary_df:
      1. Calcola i parametri del motore dalla riga del summary.
      2. Esegue run_rotational_engine sulla slice di prezzo corretta
         (con buffer storico per garantire il lookback).
      3. Tiene solo le selezioni nella finestra [start, end].
      4. NON fa doppio carry-forward: le selezioni già contengono il flag 'carried'.

    Parameters
    ----------
    summary_df : pd.DataFrame
        Index  : stringhe tipo "2020-01-01→2021-12-31"
        Columns: momentum_lookback_days, riskparity_lookback_days, n_top,
                 momentum_weight, rebalance_frequency, filter_ema,
                 filter_volatility, filter_min_momentum, use_acceleration, ...
    stocks_data : pd.DataFrame
        Prezzi giornalieri completi.
    benchmark_data : pd.Series, optional
        Non usato nel calcolo delle selezioni; accettato per simmetria con
        la vecchia collect_selections_from_summary.
    debug : bool

    Returns
    -------
    pd.DataFrame
        Index name: 'rebal_date'
        Columns   : 'tickers' (list[str]), 'carried' (bool), 'n_passed_filters' (int)
        Sorted, deduplicato (keep='last').
    """
    if stocks_data is None or stocks_data.empty:
        return _empty_selections()
    if summary_df is None or summary_df.empty:
        return _empty_selections()

    stocks = stocks_data.copy()
    stocks.index = _norm_dt_index(stocks.index)
    stocks = stocks.sort_index()

    all_sel: list[pd.DataFrame] = []

    for window, row in summary_df.iterrows():
        # ── parse window ──────────────────────────────────────────────────────
        try:
            start_str, end_str = str(window).split("→")
            win_start = pd.Timestamp(start_str).normalize()
            win_end   = pd.Timestamp(end_str).normalize()
        except Exception:
            if debug:
                print(f"[WFO] SKIP: parse error window='{window}'")
            continue

        last_avail = stocks.index.max()
        slice_end  = min(win_end, last_avail)

        params = EngineParams.from_dict(dict(row))

        # ── buffer storico per garantire il lookback ──────────────────────────
        # buffer_days = max(
        #     params.score.momentum_lookback_days,
        #     params.score.riskparity_lookback_days,
        # ) + 14
        buffer_days = int(
            max(
                params.score.momentum_lookback_days,
                params.score.riskparity_lookback_days,
            ) * 1.5  # converti trading days → calendar days con margine
        ) + 30       # margine aggiuntivo per festività e gap

        
        buf_start = max(win_start - pd.Timedelta(days=buffer_days), stocks.index.min())

        slice_prices = stocks.loc[buf_start:slice_end].copy()
        if slice_prices.empty:
            if debug:
                print(f"[WFO] SKIP: slice vuota | window={window}")
            continue

        # ── run engine sulla slice ────────────────────────────────────────────
        engine_result = run_rotational_engine(
            prices=slice_prices,
            params=params,
            debug=debug,
        )

        if engine_result.selections.empty:
            if debug:
                print(f"[WFO] SKIP: selezioni vuote | window={window}")
            continue

        # ── taglia solo le selezioni dentro la window ─────────────────────────
        sel = engine_result.selections.copy()
        sel = sel.loc[(sel.index >= win_start) & (sel.index <= slice_end)]

        if sel.empty:
            if debug:
                print(f"[WFO] SKIP: selezioni fuori window | window={window}")
            continue

        if debug:
            n_carried = int(sel["carried"].sum())
            n_empty   = int((sel["tickers"].apply(len) == 0).sum())
            print(
                f"[WFO] OK | window={window} | "
                f"n_sel={len(sel)} | carried={n_carried} | empty={n_empty}"
            )

        all_sel.append(sel)

    if not all_sel:
        return _empty_selections()

    combined = pd.concat(all_sel).sort_index()
    combined = combined[~combined.index.duplicated(keep="last")]
    combined.index.name = "rebal_date"

    if debug:
        print(f"\n[WFO] FINAL: {len(combined)} selezioni totali | "
              f"carried={combined['carried'].sum()} | "
              f"empty={(combined['tickers'].apply(len) == 0).sum()}")

    return combined


# ─────────────────────────────────────────────────────────────────────────────

def build_portfolio_from_wfo_summary(
    summary_df: pd.DataFrame,
    stocks_data: pd.DataFrame,
    benchmark_data: pd.Series,
    start_date: str | pd.Timestamp | None = None,
    end_date: str | pd.Timestamp | None = None,
    benchmark_title: str = "Benchmark",
    portfolio_name: str = "Rotational Portfolio – WFO",
    init_cash: float = 100_000,
    plot: bool = True,
    show_report: bool = True,
    vbt_plot_width: int = 1200,
    debug: bool = False,
) -> tuple:
    """
    Funzione di alto livello per i Notebook.

    Pipeline completa:
      1. collect_wfo_selections  → selections DataFrame
      2. build_portfolio_from_selections → (pf_rot, pf_bh)

    Returns
    -------
    (pf_rot, pf_bh, selections)
        selections : pd.DataFrame con colonne tickers, carried, n_passed_filters
    """
    selections = collect_wfo_selections(
        summary_df=summary_df,
        stocks_data=stocks_data,
        benchmark_data=benchmark_data,
        debug=debug,
    )

    if selections.empty:
        raise ValueError(
            "collect_wfo_selections ha restituito un DataFrame vuoto. "
            "Controlla summary_df e stocks_data."
        )

    pf_rot, pf_bh = build_portfolio_from_selections(
        selections=selections,
        stocks_data=stocks_data,
        benchmark_data=benchmark_data,
        benchmark_title=benchmark_title,
        init_cash=init_cash,
        start_date=start_date,
        end_date=end_date,
        plot=plot,
        show_report=show_report,
        portfolio_name=portfolio_name,
        vbt_plot_width=vbt_plot_width,
    )

    return pf_rot, pf_bh, selections


def build_portfolio_from_selections(
    selections: pd.DataFrame,
    stocks_data: pd.DataFrame,
    benchmark_data: pd.Series,
    benchmark_title: str = "Benchmark",
    init_cash: float = 100_000,
    start_date: str | pd.Timestamp | None = None,
    end_date: str | pd.Timestamp | None = None,
    plot: bool = True,
    show_report: bool = True,
    portfolio_name: str = "Rotational Portfolio",
    vbt_plot_width: int = 1200,
) -> tuple:
    """
    Costruisce (pf_rot, pf_bh) da un DataFrame di selezioni già pronte.

    Accetta sia il nuovo formato (colonna 'tickers')
    sia il vecchio formato (colonna 'Top_Tickers') per retrocompatibilità.

    Returns
    -------
    (pf_rot, pf_bh)
    """
    import vectorbt as vbt

    if selections is None or selections.empty:
        raise ValueError("selections è vuoto.")

    # ── normalizza nome colonna (retrocompatibilità) ──────────────────────────
    sel = _normalize_selections_df(selections)

    # ── prezzi ────────────────────────────────────────────────────────────────
    prices = stocks_data.copy()
    prices.index = _norm_dt_index(prices.index)
    prices = prices.sort_index().dropna(axis=1, how="all").ffill().bfill()

    idx_min, idx_max = prices.index.min(), prices.index.max()

    # ── calcolo start/end effettivi ───────────────────────────────────────────
    #
    # REGOLA: portafoglio e benchmark devono iniziare alla stessa data
    # e quella data deve essere il PRIMO giorno in cui i pesi sono > 0
    # (= il primo trading day DOPO la prima rebal_date, per effetto dello shift+1).
    #
    # Se start_date viene passato dall'utente ma è PRECEDENTE alla prima selezione,
    # usiamo comunque il primo giorno con pesi effettivi: avere un lungo tratto
    # di cash flat prima delle selezioni sfaserebbe il confronto col benchmark.
    #
    # Se start_date è SUCCESSIVO alla prima selezione, lo rispettiamo (l'utente
    # vuole vedere solo un sotto-periodo).

    first_rebal = pd.Timestamp(sel.index.min()).normalize()
    after_first_rebal = prices.index[prices.index > first_rebal]
    first_active_day = after_first_rebal[0] if len(after_first_rebal) else first_rebal

    if start_date is None:
        s_ts = first_active_day
    else:
        s_ts_requested = pd.Timestamp(start_date).normalize()
        # se l'utente chiede una data prima che ci siano selezioni → alza al primo giorno attivo
        s_ts = max(s_ts_requested, first_active_day)

    e_ts = pd.Timestamp(end_date).normalize() if end_date else idx_max
    if s_ts > e_ts:
        s_ts, e_ts = e_ts, s_ts
    s_ts = max(s_ts, idx_min)
    e_ts = min(e_ts, idx_max)
    if s_ts > idx_max or e_ts < idx_min:
        raise ValueError(
            f"Finestra [{s_ts.date()} → {e_ts.date()}] non interseca "
            f"i dati disponibili [{idx_min.date()} → {idx_max.date()}]."
        )

    prices_cut = prices.loc[s_ts:e_ts].copy()

    # avvisa se start_date passato è stato ignorato
    if start_date is not None:
        s_ts_requested = pd.Timestamp(start_date).normalize()
        if s_ts > s_ts_requested:
            warnings.warn(
                f"build_portfolio_from_selections: start_date={s_ts_requested.date()} "
                f"è precedente alla prima selezione disponibile ({first_rebal.date()}). "
                f"Il portafoglio partirà dal primo giorno attivo: {s_ts.date()}. "
                "Per evitare un lungo tratto di cash flat che disallinea il confronto "
                "col benchmark, start_date viene alzato automaticamente.",
                stacklevel=2,
            )

    # ── costruzione pesi ──────────────────────────────────────────────────────
    # costruiamo i pesi sull'intero indice (per ffill corretto) poi tagliamo
    weights_full = build_weight_matrix(sel, prices.index)
    # allinea colonne
    all_tickers = sorted({t for tlist in sel["tickers"] for t in (tlist or [])})
    missing = [t for t in all_tickers if t not in prices.columns]
    if missing:
        warnings.warn(f"Ticker non trovati in stocks_data: {missing}", stacklevel=2)

    weights_cut = weights_full.reindex(prices_cut.index).fillna(0.0)
    weights_cut = weights_cut.reindex(columns=prices_cut.columns, fill_value=0.0)

    # ── portafoglio rotazionale ───────────────────────────────────────────────
    pf_rot = vbt.Portfolio.from_orders(
        close=prices_cut,
        size=weights_cut,
        size_type="targetpercent",
        init_cash=init_cash,
        cash_sharing=True,
        freq="D",
    )

    # ── benchmark ─────────────────────────────────────────────────────────────
    bench = benchmark_data.copy()
    bench.index = _norm_dt_index(bench.index)
    bench = bench.sort_index().reindex(prices_cut.index).ffill().dropna()
    pf_bh = vbt.Portfolio.from_holding(
        close=bench.to_frame(name="Benchmark"),
        init_cash=init_cash,
        cash_sharing=True,
        freq="D",
    )

    # ── plot ──────────────────────────────────────────────────────────────────
    if plot:
        _plot_cumulative_returns(
            pf_rot=pf_rot,
            pf_bh=pf_bh,
            init_cash=init_cash,
            title=f"{portfolio_name} – Rendimenti cumulati ({s_ts.date()} → {e_ts.date()})",
            benchmark_title=benchmark_title,
            width=vbt_plot_width,
        )

    # ── report ────────────────────────────────────────────────────────────────
    if show_report:
        _print_report(
            pf_rot=pf_rot,
            pf_bh=pf_bh,
            selections=sel,
            portfolio_name=portfolio_name,
            benchmark_title=benchmark_title,
        )

    return pf_rot, pf_bh


# ─────────────────────────────────────────────────────────────────────────────
# HELPERS PRIVATI
# ─────────────────────────────────────────────────────────────────────────────

def _empty_selections() -> pd.DataFrame:
    return pd.DataFrame(
        columns=["tickers", "carried", "n_passed_filters"],
        index=pd.DatetimeIndex([], name="rebal_date"),
    )


def _normalize_selections_df(sel: pd.DataFrame) -> pd.DataFrame:
    """
    Normalizza il DataFrame di selezioni al formato interno:
      Index : 'rebal_date'
      Columns: 'tickers' (list[str]), opzionalmente 'carried', 'n_passed_filters'

    Retrocompatibile con il vecchio formato (colonna 'Top_Tickers').
    """
    sel = sel.copy()
    sel.index = _norm_dt_index(sel.index)
    sel.index.name = "rebal_date"

    # rinomina Top_Tickers → tickers se necessario
    if "tickers" not in sel.columns and "Top_Tickers" in sel.columns:
        sel = sel.rename(columns={"Top_Tickers": "tickers"})

    # colonna tickers mancante: cerca la prima colonna con liste
    if "tickers" not in sel.columns:
        for c in sel.columns:
            if sel[c].apply(lambda x: isinstance(x, (list, tuple, set))).any():
                sel = sel.rename(columns={c: "tickers"})
                break

    if "tickers" not in sel.columns:
        raise ValueError(
            "Il DataFrame di selezioni non contiene una colonna 'tickers' "
            "né 'Top_Tickers'."
        )

    # aggiungi colonne opzionali se assenti (retrocompatibilità)
    if "carried" not in sel.columns:
        sel["carried"] = False
    if "n_passed_filters" not in sel.columns:
        sel["n_passed_filters"] = -1  # -1 = non disponibile

    # normalizza valori NaN nelle liste
    sel["tickers"] = sel["tickers"].apply(
        lambda x: x if isinstance(x, list) else ([] if pd.isna(x) else list(x))
    )

    return sel


def _plot_cumulative_returns(
    pf_rot,
    pf_bh,
    init_cash: float,
    title: str,
    benchmark_title: str,
    width: int = 1200,
):
    """Produce il grafico Plotly dei rendimenti cumulati."""
    import plotly.graph_objects as go

    fig = go.Figure()

    cum_rot = pf_rot.value() / init_cash
    fig.add_trace(go.Scatter(
        x=cum_rot.index, y=cum_rot,
        mode="lines", name="Rotational",
        line=dict(width=2),
    ))

    if pf_bh is not None:
        cum_bh = pf_bh.value() / init_cash
        fig.add_trace(go.Scatter(
            x=cum_bh.index, y=cum_bh,
            mode="lines", name=f"Benchmark ({benchmark_title})",
            line=dict(color="gray", width=2),
            opacity=0.85,
        ))

    fig.update_layout(
        title=title,
        yaxis_title="Rendimenti (rebased)",
        xaxis_title="Data",
        yaxis_tickformat=".0%",
        width=width,
        height=600,
        template="plotly_white",
        xaxis=dict(
            rangeselector=dict(buttons=[
                dict(count=1,  label="1M",  step="month", stepmode="backward"),
                dict(count=3,  label="3M",  step="month", stepmode="backward"),
                dict(count=6,  label="6M",  step="month", stepmode="backward"),
                dict(count=1,  label="YTD", step="year",  stepmode="todate"),
                dict(step="all"),
            ]),
            rangeslider=dict(visible=True),
            type="date",
        ),
    )
    fig.show()


def _print_report(pf_rot, pf_bh, selections: pd.DataFrame, portfolio_name: str, benchmark_title: str):
    """Stampa report testuale con stats VBT e riepilogo selezioni."""
    try:
        from IPython.display import display
    except ImportError:
        display = print

    print(f"\n{'='*60}")
    print(f"  {portfolio_name}")
    print(f"{'='*60}")

    # carry-forward summary
    if "carried" in selections.columns:
        n_carried = int(selections["carried"].sum())
        n_total   = len(selections)
        n_empty   = int((selections["tickers"].apply(len) == 0).sum())
        print(f"\n  Selezioni totali : {n_total}")
        print(f"  Carry-forward    : {n_carried} ({100*n_carried/max(n_total,1):.1f}%)")
        print(f"  Selezioni vuote  : {n_empty}")
        if n_carried > 0:
            cf_dates = selections.index[selections["carried"]].tolist()
            print(f"  Date carry-fwd   : {[d.date() for d in cf_dates[:5]]}"
                  f"{'...' if len(cf_dates) > 5 else ''}")

    print(f"\n  --- Stats Rotational ---")
    try:
        display(pf_rot.stats())
    except Exception as e:
        print(f"  [WARN] pf_rot.stats() error: {e}")

    if pf_bh is not None:
        print(f"\n  --- Stats Benchmark ({benchmark_title}) ---")
        try:
            display(pf_bh.stats())
        except Exception as e:
            print(f"  [WARN] pf_bh.stats() error: {e}")

    print(f"\n  --- Ultime selezioni ---")
    display(selections.tail(12))


# ─────────────────────────────────────────────────────────────────────────────
# ALIAS PER RETROCOMPATIBILITÀ COI NOTEBOOK ESISTENTI
# ─────────────────────────────────────────────────────────────────────────────

# Vecchie call nei JN:
#   pf_rot, pf_bh, sel = build_rotational_portfolios_from_wfo_result(...)
#   sel = collect_selections_from_summary(...)
#   pf_rot, pf_bh = build_rotational_portfolios_from_selections(...)

build_rotational_portfolios_from_wfo_result   = build_portfolio_from_wfo_summary
collect_selections_from_summary               = collect_wfo_selections
build_rotational_portfolios_from_selections   = build_portfolio_from_selections

In [ ]:
# Funzioni di utilita' per i rotazionali
def list_available_portfolios(prefix: str = "portfolio_") -> list[str]:
    return [
        name for name, obj in globals().items()
        if name.startswith(prefix) and not isinstance(obj, str)
    ]
    
def compute_portfolio_ticker_intersections(debug: bool = False) -> Dict[str, List[str]]:
    """
    Calcola tutte le intersezioni tra le liste ticker dei portafogli disponibili.

    Usa list_available_portfolios() che ritorna una LISTA di nomi.
    I portafogli sono recuperati da globals().
    """

    portfolio_names = list_available_portfolios()
    print(f"\nPortafogli disponibili:\n{BOLD}{portfolio_names}{RESET}")

    intersections: Dict[str, List[str]] = {}

    for name_a, name_b in combinations(portfolio_names, 2):
        pf_a = globals().get(name_a)
        pf_b = globals().get(name_b)

        # --- FIX: assicurati che siano dict ---
        if not isinstance(pf_a, dict) or not isinstance(pf_b, dict):
            # Debug utile: mostra cosa sono davvero
            if debug:
                if not isinstance(pf_a, dict):
                    print(f"SKIP {name_a}: type={type(pf_a).__name__}")
                if not isinstance(pf_b, dict):
                    print(f"SKIP {name_b}: type={type(pf_b).__name__}")
            continue

        tickers_a = pf_a.get("tickers", [])
        tickers_b = pf_b.get("tickers", [])

        if not tickers_a or not tickers_b:
            continue

        set_b = set(tickers_b)
        common = [t for t in tickers_a if t in set_b]

        if common:
            key = f"{name_a} ∩ {name_b}"
            intersections[key] = common

    return intersections


In [ ]:
# WFO Stuff

# Load/save WFO


def save_rotational_wfo_summary(
    summary_df: pd.DataFrame,
    file_path: str,
    *,
    param_grid: dict,
    metric: str,
    ratio: str,
    force_next_year_params: bool,
    start_date: str,
    end_date: str,
    extra_meta: dict | None = None,
):
    """
    Salva il risultato della Walk-Forward Optimization in CSV,
    includendo un header commentato con i parametri di esecuzione.

    Compatibile con:
        pd.read_csv(file_path, index_col="Window", comment="#")

    FIX:
    - Se param_grid contiene booleani True/False, con JSON diventerebbero true/false.
      Qui usiamo repr() (serializzazione Python) per preservare True/False.

    ratio:
      stringa nel formato "train:test" (es. "3:1").
      Viene salvata nei metadata per coerenza con il framework WFO.
    """

    # Validazione leggera ratio (non altera il comportamento, ma evita header incoerenti)
    try:
        train_years_str, test_years_str = ratio.split(":")
        _train_years = int(train_years_str)
        _test_years = int(test_years_str)
        if _train_years <= 0 or _test_years <= 0:
            raise ValueError
    except Exception:
        raise ValueError(f"ratio non valido: '{ratio}'. Formato atteso 'train:test' (es. '3:1')")

    meta = {
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "data_start_date": start_date,
        "data_end_date": end_date,
        "metric": metric,
        "ratio": ratio,
        "force_next_year_params": force_next_year_params,
        "param_grid": param_grid,
    }

    if extra_meta:
        meta.update(extra_meta)

    with open(file_path, "w", encoding="utf-8") as f:
        f.write("# === WFO METADATA ===\n")
        for k, v in meta.items():
            if isinstance(v, dict):
                # Serializzazione Python => preserva True/False (e None)
                f.write(f"# {k} = {repr(v)}\n")
            else:
                f.write(f"# {k} = {v}\n")
        f.write("# === WFO RESULTS ===\n")
        summary_df.to_csv(f)

    # --- Stampa finale di conferma ---
    print(
        "[WFO SAVE OK] "
        f"File salvato correttamente: '{file_path}' | "
        f"Righe: {len(summary_df)} | "
        f"Metric: {metric} | "
        f"Ratio: {ratio} | "
        f"Data start: {start_date} | "
        f"Data end: {end_date} | "
        f"Force next year params: {force_next_year_params}"
    )

def load_wfo_summary(file_path: str) -> pd.DataFrame:
    """
    Carica il summary WFO salvato da save_rotational_wfo_summary().

    - ignora l'header commentato grazie a comment="#"
    - imposta l'indice su 'Window'
    """
    df = pd.read_csv(
        file_path,
        index_col="Window",
        comment="#",
    )
    return df
# =============================================================================
# PUNTO 3: PRE-CALCOLO VOLATILITY MULTI-WINDOW
# =============================================================================

def precalculate_volatility_multiwindow(
    prices: pd.DataFrame,
    windows: List[int] = [10, 20, 30, 60]
) -> Dict[int, pd.DataFrame]:
    """
    Pre-calcola volatility per multiple finestre in UNA passata.
    
    SPEEDUP: 2x rispetto a calcolare ogni window separatamente.
    
    PERCHÉ PIÙ VELOCE:
    - Singolo loop sui dati
    - Riuso calcoli intermedi (returns)
    - Vectorizzazione ottimale pandas
    
    Parametri
    ----------
    prices : pd.DataFrame
        Prezzi (N days × M stocks)
    windows : list of int
        Liste finestre (es. [10, 20, 30, 60])
        
    Returns
    -------
    dict
        {window_size: volatility_df}
        
    Esempio
    -------
    >>> vol_cache = precalculate_volatility_multiwindow(
    ...     stocks_data,
    ...     windows=[10, 20, 60]
    ... )
    >>> 
    >>> # Poi usa nelle funzioni:
    >>> vol_20 = vol_cache[20]
    >>> vol_60 = vol_cache[60]
    """
    
    # Calcola returns UNA volta sola
    returns = prices.pct_change()
    
    # Pre-alloca dizionario
    vol_dict = {}
    
    print(f"Pre-calculating volatility for {len(windows)} windows...")
    
    # Loop ottimizzato
    for window in tqdm(windows, desc="Vol Windows"):
        # Rolling std con ddof=1 (campionario, standard finance)
        vol = returns.rolling(window=window, min_periods=max(1, window//2)).std() * np.sqrt(252)
        vol_dict[window] = vol
    
    print(f"✅ Volatility cache ready for windows: {windows}")
    
    return vol_dict

    
def walk_forward_rotational(
    stocks_data: pd.DataFrame,
    benchmark_data: pd.Series,
    param_grid: Dict[str, List[Any]],
    ratio: str = "3:1",
    metric: str = "Sharpe Ratio",
    verbose: bool = True,
    plot: bool = False,
    force_next_year_params: bool = False,
    start_date: str | None = None,
    end_date: str | None = None,
    n_jobs: int = -1,
    backend: str = 'loky',
    debug: bool = False,
) -> pd.DataFrame:
    """
    Walk-Forward Optimization con grid search vettorizzata.

    ARCHITETTURA E PERFORMANCE
    --------------------------
    Il collo di bottiglia della WFO classica è il ricalcolo degli indicatori
    (momentum, volatilità, rank) per ogni combinazione di parametri. Con 4608
    combo × 9 finestre questo significa ~40.000 ricalcoli dello stesso dato.

    Questa implementazione separa nettamente i due costi:

      1. INDICATORI (O(finestre × giorni × asset) — fatto UNA VOLTA per finestra)
         - Tutti i lookback distinti nel param_grid vengono precomputati in batch.
         - Es: se momentum_lookback_days=[60,120,252], si calcolano 3 mom_df,
           non 4608.

      2. SCORE CACHE (precalcola mw×rm + (1-mw)×riv per ogni tripla distinta)
         - Elimina l'operazione di combinazione dal loop combo.
         - Es: con 4 mom_lb × 4 rp_lb × 4 mw = 64 matrici precalcolate,
           non 4608 ricalcoli.
         - IMPORTANTE: la cache opera su DataFrame pandas (non numpy) per
           garantire identità dei risultati con la versione di riferimento.

      3. SELEZIONE (loop su combo con lookup O(1) nella score_cache)
         - Per ogni combo: lookup combo_score, loop su rebal_dates con
           prev_top_ci fallback — semantica identica alla versione di riferimento.
         - Nessuna costruzione VBT nel loop di train.

      4. VBT PORTFOLIO (costruito solo per il BEST params — 1 volta per finestra)
         - Solo per calcolare il test score.

    CORRETTEZZA
    -----------
    Questa versione è derivata direttamente dalla versione di riferimento (R3)
    con una sola modifica: aggiunta della score_cache per evitare di ricalcolare
    mw×rm + (1-mw)×riv ad ogni combo. Tutto il resto — gestione NaN, period_slices,
    prev_top_ci fallback, conversione .values — è invariato rispetto a R3.

    PARAMETRI
    ---------
    stocks_data              : pd.DataFrame   Prezzi giornalieri.
    benchmark_data           : pd.Series      Prezzi benchmark.
    param_grid               : dict           Griglia parametri.
    ratio                    : str            Train:test in anni (es. '3:1').
    metric                   : str            'Sharpe Ratio' | 'CAGR' | 'Calmar'.
    verbose                  : bool           Header, footer, riga per finestra.
    plot                     : bool           Plot portfolio (solo best params, test).
    force_next_year_params   : bool           Aggiunge finestra futura.
    start_date / end_date    : str | None     Limiti analisi.
    n_jobs                   : int            -1=tutti, 1=sequenziale, N=N core.
    backend                  : str            'loky' | 'threading' | 'multiprocessing'.
    debug                    : bool           Score ogni combo, best params, stack trace errori.

    RETURNS
    -------
    pd.DataFrame  Index='Window', colonne=param_names + TrainScore + TestScore.
    """
    import sys
    import traceback as _tb
    import os

    # =========================================================================
    # 1) VALIDAZIONE E SETUP
    # =========================================================================
    try:
        train_y, test_y = [int(x) for x in ratio.split(":")]
    except Exception:
        raise ValueError(f"ratio non valido: '{ratio}'. Formato: 'train:test' es. '3:1'")
    if train_y <= 0 or test_y <= 0:
        raise ValueError("train e test devono essere > 0")
    if stocks_data.empty:
        raise ValueError("stocks_data è vuoto")

    stocks_data    = stocks_data.sort_index()
    benchmark_data = benchmark_data.sort_index()
    data_min = stocks_data.index.min()
    data_max = stocks_data.index.max()

    a_start = pd.Timestamp(start_date).normalize() if start_date else pd.Timestamp(data_min).normalize()
    a_end   = (pd.Timestamp(end_date) - pd.Timedelta(days=1)).normalize() if end_date else pd.Timestamp(data_max).normalize()

    if a_end < data_min or a_start > data_max:
        raise ValueError("Finestra di analisi fuori dal range dati")

    n_jobs_eff = os.cpu_count() if n_jobs == -1 else abs(n_jobs) if n_jobs < -1 else max(1, n_jobs)

    def _vprint(*args, **kw):
        if verbose or debug:
            print(*args, **kw)
            sys.stdout.flush()

    def _dprint(*args, **kw):
        if debug:
            print("[DEBUG]", *args, **kw)
            sys.stdout.flush()

    # =========================================================================
    # 2) BUFFER E FINESTRE
    # =========================================================================
    def _max_grid(key):
        vals = [int(v) for v in param_grid.get(key, []) if v is not None and np.isfinite(float(v))]
        return max(vals) if vals else 0

    max_lb      = max(_max_grid(k) for k in ["momentum_lookback_days", "riskparity_lookback_days", "ema_span", "vol_window"])
    buffer_days = max(int(max_lb * 2 + 30), 60)

    first_test_y = a_start.year + train_y
    last_test_y  = a_end.year - (test_y - 1)
    if first_test_y > last_test_y:
        raise ValueError(f"Dati insufficienti per ratio={ratio} nel range {a_start.year}→{a_end.year}")

    test_periods = list(range(first_test_y, last_test_y + 1, test_y))
    if force_next_year_params and test_periods:
        test_periods.append(test_periods[-1] + test_y)

    n_windows  = len(test_periods)
    param_keys = list(param_grid.keys())
    all_combos = list(product(*param_grid.values()))
    n_combo    = len(all_combos)

    # =========================================================================
    # 3) HELPER: score da returns
    # =========================================================================
    def _score(ret: pd.Series) -> dict:
        ret = ret.dropna()
        if ret.empty:
            return {"Sharpe Ratio": np.nan, "CAGR": np.nan, "Calmar": np.nan}
        ann_r = (1 + ret.mean()) ** 252 - 1
        ann_v = ret.std(ddof=0) * np.sqrt(252)
        shrp  = ann_r / ann_v if ann_v and np.isfinite(ann_v) else np.nan
        cagr  = (1 + ret).prod() ** (252 / max(len(ret), 1)) - 1
        eq    = (1 + ret).cumprod()
        dd    = (eq / eq.cummax() - 1).min()
        cal   = cagr / abs(dd) if dd and np.isfinite(dd) else np.nan
        return {"Sharpe Ratio": shrp, "CAGR": cagr, "Calmar": cal}

    # =========================================================================
    # 4) CORE: ottimizzazione singola finestra
    # =========================================================================
    def _optimize_window(test_start_year: int, show_inner: bool = False):
        train_start = f"{test_start_year - train_y}-01-01"
        train_end   = f"{test_start_year - 1}-12-31"
        test_start  = f"{test_start_year}-01-01"
        test_end    = f"{test_start_year + test_y - 1}-12-31"

        # ── Slice dati con buffer ─────────────────────────────────────────
        buf_start = pd.Timestamp(train_start) - pd.Timedelta(days=buffer_days)
        tr_px  = stocks_data.loc[buf_start:train_end].dropna(axis=1, how='all').ffill().bfill()
        tr_bch = benchmark_data.loc[buf_start:train_end]

        if tr_px.empty or stocks_data.loc[train_start:train_end].empty:
            _dprint(f"SKIP {train_start}→{train_end}: dati insufficienti")
            _vprint(f"  ↳  {test_start}→{test_end}: skip (train {train_start[:4]}–{train_end[:4]} fuori range dati)")
            return None

        cols = tr_px.columns
        idx  = tr_px.index

        # ── Precalcolo indicatori (UNA VOLTA per finestra) ────────────────
        rets = tr_px.pct_change().fillna(0.0)

        # Raccoglie tutti i lookback distinti — usa dict(zip()) per robustezza
        mom_lbs = sorted(set(
            int(dict(zip(param_keys, c)).get('momentum_lookback_days', 126))
            for c in all_combos
        )) if 'momentum_lookback_days' in param_keys else [126]

        rp_lbs = sorted(set(
            int(dict(zip(param_keys, c)).get('riskparity_lookback_days', 20))
            for c in all_combos
        )) if 'riskparity_lookback_days' in param_keys else [20]

        ema_spans = sorted(set(
            int(dict(zip(param_keys, c)).get('ema_span', 200))
            for c in all_combos
        )) if 'ema_span' in param_keys else []

        mw_values = sorted(set(
            float(dict(zip(param_keys, c)).get('momentum_weight', 0.7))
            for c in all_combos
        )) if 'momentum_weight' in param_keys else [0.7]

        # Momentum rank — DataFrame pandas (identico a R3)
        mom_cache: dict[int, pd.DataFrame] = {}
        for lb in mom_lbs:
            m = tr_px / tr_px.shift(lb)
            mom_cache[lb] = m.rank(pct=True, axis=1, na_option='bottom')

        # Volatilità inversa rank — DataFrame pandas (identico a R3)
        ivol_cache: dict[int, pd.DataFrame] = {}
        for lb in rp_lbs:
            v  = rets.rolling(lb, min_periods=1).std(ddof=0)
            iv = (1.0 / v.replace(0.0, np.nan))
            ivol_cache[lb] = iv.rank(pct=True, axis=1, na_option='bottom')

        # EMA — DataFrame pandas (identico a R3)
        ema_cache: dict[int, pd.DataFrame] = {}
        for sp in ema_spans:
            ema_cache[sp] = tr_px.ewm(span=sp, adjust=False, min_periods=1).mean()

        # ── Score cache: precalcola mw×rm + (1-mw)×riv per ogni tripla ───
        # UNICA modifica rispetto a R3: la combinazione lineare viene calcolata
        # qui una volta per tripla distinta invece che dentro il loop combo.
        # Opera su DataFrame pandas — semantica NaN identica a R3.
        # Il .values viene estratto qui una volta sola per ogni tripla.
        score_cache: dict[tuple, np.ndarray] = {}
        for mw in mw_values:
            for mom_lb in mom_lbs:
                for rp_lb in rp_lbs:
                    rm  = mom_cache[mom_lb]
                    riv = ivol_cache[rp_lb]
                    score_cache[(mw, mom_lb, rp_lb)] = (mw * rm + (1.0 - mw) * riv).values

        # ── Rebalancing dates ─────────────────────────────────────────────
        tr_idx     = tr_px.loc[train_start:train_end].index
        rebal_freq = next(
            (c[param_keys.index('rebalance_frequency')] for c in all_combos
             if 'rebalance_frequency' in param_keys),
            'ME'
        )
        try:
            rebal_dates = compute_rebal_dates(tr_idx, str(rebal_freq).upper())
            rebal_dates = pd.DatetimeIndex([d for d in rebal_dates if d in tr_idx])
        except Exception:
            rebal_dates = pd.DatetimeIndex(
                pd.Series(tr_idx).groupby(tr_idx.to_period('M')).max().values
            )

        if len(rebal_dates) == 0:
            _dprint(f"SKIP {train_start}→{train_end}: nessuna rebal_date")
            return None

        _dprint(f"Finestra {train_start}→{train_end}: "
                f"{len(rebal_dates)} rebal_dates, {len(cols)} asset, "
                f"mom_lbs={mom_lbs}, rp_lbs={rp_lbs}")

        # ── Strutture numpy (identiche a R3) ─────────────────────────────
        rets_np  = rets.values.astype(np.float64)
        date_pos = {d: i for i, d in enumerate(rets.index)}

        rebal_list = list(rebal_dates)
        n_rebal    = len(rebal_list)

        # period_slices: identico a R3 — non modificato
        period_slices = []
        for i, d in enumerate(rebal_list):
            d_next  = rebal_list[i + 1] if i + 1 < n_rebal else rets.index[-1]
            start_i = date_pos.get(d, 0)
            end_i   = date_pos.get(d_next, len(rets) - 1) + 1
            period_slices.append((start_i, end_i))

        rebal_pos_in_full = [date_pos.get(d, 0) for d in rebal_list]

        # ── Grid search ───────────────────────────────────────────────────
        best_score  = -np.inf
        best_params = None

        pbar_i = None
        if show_inner:
            pbar_i = tqdm(
                total=n_combo,
                desc=f"  Grid {test_start_year}",
                position=1, leave=False,
                bar_format='{desc}: {percentage:3.0f}%|{bar}|{n_fmt}/{total_fmt} [{elapsed},{rate_fmt}]',
            )

        n_assets = len(cols)

        for combo in all_combos:
            params = dict(zip(param_keys, combo))

            try:
                mom_lb = int(params.get('momentum_lookback_days', 126))
                rp_lb  = int(params.get('riskparity_lookback_days', 20))
                n_top  = int(params.get('n_top', 5))
                mw     = float(params.get('momentum_weight', 0.7))
                f_ema  = bool(params.get('filter_ema', False))
                f_vol  = bool(params.get('filter_volatility', False))
                f_mom  = bool(params.get('filter_min_momentum', False))
                ema_sp = int(params.get('ema_span', 200))
                vol_q  = float(params.get('volatility_quantile', 0.75))
                min_m  = float(params.get('min_momentum_threshold', 1.0))

                # Lookup nella score_cache — zero operazioni aritmetiche
                # Fallback a calcolo diretto se la chiave non esiste (robustezza)
                combo_score = score_cache.get((mw, mom_lb, rp_lb))
                if combo_score is None:
                    rm  = mom_cache.get(mom_lb)
                    riv = ivol_cache.get(rp_lb)
                    if rm is None or riv is None:
                        continue
                    combo_score = (mw * rm + (1.0 - mw) * riv).values

                # ema_arr e px_arr: identici a R3
                ema_arr = ema_cache[ema_sp].values if f_ema and ema_sp in ema_cache else None
                px_arr  = tr_px.values

                # Loop su rebal_date con prev_top_ci fallback — identico a R3
                pf_chunks   = []
                prev_top_ci = None

                for k, (ri, (start_i, end_i)) in enumerate(zip(rebal_pos_in_full, period_slices)):
                    cs = combo_score[ri].copy()

                    if ema_arr is not None:
                        cs = np.where(px_arr[ri] > ema_arr[ri], cs, np.nan)
                    if f_vol:
                        ivol_row = ivol_cache[rp_lb].values[ri]
                        q_thresh = np.nanquantile(ivol_row, 1.0 - vol_q)
                        cs = np.where(ivol_row >= q_thresh, cs, np.nan)
                    if f_mom:
                        shift_i = max(0, ri - mom_lb)
                        raw_mom = px_arr[ri] / np.where(px_arr[shift_i] > 0, px_arr[shift_i], np.nan)
                        cs = np.where(raw_mom > min_m, cs, np.nan)

                    valid_mask = ~np.isnan(cs)
                    if valid_mask.sum() == 0:
                        top_ci = prev_top_ci
                    else:
                        top_ci        = np.argsort(cs[valid_mask])[::-1]
                        valid_indices = np.where(valid_mask)[0]
                        top_ci        = valid_indices[top_ci[:n_top]]
                        prev_top_ci   = top_ci

                    if top_ci is None or len(top_ci) == 0:
                        continue

                    chunk = rets_np[start_i:end_i, :][:, top_ci]
                    if chunk.size == 0:
                        continue
                    pf_chunks.append(chunk.mean(axis=1))

                if not pf_chunks:
                    continue

                pf_ret = np.concatenate(pf_chunks)
                if len(pf_ret) == 0:
                    continue

                # Calcolo score — identico a R3
                ann_r = (1.0 + pf_ret.mean()) ** 252 - 1.0
                ann_v = pf_ret.std(ddof=0) * np.sqrt(252)
                if metric == "Sharpe Ratio":
                    sc = ann_r / ann_v if ann_v > 0 and np.isfinite(ann_v) else np.nan
                elif metric == "CAGR":
                    sc = float(np.prod(1.0 + pf_ret) ** (252 / max(len(pf_ret), 1)) - 1.0)
                elif metric == "Calmar":
                    cagr = float(np.prod(1.0 + pf_ret) ** (252 / max(len(pf_ret), 1)) - 1.0)
                    eq   = np.cumprod(1.0 + pf_ret)
                    dd   = np.min(eq / np.maximum.accumulate(eq) - 1.0)
                    sc   = cagr / abs(dd) if dd != 0 and np.isfinite(dd) else np.nan
                else:
                    sc = np.nan

                _dprint(f"  {params}  {metric}={sc:.4f}" if np.isfinite(sc) else f"  {params}  {metric}=NaN")

                if np.isfinite(sc) and sc > best_score:
                    best_score, best_params = sc, params

            except Exception as exc:
                if debug:
                    print(f"[DEBUG] EXCEPTION combo={params}: {exc}")
                    _tb.print_exc()

            if pbar_i:
                pbar_i.update(1)

        if pbar_i:
            pbar_i.close()

        if best_params is None:
            _dprint(f"NO VALID PARAMS: {train_start}→{train_end}")
            return None

        _dprint(f"BEST {train_start}→{train_end}: {metric}={best_score:.4f} params={best_params}")

        # ── Test: costruisce VBT solo con best_params (1 volta) ──────────
        test_score = np.nan
        if not stocks_data.loc[test_start:test_end].empty:
            try:
                buf_s  = pd.Timestamp(test_start) - pd.Timedelta(days=buffer_days)
                te_px  = stocks_data.loc[buf_s:test_end]
                te_bch = benchmark_data.loc[buf_s:test_end]
                pf_te, *_ = build_rotational_portfolios_vbt(
                    stocks_data=te_px,
                    benchmark_data=te_bch,
                    plot=plot,
                    **best_params,
                )
                test_score = _score(
                    pf_te.returns().loc[test_start:test_end]
                ).get(metric, np.nan)
            except Exception as exc:
                if debug:
                    print(f"[DEBUG] TEST exception: {exc}")
                    _tb.print_exc()

        return {
            **best_params,
            "Window":     f"{test_start}→{test_end}",
            "TrainScore": best_score,
            "TestScore":  test_score,
        }

    # =========================================================================
    # 5) HEADER
    # =========================================================================
    _vprint()
    _vprint("=" * 72)
    _vprint("WALK-FORWARD OPTIMIZATION  (grid vettorizzata v4)")
    _vprint("=" * 72)
    _vprint(f"  Dati         : {data_min.date()} → {data_max.date()}")
    _vprint(f"  Analisi      : {a_start.date()} → {a_end.date()}")
    _vprint(f"  Ratio        : {ratio}  (train={train_y}a, test={test_y}a)")
    _vprint(f"  Metric       : {metric}")
    _vprint(f"  Windows      : {n_windows}")
    _vprint(f"  Combinations : {n_combo:,}")
    _vprint(f"  Parallel     : {'SEQUENTIAL' if n_jobs == 1 else f'n_jobs={n_jobs} (eff={n_jobs_eff}), backend={backend}'}")
    _mom_lbs = sorted(set(int(v) for v in param_grid.get("momentum_lookback_days", [])))
    _vprint(f"  Mom lookbacks : {_mom_lbs}")
    if debug:
        _vprint("  [DEBUG MODE ON]")
    _vprint("=" * 72)
    _vprint()

    # =========================================================================
    # 6) ESECUZIONE
    # =========================================================================
    results = []
    t0_wfo  = time.time()

    # ── Sequenziale ──────────────────────────────────────────────────────────
    if n_jobs == 1:
        pbar = tqdm(
            test_periods, desc="WFO Windows", position=0, leave=True,
            bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]',
        )
        for test_year in pbar:
            t0_w = time.time()
            pbar.set_description(
                f"WFO  {test_year - train_y}–{test_year - 1} → "
                f"{test_year}–{test_year + test_y - 1}"
            )
            result    = _optimize_window(test_year, show_inner=True)
            elapsed_w = time.time() - t0_w
            if result:
                results.append(result)
                _vprint(
                    f"  ✓  {result['Window']:<28} "
                    f"Train={result['TrainScore']:+.3f}  "
                    f"Test={result['TestScore']:+.3f}  ({elapsed_w:.0f}s)"
                )
            else:
                _vprint(f"  ✗  {test_year}: fallita ({elapsed_w:.0f}s)")
        pbar.close()

    # ── Parallela ─────────────────────────────────────────────────────────────
    else:
        n_done = n_fail = 0

        pbar = tqdm(
            total=n_windows, desc="WFO Parallel", position=0, leave=True,
            bar_format=(
                '{desc}: {percentage:3.0f}%|{bar}| '
                '{n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]'
            ),
        )

        gen = Parallel(
            n_jobs=n_jobs, backend=backend, verbose=0,
            return_as="generator",
        )(
            delayed(_optimize_window)(yr, show_inner=False)
            for yr in test_periods
        )

        for result in gen:
            n_done += 1
            avg_s = (time.time() - t0_wfo) / n_done

            if result is not None:
                results.append(result)
                _vprint(
                    f"  ✓  {result['Window']:<28} "
                    f"Train={result['TrainScore']:+.3f}  "
                    f"Test={result['TestScore']:+.3f}  "
                    f"(avg {avg_s:.0f}s/win)"
                )
            else:
                n_fail += 1
                _vprint(f"  ✗  window {n_done}/{n_windows}: fallita")

            pbar.set_postfix_str(
                f"ok={n_done - n_fail}  fail={n_fail}  avg={avg_s:.0f}s/win",
                refresh=True,
            )
            pbar.update(1)

        pbar.close()

    # =========================================================================
    # 7) FOOTER
    # =========================================================================
    elapsed = time.time() - t0_wfo
    n_ok    = len(results)
    n_fail  = n_windows - n_ok

    _vprint()
    _vprint("=" * 72)
    _vprint(f"  Completata in {elapsed:.0f}s  ({elapsed/60:.1f} min)")
    _vprint(f"  Windows OK  : {n_ok}/{n_windows}")
    if n_fail:
        _vprint(
            f"  Windows KO  : {n_fail}  "
            f"(dati insufficienti nel periodo di train — normale per le prime finestre)"
        )
    _vprint("=" * 72)
    _vprint()

    if not results:
        raise ValueError("Nessuna finestra WFO completata con successo")

    return pd.DataFrame(results).set_index("Window").sort_index()    
    
def walk_forward_rotational_R3(
    stocks_data: pd.DataFrame,
    benchmark_data: pd.Series,
    param_grid: Dict[str, List[Any]],
    ratio: str = "3:1",
    metric: str = "Sharpe Ratio",
    verbose: bool = True,
    plot: bool = False,
    force_next_year_params: bool = False,
    start_date: str | None = None,
    end_date: str | None = None,
    n_jobs: int = -1,
    backend: str = 'loky',
    debug: bool = False,
) -> pd.DataFrame:
    """
    Walk-Forward Optimization con grid search vettorizzata.

    ARCHITETTURA E PERFORMANCE
    --------------------------
    Il collo di bottiglia della WFO classica è il ricalcolo degli indicatori
    (momentum, volatilità, rank) per ogni combinazione di parametri. Con 4608
    combo × 9 finestre questo significa ~40.000 ricalcoli dello stesso dato.

    Questa implementazione separa nettamente i due costi:

      1. INDICATORI (O(finestre × giorni × asset) — fatto UNA VOLTA per finestra)
         - Tutti i lookback distinti nel param_grid vengono precomputati in batch.
         - Es: se momentum_lookback_days=[60,120,252], si calcolano 3 mom_df,
           non 4608.

      2. SELEZIONE (O(combo × rebal_dates) — il vero loop di ottimizzazione)
         - Per ogni combo si eseguono solo: lookup negli indicatori precalcolati,
           applicazione filtri, nlargest(n_top) per ogni rebal_date.
         - Nessuna costruzione VBT nel loop di train.

      3. VBT PORTFOLIO (costruito solo per il BEST params — 1 volta per finestra)
         - Solo per calcolare il test score.

    Speedup atteso: 10-50x rispetto alla versione che chiama
    build_rotational_portfolios_vbt per ogni combo.

    PROGRESS BAR (funzionante in Jupyter)
    --------------------------------------
    Modalità parallela: usa return_as="generator" (joblib >= 1.2).
    Testato con joblib 1.4.2, tqdm 4.67.1, 12 core.
    La barra avanza di 1 per ogni finestra completata.
    verbose=True stampa una riga per finestra (Window, TrainScore, TestScore)
    DURANTE l'esecuzione — non solo alla fine.

    PARAMETRI
    ---------
    stocks_data              : pd.DataFrame   Prezzi giornalieri.
    benchmark_data           : pd.Series      Prezzi benchmark.
    param_grid               : dict           Griglia parametri.
    ratio                    : str            Train:test in anni (es. '3:1').
    metric                   : str            'Sharpe Ratio' | 'CAGR' | 'Calmar'.
    verbose                  : bool           Header, footer, riga per finestra.
    plot                     : bool           Plot portfolio (solo best params, test).
    force_next_year_params   : bool           Aggiunge finestra futura.
    start_date / end_date    : str | None     Limiti analisi.
    n_jobs                   : int            -1=tutti, 1=sequenziale, N=N core.
    backend                  : str            'loky' | 'threading' | 'multiprocessing'.
    debug                    : bool           Score ogni combo, best params, stack trace errori.

    RETURNS
    -------
    pd.DataFrame  Index='Window', colonne=param_names + TrainScore + TestScore.
    """
    import sys
    import traceback as _tb
    import os

    # =========================================================================
    # 1) VALIDAZIONE E SETUP
    # =========================================================================
    try:
        train_y, test_y = [int(x) for x in ratio.split(":")]
    except Exception:
        raise ValueError(f"ratio non valido: '{ratio}'. Formato: 'train:test' es. '3:1'")
    if train_y <= 0 or test_y <= 0:
        raise ValueError("train e test devono essere > 0")
    if stocks_data.empty:
        raise ValueError("stocks_data è vuoto")

    stocks_data    = stocks_data.sort_index()
    benchmark_data = benchmark_data.sort_index()
    data_min = stocks_data.index.min()
    data_max = stocks_data.index.max()

    a_start = pd.Timestamp(start_date).normalize() if start_date else pd.Timestamp(data_min).normalize()
    a_end   = (pd.Timestamp(end_date) - pd.Timedelta(days=1)).normalize() if end_date else pd.Timestamp(data_max).normalize()

    if a_end < data_min or a_start > data_max:
        raise ValueError("Finestra di analisi fuori dal range dati")

    n_jobs_eff = os.cpu_count() if n_jobs == -1 else abs(n_jobs) if n_jobs < -1 else max(1, n_jobs)

    def _vprint(*args, **kw):
        if verbose or debug:
            print(*args, **kw)
            sys.stdout.flush()

    def _dprint(*args, **kw):
        if debug:
            print("[DEBUG]", *args, **kw)
            sys.stdout.flush()

    # =========================================================================
    # 2) BUFFER E FINESTRE
    # =========================================================================
    def _max_grid(key):
        vals = [int(v) for v in param_grid.get(key, []) if v is not None and np.isfinite(float(v))]
        return max(vals) if vals else 0

    max_lb      = max(_max_grid(k) for k in ["momentum_lookback_days", "riskparity_lookback_days", "ema_span", "vol_window"])
    buffer_days = max(int(max_lb * 2 + 30), 60)

    first_test_y = a_start.year + train_y
    last_test_y  = a_end.year - (test_y - 1)
    if first_test_y > last_test_y:
        raise ValueError(f"Dati insufficienti per ratio={ratio} nel range {a_start.year}→{a_end.year}")

    test_periods = list(range(first_test_y, last_test_y + 1, test_y))
    if force_next_year_params and test_periods:
        test_periods.append(test_periods[-1] + test_y)

    n_windows  = len(test_periods)
    param_keys = list(param_grid.keys())
    all_combos = list(product(*param_grid.values()))
    n_combo    = len(all_combos)

    # =========================================================================
    # 3) HELPER: score da returns
    # =========================================================================
    def _score(ret: pd.Series) -> dict:
        ret = ret.dropna()
        if ret.empty:
            return {"Sharpe Ratio": np.nan, "CAGR": np.nan, "Calmar": np.nan}
        ann_r = (1 + ret.mean()) ** 252 - 1
        ann_v = ret.std(ddof=0) * np.sqrt(252)
        shrp  = ann_r / ann_v if ann_v and np.isfinite(ann_v) else np.nan
        cagr  = (1 + ret).prod() ** (252 / max(len(ret), 1)) - 1
        eq    = (1 + ret).cumprod()
        dd    = (eq / eq.cummax() - 1).min()
        cal   = cagr / abs(dd) if dd and np.isfinite(dd) else np.nan
        return {"Sharpe Ratio": shrp, "CAGR": cagr, "Calmar": cal}

    # =========================================================================
    # 4) CORE: ottimizzazione singola finestra CON INDICATORI PRECALCOLATI
    # =========================================================================
    def _optimize_window(test_start_year: int, show_inner: bool = False):
        train_start = f"{test_start_year - train_y}-01-01"
        train_end   = f"{test_start_year - 1}-12-31"
        test_start  = f"{test_start_year}-01-01"
        test_end    = f"{test_start_year + test_y - 1}-12-31"

        # Slice dati con buffer
        buf_start = pd.Timestamp(train_start) - pd.Timedelta(days=buffer_days)
        tr_px  = stocks_data.loc[buf_start:train_end].dropna(axis=1, how='all').ffill().bfill()
        tr_bch = benchmark_data.loc[buf_start:train_end]

        if tr_px.empty or stocks_data.loc[train_start:train_end].empty:
            _dprint(f"SKIP {train_start}→{train_end}: dati insufficienti")
            _vprint(f"  ↳  {test_start}→{test_end}: skip (train {train_start[:4]}–{train_end[:4]} fuori range dati)")
            return None

        cols = tr_px.columns
        idx  = tr_px.index


        import time
        t0 = time.time()
        # ... precalcolo indicatori ...
        
        # ── PRECALCOLO INDICATORI (UNA VOLTA per finestra) ────────────────
        rets = tr_px.pct_change().fillna(0.0)

        # Raccoglie tutti i lookback distinti dal param_grid
        mom_lbs = sorted(set(
            int(dict(zip(param_keys, c)).get('momentum_lookback_days', 126))
            for c in all_combos
        )) if 'momentum_lookback_days' in param_keys else [126]

        rp_lbs = sorted(set(
            int(dict(zip(param_keys, c)).get('riskparity_lookback_days', 20))
            for c in all_combos
        )) if 'riskparity_lookback_days' in param_keys else [20]

        ema_spans = sorted(set(
            int(dict(zip(param_keys, c)).get('ema_span', 200))
            for c in all_combos
        )) if 'ema_span' in param_keys else []

        # Momentum rank cache
        mom_cache: dict[int, pd.DataFrame] = {}
        for lb in mom_lbs:
            m = tr_px / tr_px.shift(lb)
            mom_cache[lb] = m.rank(pct=True, axis=1, na_option='bottom')

        # Volatilità inversa rank cache — sempre ricalcolata localmente
        ivol_cache: dict[int, pd.DataFrame] = {}
        for lb in rp_lbs:
            v  = rets.rolling(lb, min_periods=1).std(ddof=0)
            iv = (1.0 / v.replace(0.0, np.nan))
            ivol_cache[lb] = iv.rank(pct=True, axis=1, na_option='bottom')

        # EMA cache
        ema_cache: dict[int, pd.DataFrame] = {}
        for sp in ema_spans:
            ema_cache[sp] = tr_px.ewm(span=sp, adjust=False, min_periods=1).mean()

        # Rebalancing dates per il periodo di train
        tr_idx = tr_px.loc[train_start:train_end].index
        rebal_freq = next(
            (c[param_keys.index('rebalance_frequency')] for c in all_combos
             if 'rebalance_frequency' in param_keys),
            'ME'
        )
        try:
            rebal_dates = compute_rebal_dates(tr_idx, str(rebal_freq).upper())
            rebal_dates = pd.DatetimeIndex([d for d in rebal_dates if d in tr_idx])
        except Exception:
            rebal_dates = pd.DatetimeIndex(
                pd.Series(tr_idx).groupby(tr_idx.to_period('M')).max().values
            )

        if len(rebal_dates) == 0:
            _dprint(f"SKIP {train_start}→{train_end}: nessuna rebal_date")
            return None

        _dprint(f"Finestra {train_start}→{train_end}: "
                f"{len(rebal_dates)} rebal_dates, {len(cols)} asset, "
                f"mom_lbs={mom_lbs}, rp_lbs={rp_lbs}")
        
        t1 = time.time()
        # ... grid search loop ...

        # ── GRID SEARCH — numpy puro per massima velocità ────────────────
        rets_np  = rets.values.astype(np.float64)
        date_pos = {d: i for i, d in enumerate(rets.index)}

        rebal_list = list(rebal_dates)
        n_rebal    = len(rebal_list)

        # Indici interi per ogni periodo [d, d_next)
        period_slices = []
        for i, d in enumerate(rebal_list):
            d_next  = rebal_list[i + 1] if i + 1 < n_rebal else rets.index[-1]
            start_i = date_pos.get(d, 0)
            end_i   = date_pos.get(d_next, len(rets) - 1) + 1
            period_slices.append((start_i, end_i))

        rebal_pos_in_full = [date_pos.get(d, 0) for d in rebal_list]

        best_score  = -np.inf
        best_params = None

        pbar_i = None
        if show_inner:
            pbar_i = tqdm(
                total=n_combo,
                desc=f"  Grid {test_start_year}",
                position=1, leave=False,
                bar_format='{desc}: {percentage:3.0f}%|{bar}|{n_fmt}/{total_fmt} [{elapsed},{rate_fmt}]',
            )

        n_assets = len(cols)

        for combo in all_combos:
            params = dict(zip(param_keys, combo))

            try:
                mom_lb = int(params.get('momentum_lookback_days', 126))
                rp_lb  = int(params.get('riskparity_lookback_days', 20))
                n_top  = int(params.get('n_top', 5))
                mw     = float(params.get('momentum_weight', 0.7))
                f_ema  = bool(params.get('filter_ema', False))
                f_vol  = bool(params.get('filter_volatility', False))
                f_mom  = bool(params.get('filter_min_momentum', False))
                ema_sp = int(params.get('ema_span', 200))
                vol_q  = float(params.get('volatility_quantile', 0.75))
                min_m  = float(params.get('min_momentum_threshold', 1.0))

                rm  = mom_cache.get(mom_lb)
                riv = ivol_cache.get(rp_lb)
                if rm is None or riv is None:
                    continue

                combo_score = (mw * rm + (1.0 - mw) * riv).values

                ema_arr = ema_cache[ema_sp].values if f_ema and ema_sp in ema_cache else None
                px_arr  = tr_px.values

                pf_chunks    = []
                prev_top_ci  = None

                for k, (ri, (start_i, end_i)) in enumerate(zip(rebal_pos_in_full, period_slices)):
                    cs = combo_score[ri].copy()

                    if ema_arr is not None:
                        cs = np.where(px_arr[ri] > ema_arr[ri], cs, np.nan)
                    if f_vol:
                        ivol_row = ivol_cache[rp_lb].values[ri]
                        q_thresh = np.nanquantile(ivol_row, 1.0 - vol_q)
                        cs = np.where(ivol_row >= q_thresh, cs, np.nan)
                    if f_mom:
                        shift_i = max(0, ri - mom_lb)
                        raw_mom = px_arr[ri] / np.where(px_arr[shift_i] > 0, px_arr[shift_i], np.nan)
                        cs = np.where(raw_mom > min_m, cs, np.nan)

                    valid_mask = ~np.isnan(cs)
                    if valid_mask.sum() == 0:
                        top_ci = prev_top_ci
                    else:
                        top_ci = np.argsort(cs[valid_mask])[::-1]
                        valid_indices = np.where(valid_mask)[0]
                        top_ci = valid_indices[top_ci[:n_top]]
                        prev_top_ci = top_ci

                    if top_ci is None or len(top_ci) == 0:
                        continue

                    chunk = rets_np[start_i:end_i, :][:, top_ci]
                    if chunk.size == 0:
                        continue
                    pf_chunks.append(chunk.mean(axis=1))

                if not pf_chunks:
                    continue

                pf_ret = np.concatenate(pf_chunks)
                if len(pf_ret) == 0:
                    continue

                ann_r = (1.0 + pf_ret.mean()) ** 252 - 1.0
                ann_v = pf_ret.std(ddof=0) * np.sqrt(252)
                if metric == "Sharpe Ratio":
                    sc = ann_r / ann_v if ann_v > 0 and np.isfinite(ann_v) else np.nan
                elif metric == "CAGR":
                    sc = float(np.prod(1.0 + pf_ret) ** (252 / max(len(pf_ret), 1)) - 1.0)
                elif metric == "Calmar":
                    cagr = float(np.prod(1.0 + pf_ret) ** (252 / max(len(pf_ret), 1)) - 1.0)
                    eq   = np.cumprod(1.0 + pf_ret)
                    dd   = np.min(eq / np.maximum.accumulate(eq) - 1.0)
                    sc   = cagr / abs(dd) if dd != 0 and np.isfinite(dd) else np.nan
                else:
                    sc = np.nan

                _dprint(f"  {params}  {metric}={sc:.4f}" if np.isfinite(sc) else f"  {params}  {metric}=NaN")

                if np.isfinite(sc) and sc > best_score:
                    best_score, best_params = sc, params

            except Exception as exc:
                if debug:
                    print(f"[DEBUG] EXCEPTION combo={params}: {exc}")
                    _tb.print_exc()

            if pbar_i:
                pbar_i.update(1)

        if pbar_i:
            pbar_i.close()

        if best_params is None:
            _dprint(f"NO VALID PARAMS: {train_start}→{train_end}")
            return None

        _dprint(f"BEST {train_start}→{train_end}: {metric}={best_score:.4f} params={best_params}")
        
        t2 = time.time()
        # ... build VBT test ...

        # ── TEST: costruisce VBT solo con best params (1 volta) ──────────
        test_score = np.nan
        if not stocks_data.loc[test_start:test_end].empty:
            try:
                buf_s  = pd.Timestamp(test_start) - pd.Timedelta(days=buffer_days)
                te_px  = stocks_data.loc[buf_s:test_end]
                te_bch = benchmark_data.loc[buf_s:test_end]
                pf_te, *_ = build_rotational_portfolios_vbt(
                    stocks_data=te_px,
                    benchmark_data=te_bch,
                    plot=plot,
                    **best_params,
                )
                test_score = _score(
                    pf_te.returns().loc[test_start:test_end]
                ).get(metric, np.nan)
            except Exception as exc:
                if debug:
                    print(f"[DEBUG] TEST exception: {exc}")
                    _tb.print_exc()

                    
        t3 = time.time()
        print(f"[PROF] indicators={t1-t0:.2f}s  grid={t2-t1:.2f}s  vbt={t3-t2:.2f}s")

        return {
            **best_params,
            "Window":     f"{test_start}→{test_end}",
            "TrainScore": best_score,
            "TestScore":  test_score,
        }

    # =========================================================================
    # 5) HEADER
    # =========================================================================
    _vprint()
    _vprint("=" * 72)
    _vprint("WALK-FORWARD OPTIMIZATION  (grid vettorizzata)")
    _vprint("=" * 72)
    _vprint(f"  Dati         : {data_min.date()} → {data_max.date()}")
    _vprint(f"  Analisi      : {a_start.date()} → {a_end.date()}")
    _vprint(f"  Ratio        : {ratio}  (train={train_y}a, test={test_y}a)")
    _vprint(f"  Metric       : {metric}")
    _vprint(f"  Windows      : {n_windows}")
    _vprint(f"  Combinations : {n_combo:,}")
    _vprint(f"  Parallel     : {'SEQUENTIAL' if n_jobs == 1 else f'n_jobs={n_jobs} (eff={n_jobs_eff}), backend={backend}'}")
    _mom_lbs = sorted(set(int(v) for v in param_grid.get("momentum_lookback_days", [])))
    _vprint(f"  Mom lookbacks : {_mom_lbs}")
    if debug:
        _vprint("  [DEBUG MODE ON]")
    _vprint("=" * 72)
    _vprint()

    # =========================================================================
    # 6) ESECUZIONE
    # =========================================================================
    results = []
    t0_wfo  = time.time()

    # ── SEQUENZIALE ──────────────────────────────────────────────────────────
    if n_jobs == 1:
        pbar = tqdm(
            test_periods, desc="WFO Windows", position=0, leave=True,
            bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]',
        )
        for test_year in pbar:
            t0_w = time.time()
            pbar.set_description(
                f"WFO  {test_year-train_y}–{test_year-1} → {test_year}–{test_year+test_y-1}"
            )
            result    = _optimize_window(test_year, show_inner=True)
            elapsed_w = time.time() - t0_w
            if result:
                results.append(result)
                _vprint(
                    f"  ✓  {result['Window']:<28} "
                    f"Train={result['TrainScore']:+.3f}  "
                    f"Test={result['TestScore']:+.3f}  ({elapsed_w:.0f}s)"
                )
            else:
                _vprint(f"  ✗  {test_year}: fallita ({elapsed_w:.0f}s)")
        pbar.close()

    # ── PARALLELA ─────────────────────────────────────────────────────────────
    else:
        n_done = n_fail = 0

        pbar = tqdm(
            total=n_windows, desc="WFO Parallel", position=0, leave=True,
            bar_format=(
                '{desc}: {percentage:3.0f}%|{bar}| '
                '{n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]'
            ),
        )

        gen = Parallel(
            n_jobs=n_jobs, backend=backend, verbose=0,
            return_as="generator",
        )(
            delayed(_optimize_window)(yr, show_inner=False)
            for yr in test_periods
        )

        for result in gen:
            n_done += 1
            avg_s = (time.time() - t0_wfo) / n_done

            if result is not None:
                results.append(result)
                _vprint(
                    f"  ✓  {result['Window']:<28} "
                    f"Train={result['TrainScore']:+.3f}  "
                    f"Test={result['TestScore']:+.3f}  "
                    f"(avg {avg_s:.0f}s/win)"
                )
            else:
                n_fail += 1
                _vprint(f"  ✗  window {n_done}/{n_windows}: fallita")

            pbar.set_postfix_str(
                f"ok={n_done-n_fail}  fail={n_fail}  avg={avg_s:.0f}s/win",
                refresh=True,
            )
            pbar.update(1)

        pbar.close()

    # =========================================================================
    # 7) FOOTER
    # =========================================================================
    elapsed = time.time() - t0_wfo
    n_ok    = len(results)
    n_fail  = n_windows - n_ok

    _vprint()
    _vprint("=" * 72)
    _vprint(f"  Completata in {elapsed:.0f}s  ({elapsed/60:.1f} min)")
    _vprint(f"  Windows OK  : {n_ok}/{n_windows}")
    if n_fail:
        _vprint(f"  Windows KO  : {n_fail}  (dati insufficienti nel periodo di train — normale per le prime finestre)")
    _vprint("=" * 72)
    _vprint()

    if not results:
        raise ValueError("Nessuna finestra WFO completata con successo")

    return pd.DataFrame(results).set_index("Window").sort_index()  
    
def walk_forward_rotational_R2(
    stocks_data: pd.DataFrame,
    benchmark_data: pd.Series,
    param_grid: Dict[str, List[Any]],
    ratio: str = "3:1",
    metric: str = "Sharpe Ratio",
    verbose: bool = True,
    plot: bool = False,
    force_next_year_params: bool = False,
    start_date: str | None = None,
    end_date: str | None = None,
    n_jobs: int = -1,
    backend: str = 'loky',
    vol_cache: dict | None = None,
    debug: bool = False,
) -> pd.DataFrame:
    """
    Walk-Forward Optimization con grid search vettorizzata.

    ARCHITETTURA E PERFORMANCE
    --------------------------
    Il collo di bottiglia della WFO classica è il ricalcolo degli indicatori
    (momentum, volatilità, rank) per ogni combinazione di parametri. Con 4608
    combo × 9 finestre questo significa ~40.000 ricalcoli dello stesso dato.

    Questa implementazione separa nettamente i due costi:

      1. INDICATORI (O(finestre × giorni × asset) — fatto UNA VOLTA per finestra)
         - Tutti i lookback distinti nel param_grid vengono precomputati in batch.
         - Es: se momentum_lookback_days=[60,120,252], si calcolano 3 mom_df,
           non 4608.

      2. SELEZIONE (O(combo × rebal_dates) — il vero loop di ottimizzazione)
         - Per ogni combo si eseguono solo: lookup negli indicatori precalcolati,
           applicazione filtri, nlargest(n_top) per ogni rebal_date.
         - Nessuna costruzione VBT nel loop di train.

      3. VBT PORTFOLIO (costruito solo per il BEST params — 1 volta per finestra)
         - Solo per calcolare il test score.

    Speedup atteso: 10-50x rispetto alla versione che chiama
    build_rotational_portfolios_vbt per ogni combo.

    PROGRESS BAR (funzionante in Jupyter)
    --------------------------------------
    Modalità parallela: usa return_as="generator" (joblib >= 1.2).
    Testato con joblib 1.4.2, tqdm 4.67.1, 12 core.
    La barra avanza di 1 per ogni finestra completata.
    verbose=True stampa una riga per finestra (Window, TrainScore, TestScore)
    DURANTE l'esecuzione — non solo alla fine.

    PARAMETRI
    ---------
    stocks_data              : pd.DataFrame   Prezzi giornalieri.
    benchmark_data           : pd.Series      Prezzi benchmark.
    param_grid               : dict           Griglia parametri.
    ratio                    : str            Train:test in anni (es. '3:1').
    metric                   : str            'Sharpe Ratio' | 'CAGR' | 'Calmar'.
    verbose                  : bool           Header, footer, riga per finestra.
    plot                     : bool           Plot portfolio (solo best params, test).
    force_next_year_params   : bool           Aggiunge finestra futura.
    start_date / end_date    : str | None     Limiti analisi.
    n_jobs                   : int            -1=tutti, 1=sequenziale, N=N core.
    backend                  : str            'loky' | 'threading' | 'multiprocessing'.
    vol_cache                : dict | None    {lookback: vol_df} da precalculate_volatility_multiwindow.
    debug                    : bool           Score ogni combo, best params, stack trace errori.

    RETURNS
    -------
    pd.DataFrame  Index='Window', colonne=param_names + TrainScore + TestScore.
    """
    import sys
    import traceback as _tb
    import os

    # =========================================================================
    # 1) VALIDAZIONE E SETUP
    # =========================================================================
    try:
        train_y, test_y = [int(x) for x in ratio.split(":")]
    except Exception:
        raise ValueError(f"ratio non valido: '{ratio}'. Formato: 'train:test' es. '3:1'")
    if train_y <= 0 or test_y <= 0:
        raise ValueError("train e test devono essere > 0")
    if stocks_data.empty:
        raise ValueError("stocks_data è vuoto")

    stocks_data    = stocks_data.sort_index()
    benchmark_data = benchmark_data.sort_index()
    data_min = stocks_data.index.min()
    data_max = stocks_data.index.max()

    a_start = pd.Timestamp(start_date).normalize() if start_date else pd.Timestamp(data_min).normalize()
    a_end   = (pd.Timestamp(end_date) - pd.Timedelta(days=1)).normalize() if end_date else pd.Timestamp(data_max).normalize()

    if a_end < data_min or a_start > data_max:
        raise ValueError("Finestra di analisi fuori dal range dati")

    n_jobs_eff = os.cpu_count() if n_jobs == -1 else abs(n_jobs) if n_jobs < -1 else max(1, n_jobs)

    def _vprint(*args, **kw):
        if verbose or debug:
            print(*args, **kw)
            sys.stdout.flush()

    def _dprint(*args, **kw):
        if debug:
            print("[DEBUG]", *args, **kw)
            sys.stdout.flush()

    # =========================================================================
    # 2) BUFFER E FINESTRE
    # =========================================================================
    def _max_grid(key):
        vals = [int(v) for v in param_grid.get(key, []) if v is not None and np.isfinite(float(v))]
        return max(vals) if vals else 0

    max_lb     = max(_max_grid(k) for k in ["momentum_lookback_days","riskparity_lookback_days","ema_span","vol_window"])
    buffer_days = max(int(max_lb * 2 + 30), 60)

    first_test_y = a_start.year + train_y
    last_test_y  = a_end.year - (test_y - 1)
    if first_test_y > last_test_y:
        raise ValueError(f"Dati insufficienti per ratio={ratio} nel range {a_start.year}→{a_end.year}")

    test_periods = list(range(first_test_y, last_test_y + 1, test_y))
    if force_next_year_params and test_periods:
        test_periods.append(test_periods[-1] + test_y)

    n_windows = len(test_periods)
    param_keys = list(param_grid.keys())
    all_combos = list(product(*param_grid.values()))
    n_combo    = len(all_combos)

    # =========================================================================
    # 3) HELPER: score da returns
    # =========================================================================
    def _score(ret: pd.Series) -> dict:
        ret = ret.dropna()
        if ret.empty:
            return {"Sharpe Ratio": np.nan, "CAGR": np.nan, "Calmar": np.nan}
        ann_r = (1 + ret.mean()) ** 252 - 1
        ann_v = ret.std(ddof=0) * np.sqrt(252)
        shrp  = ann_r / ann_v if ann_v and np.isfinite(ann_v) else np.nan
        cagr  = (1 + ret).prod() ** (252 / max(len(ret), 1)) - 1
        eq    = (1 + ret).cumprod()
        dd    = (eq / eq.cummax() - 1).min()
        cal   = cagr / abs(dd) if dd and np.isfinite(dd) else np.nan
        return {"Sharpe Ratio": shrp, "CAGR": cagr, "Calmar": cal}

    # =========================================================================
    # 4) CORE: ottimizzazione singola finestra CON INDICATORI PRECALCOLATI
    # =========================================================================
    def _optimize_window(test_start_year: int, show_inner: bool = False):
        train_start = f"{test_start_year - train_y}-01-01"
        train_end   = f"{test_start_year - 1}-12-31"
        test_start  = f"{test_start_year}-01-01"
        test_end    = f"{test_start_year + test_y - 1}-12-31"

        # Slice dati con buffer
        buf_start = pd.Timestamp(train_start) - pd.Timedelta(days=buffer_days)
        tr_px  = stocks_data.loc[buf_start:train_end].dropna(axis=1, how='all').ffill().bfill()
        tr_bch = benchmark_data.loc[buf_start:train_end]

        if tr_px.empty or stocks_data.loc[train_start:train_end].empty:
            _dprint(f"SKIP {train_start}→{train_end}: dati insufficienti")
            _vprint(f"  ↳  {test_start}→{test_end}: skip (train {train_start[:4]}–{train_end[:4]} fuori range dati)")
            return None

        cols = tr_px.columns
        idx  = tr_px.index

        # ── PRECALCOLO INDICATORI (UNA VOLTA per finestra) ────────────────
        # Raccoglie tutti i lookback distinti dal param_grid e li calcola in batch.
        # Es: momentum_lookback_days=[60,120,252] → 3 calcoli, non n_combo calcoli.

        rets = tr_px.pct_change().fillna(0.0)

        # momentum: uno per ogni lookback distinto
        mom_lbs = sorted(set(
            int(c[param_keys.index('momentum_lookback_days')])
            for c in all_combos
            if 'momentum_lookback_days' in param_keys
        )) if 'momentum_lookback_days' in param_keys else [126]

        # volatilità: uno per ogni rp_lookback distinto
        rp_lbs = sorted(set(
            int(c[param_keys.index('riskparity_lookback_days')])
            for c in all_combos
            if 'riskparity_lookback_days' in param_keys
        )) if 'riskparity_lookback_days' in param_keys else [20]

        # ema span: uno per ogni span distinto
        ema_spans = sorted(set(
            int(c[param_keys.index('ema_span')])
            for c in all_combos
            if 'ema_span' in param_keys
        )) if 'ema_span' in param_keys else []

        # momentum_weight values (non richiedono precalcolo separato)
        # n_top values (non richiedono precalcolo separato)

        # calcola tutti i mom_df necessari
        mom_cache: dict[int, pd.DataFrame] = {}
        for lb in mom_lbs:
            m = tr_px / tr_px.shift(lb)
            mom_cache[lb] = m.rank(pct=True, axis=1, na_option='bottom')

        # calcola tutti i vol_df necessari
        ivol_cache: dict[int, pd.DataFrame] = {}
        
        # for lb in rp_lbs:
        #     if vol_cache is not None and lb in vol_cache:
        #         v = vol_cache[lb].reindex(idx)
        #     else:
        #         v = rets.rolling(lb, min_periods=1).std(ddof=0)
        #     iv = (1.0 / v.replace(0.0, np.nan))
        #     ivol_cache[lb] = iv.rank(pct=True, axis=1, na_option='bottom')

        for lb in rp_lbs:
            if (vol_cache is not None 
                    and lb in vol_cache 
                    and idx[0] in vol_cache[lb].index 
                    and idx[-1] in vol_cache[lb].index):
                v = vol_cache[lb].reindex(idx).ffill().bfill()
                nan_frac = v.isna().mean().mean()
                if nan_frac > 0.05:          # >5% NaN → ricalcola
                    v = rets.rolling(lb, min_periods=1).std(ddof=0)
            else:
                v = rets.rolling(lb, min_periods=1).std(ddof=0)
            iv = (1.0 / v.replace(0.0, np.nan))
            ivol_cache[lb] = iv.rank(pct=True, axis=1, na_option='bottom')
        
        # calcola ema se necessario
        ema_cache: dict[int, pd.DataFrame] = {}
        for sp in ema_spans:
            ema_cache[sp] = tr_px.ewm(span=sp, adjust=False, min_periods=1).mean()

        # rebal dates per il periodo di train
        tr_idx = tr_px.loc[train_start:train_end].index
        rebal_freq = next(
            (c[param_keys.index('rebalance_frequency')] for c in all_combos
             if 'rebalance_frequency' in param_keys),
            'ME'
        )
        try:
            rebal_dates = compute_rebal_dates(tr_idx, str(rebal_freq).upper())
            rebal_dates = pd.DatetimeIndex([d for d in rebal_dates if d in tr_idx])
        except Exception:
            # fallback mensile
            rebal_dates = pd.DatetimeIndex(
                pd.Series(tr_idx).groupby(tr_idx.to_period('M')).max().values
            )

        if len(rebal_dates) == 0:
            _dprint(f"SKIP {train_start}→{train_end}: nessuna rebal_date")
            return None

        _dprint(f"Finestra {train_start}→{train_end}: "
                f"{len(rebal_dates)} rebal_dates, {len(cols)} asset, "
                f"mom_lbs={mom_lbs}, rp_lbs={rp_lbs}")

        # ── GRID SEARCH — numpy puro per massima velocità ────────────────
        #
        # Ottimizzazione chiave: precalcola le matrici di returns per periodo
        # come array numpy. Ogni combo diventa:
        #   1. lookup combo_score (già in cache come DataFrame)
        #   2. nlargest per ogni rebal_date → indici colonna numpy
        #   3. concatenazione array numpy → sharpe/cagr senza pandas overhead
        #
        # Questo elimina il collo di bottiglia precedente: N_combo × N_rebal_dates
        # chiamate a rets.loc[d:d_next, top].mean(axis=1) (pandas slice = ~1ms/call).
        # Con numpy: ~0.01ms/call → speedup 50-100x sulla grid search.

        rets_np  = rets.values.astype(np.float64)          # (n_days, n_assets)
        date_pos = {d: i for i, d in enumerate(rets.index)} # O(1) lookup

        rebal_list = list(rebal_dates)
        n_rebal    = len(rebal_list)

        # Precomputa indici interi per ogni periodo [d, d_next)
        period_slices = []   # list of (start_i, end_i)
        for i, d in enumerate(rebal_list):
            d_next  = rebal_list[i + 1] if i + 1 < n_rebal else rets.index[-1]
            start_i = date_pos.get(d, 0)
            end_i   = date_pos.get(d_next, len(rets) - 1) + 1
            period_slices.append((start_i, end_i))

        # Per i filtri opzionali: numpy arrays per rebal_date
        # (accessibili per indice intero invece di .loc[d])
        rebal_pos_in_full = [date_pos.get(d, 0) for d in rebal_list]

        best_score  = -np.inf
        best_params = None

        pbar_i = None
        if show_inner:
            pbar_i = tqdm(
                total=n_combo,
                desc=f"  Grid {test_start_year}",
                position=1, leave=False,
                bar_format='{desc}: {percentage:3.0f}%|{bar}|{n_fmt}/{total_fmt} [{elapsed},{rate_fmt}]',
            )

        col_list = list(cols)
        n_assets = len(col_list)

        for combo in all_combos:
            params = dict(zip(param_keys, combo))

            try:
                mom_lb = int(params.get('momentum_lookback_days', 126))
                rp_lb  = int(params.get('riskparity_lookback_days', 20))
                n_top  = int(params.get('n_top', 5))
                mw     = float(params.get('momentum_weight', 0.7))
                f_ema  = bool(params.get('filter_ema', False))
                f_vol  = bool(params.get('filter_volatility', False))
                f_mom  = bool(params.get('filter_min_momentum', False))
                ema_sp = int(params.get('ema_span', 200))
                vol_q  = float(params.get('volatility_quantile', 0.75))
                min_m  = float(params.get('min_momentum_threshold', 1.0))

                rm  = mom_cache.get(mom_lb)
                riv = ivol_cache.get(rp_lb)
                if rm is None or riv is None:
                    continue

                combo_score = (mw * rm + (1.0 - mw) * riv).values  # (n_days, n_assets) numpy

                # Filtri opzionali come maschere numpy per ogni rebal_date
                ema_arr = ema_cache[ema_sp].values if f_ema and ema_sp in ema_cache else None
                px_arr  = tr_px.values

                pf_chunks = []
                prev_top_ci = None   # indici colonna numpy

                for k, (ri, (start_i, end_i)) in enumerate(zip(rebal_pos_in_full, period_slices)):
                    cs = combo_score[ri].copy()  # shape (n_assets,)

                    # filtri
                    if ema_arr is not None:
                        cs = np.where(px_arr[ri] > ema_arr[ri], cs, np.nan)
                    if f_vol:
                        # ivol_cache contiene rank, non vol raw: usiamo threshold quantile sui rank
                        ivol_row = ivol_cache[rp_lb].values[ri]
                        q_thresh = np.nanquantile(ivol_row, 1.0 - vol_q)
                        cs = np.where(ivol_row >= q_thresh, cs, np.nan)
                    if f_mom:
                        shift_i  = max(0, ri - mom_lb)
                        raw_mom  = px_arr[ri] / np.where(px_arr[shift_i] > 0, px_arr[shift_i], np.nan)
                        cs = np.where(raw_mom > min_m, cs, np.nan)

                    valid_mask = ~np.isnan(cs)
                    if valid_mask.sum() == 0:
                        top_ci = prev_top_ci
                    else:
                        top_ci = np.argsort(cs[valid_mask])[::-1]
                        # mappa da indici-filtered a indici-full
                        valid_indices = np.where(valid_mask)[0]
                        top_ci = valid_indices[top_ci[:n_top]]
                        prev_top_ci = top_ci

                    if top_ci is None or len(top_ci) == 0:
                        continue

                    # returns del periodo: numpy slice O(1)
                    chunk = rets_np[start_i:end_i, :][:, top_ci]   # (days_in_period, n_top)
                    if chunk.size == 0:
                        continue
                    pf_chunks.append(chunk.mean(axis=1))            # equal weight

                if not pf_chunks:
                    continue

                pf_ret = np.concatenate(pf_chunks)
                if len(pf_ret) == 0:
                    continue

                # score direttamente in numpy (nessun pd.Series overhead)
                ann_r = (1.0 + pf_ret.mean()) ** 252 - 1.0
                ann_v = pf_ret.std(ddof=0) * np.sqrt(252)
                if metric == "Sharpe Ratio":
                    sc = ann_r / ann_v if ann_v > 0 and np.isfinite(ann_v) else np.nan
                elif metric == "CAGR":
                    sc = float(np.prod(1.0 + pf_ret) ** (252 / max(len(pf_ret), 1)) - 1.0)
                elif metric == "Calmar":
                    cagr = float(np.prod(1.0 + pf_ret) ** (252 / max(len(pf_ret), 1)) - 1.0)
                    eq   = np.cumprod(1.0 + pf_ret)
                    dd   = np.min(eq / np.maximum.accumulate(eq) - 1.0)
                    sc   = cagr / abs(dd) if dd != 0 and np.isfinite(dd) else np.nan
                else:
                    sc = np.nan

                _dprint(f"  {params}  {metric}={sc:.4f}" if np.isfinite(sc) else f"  {params}  {metric}=NaN")

                if np.isfinite(sc) and sc > best_score:
                    best_score, best_params = sc, params

            except Exception as exc:
                if debug:
                    print(f"[DEBUG] EXCEPTION combo={params}: {exc}")
                    _tb.print_exc()

            if pbar_i:
                pbar_i.update(1)

        if pbar_i:
            pbar_i.close()

        if best_params is None:
            _dprint(f"NO VALID PARAMS: {train_start}→{train_end}")
            return None

        _dprint(f"BEST {train_start}→{train_end}: {metric}={best_score:.4f} params={best_params}")

        # ── TEST: costruisce VBT solo con best params (1 volta) ──────────
        test_score = np.nan
        if not stocks_data.loc[test_start:test_end].empty:
            try:
                buf_s   = pd.Timestamp(test_start) - pd.Timedelta(days=buffer_days)
                te_px   = stocks_data.loc[buf_s:test_end]
                te_bch  = benchmark_data.loc[buf_s:test_end]
                pf_te, *_ = build_rotational_portfolios_vbt(
                    stocks_data=te_px,
                    benchmark_data=te_bch,
                    plot=plot,
                    vol_cache=vol_cache,
                    **best_params,
                )
                test_score = _score(
                    pf_te.returns().loc[test_start:test_end]
                ).get(metric, np.nan)
            except Exception as exc:
                if debug:
                    print(f"[DEBUG] TEST exception: {exc}")
                    _tb.print_exc()

        return {
            **best_params,
            "Window":     f"{test_start}→{test_end}",
            "TrainScore": best_score,
            "TestScore":  test_score,
        }

    # =========================================================================
    # 5) HEADER
    # =========================================================================
    _vprint()
    _vprint("=" * 72)
    _vprint("WALK-FORWARD OPTIMIZATION  (grid vettorizzata)")
    _vprint("=" * 72)
    _vprint(f"  Dati         : {data_min.date()} → {data_max.date()}")
    _vprint(f"  Analisi      : {a_start.date()} → {a_end.date()}")
    _vprint(f"  Ratio        : {ratio}  (train={train_y}a, test={test_y}a)")
    _vprint(f"  Metric       : {metric}")
    _vprint(f"  Windows      : {n_windows}")
    _vprint(f"  Combinations : {n_combo:,}")
    _vprint(f"  Parallel     : {'SEQUENTIAL' if n_jobs == 1 else f'n_jobs={n_jobs} (eff={n_jobs_eff}), backend={backend}'}")
    _mom_lbs = sorted(set(int(v) for v in param_grid.get("momentum_lookback_days", [])))
    _vprint(f"  Mom lookbacks : {_mom_lbs}")
    if vol_cache:
        _vprint(f"  Vol cache    : {sorted(vol_cache.keys())}")
    if debug:
        _vprint("  [DEBUG MODE ON]")
    _vprint("=" * 72)
    _vprint()

    # =========================================================================
    # 6) ESECUZIONE
    # =========================================================================
    results   = []
    t0_wfo    = time.time()

    # ── SEQUENZIALE ──────────────────────────────────────────────────────────
    if n_jobs == 1:
        pbar = tqdm(
            test_periods, desc="WFO Windows", position=0, leave=True,
            bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]',
        )
        for test_year in pbar:
            t0_w = time.time()
            pbar.set_description(
                f"WFO  {test_year-train_y}–{test_year-1} → {test_year}–{test_year+test_y-1}"
            )
            result = _optimize_window(test_year, show_inner=True)
            elapsed_w = time.time() - t0_w
            if result:
                results.append(result)
                _vprint(
                    f"  ✓  {result['Window']:<28} "
                    f"Train={result['TrainScore']:+.3f}  "
                    f"Test={result['TestScore']:+.3f}  ({elapsed_w:.0f}s)"
                )
            else:
                _vprint(f"  ✗  {test_year}: fallita ({elapsed_w:.0f}s)")
        pbar.close()

    # ── PARALLELA ─────────────────────────────────────────────────────────────
    else:
        # return_as="generator": i risultati arrivano in streaming man mano
        # che ogni worker finisce. Testato: joblib 1.4.2, latenza primo result
        # ~0.7s, aggiornamenti incrementali confermati (vedere diagnostic output).
        n_done = n_fail = 0

        pbar = tqdm(
            total=n_windows, desc="WFO Parallel", position=0, leave=True,
            bar_format=(
                '{desc}: {percentage:3.0f}%|{bar}| '
                '{n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]'
            ),
        )

        gen = Parallel(
            n_jobs=n_jobs, backend=backend, verbose=0,
            return_as="generator",
        )(
            delayed(_optimize_window)(yr, show_inner=False)
            for yr in test_periods
        )

        for result in gen:
            n_done += 1
            avg_s = (time.time() - t0_wfo) / n_done

            if result is not None:
                results.append(result)
                _vprint(
                    f"  ✓  {result['Window']:<28} "
                    f"Train={result['TrainScore']:+.3f}  "
                    f"Test={result['TestScore']:+.3f}  "
                    f"(avg {avg_s:.0f}s/win)"
                )
            else:
                n_fail += 1
                _vprint(f"  ✗  window {n_done}/{n_windows}: fallita")

            pbar.set_postfix_str(
                f"ok={n_done-n_fail}  fail={n_fail}  avg={avg_s:.0f}s/win",
                refresh=True,
            )
            pbar.update(1)

        pbar.close()

    # =========================================================================
    # 7) FOOTER
    # =========================================================================
    elapsed = time.time() - t0_wfo
    n_ok    = len(results)
    n_fail  = n_windows - n_ok

    _vprint()
    _vprint("=" * 72)
    _vprint(f"  Completata in {elapsed:.0f}s  ({elapsed/60:.1f} min)")
    _vprint(f"  Windows OK  : {n_ok}/{n_windows}")
    if n_fail:
        _vprint(f"  Windows KO  : {n_fail}  (dati insufficienti nel periodo di train — normale per le prime finestre)")
    _vprint("=" * 72)
    _vprint()

    if not results:
        raise ValueError("Nessuna finestra WFO completata con successo")

    return pd.DataFrame(results).set_index("Window").sort_index()
    
def walk_forward_rotational_R1(
    stocks_data: pd.DataFrame,
    benchmark_data: pd.Series,
    param_grid: Dict[str, List[Any]],
    ratio: str = "3:1",
    metric: str = "Sharpe Ratio",
    verbose: bool = True,
    plot: bool = False,
    force_next_year_params: bool = False,
    start_date: str | None = None,
    end_date: str | None = None,
    n_jobs: int = -1,
    backend: str = 'loky',    
    vol_cache: dict | None = None,  # ← TUA MODIFICA PRESERVATA
) -> pd.DataFrame:
    """
    Walk-Forward Optimization PARALLELIZZATA con PROGRESS BARS e vol_cache support.
    
    Speedup 4-8x su CPU multi-core con feedback visivo dettagliato.
    
    PROGRESS BARS:
    --------------
    Modalità SEQUENZIALE (n_jobs=1):
    - Progress bar finestre WFO (outer loop)
    - Progress bar combinazioni grid PER FINESTRA (inner loop) ← NUOVO
    - Mostra Train/Test years per finestra
    
    Modalità PARALLELA (n_jobs=-1):
    - Progress bar principale (finestre completate)
    - Progress bar per ogni WORKER attivo (grid search) ← NUOVO
    - Update real-time con timing
    - Summary finale
    
    STRATEGIA PARALLELIZZAZIONE:
    - Level 1: Parallelizza FINESTRE WFO (indipendenti)
    - Level 2: Ogni finestra ottimizza griglia sequenzialmente
    
    Parametri
    ----------
    stocks_data : pd.DataFrame
        Prezzi stocks (daily)
    benchmark_data : pd.Series
        Prezzi benchmark
    param_grid : dict
        Griglia parametri da ottimizzare
    ratio : str, default="3:1"
        Ratio train:test in anni
    metric : str, default="Sharpe Ratio"
        Metrica ottimizzazione ("Sharpe Ratio", "CAGR", "Calmar")
    verbose : bool, default=True
        Se True, stampa info dettagliate
    plot : bool, default=False
        Plot risultati (passato a build function)
    force_next_year_params : bool, default=False
        Se True, aggiunge finestra futura
    start_date : str, optional
        Data inizio analisi
    end_date : str, optional
        Data fine analisi
    n_jobs : int, default=-1
        Numero core paralleli:
        -1 = tutti disponibili
        1 = sequenziale (con progress bars nested)
        2-8 = numero specifico cores
    backend : str, default='loky'
        Backend joblib:
        'loky': raccomandato, processi indipendenti
        'threading': threads (GIL limit)
        'multiprocessing': processi (più overhead)
    vol_cache : dict, optional
        Cache volatility pre-calcolata da precalculate_volatility_multiwindow()
        Se fornito, passa a build_rotational_portfolios_vbt per speedup
        
    Returns
    -------
    pd.DataFrame
        Summary WFO con best params per finestra
        Index: "Window" (es. "2023-01-01→2023-12-31")
        Columns: param names + TrainScore + TestScore
        
    Esempio
    -------
    >>> from optimizations import (
    ...     walk_forward_rotational_parallel,
    ...     precalculate_volatility_multiwindow
    ... )
    >>> 
    >>> # Pre-calcola vol cache (opzionale ma raccomandato)
    >>> vol_cache = precalculate_volatility_multiwindow(
    ...     stocks_data,
    ...     windows=[10, 20, 60]
    ... )
    >>> 
    >>> # WFO parallela con cache
    >>> summary = walk_forward_rotational_parallel(
    ...     stocks_data=stocks_data,
    ...     benchmark_data=benchmark_data,
    ...     param_grid={
    ...         'momentum_lookback_days': [60, 120],
    ...         'riskparity_lookback_days': [20, 60],
    ...         'n_top': [3, 5],
    ...         'momentum_weight': [0.5, 0.7]
    ...     },
    ...     ratio="3:1",
    ...     n_jobs=-1,
    ...     vol_cache=vol_cache  # ← passa cache per speedup
    ... )
    >>> 
    >>> print(summary)
    
    Note
    ----
    - Su 6-core CPU: ~5x speedup vs sequenziale
    - Su 12-core CPU: ~7x speedup
    - vol_cache aggiunge ulteriore ~2x speedup se usi filtri volatility
    - Overhead significativo se grid < 100 combos
    - Progress bars funzionano in Jupyter e terminale
    """
    
    # =========================================================================
    # SETUP PARAMETRI WFO
    # =========================================================================
    
    # Decode ratio
    try:
        train_years_str, test_years_str = ratio.split(":")
        train_years = int(train_years_str)
        test_years = int(test_years_str)
    except Exception:
        raise ValueError(f"ratio non valido: '{ratio}'. Formato atteso 'train:test'")
    
    if train_years <= 0 or test_years <= 0:
        raise ValueError("train_years e test_years devono essere > 0")
    
    if stocks_data.empty:
        raise ValueError("stocks_data è vuoto")
    
    # Normalizza indici
    stocks_data = stocks_data.sort_index()
    benchmark_data = benchmark_data.sort_index()
    
    data_min = stocks_data.index.min()
    data_max = stocks_data.index.max()
    
    # Finestra analisi
    analysis_start = pd.Timestamp(start_date) if start_date else pd.Timestamp(data_min).normalize()
    analysis_end = pd.Timestamp(end_date) - pd.Timedelta(days=1) if end_date else pd.Timestamp(data_max).normalize()
    
    if analysis_end < data_min or analysis_start > data_max:
        raise ValueError("Finestra di analisi fuori dal range dati disponibili")
    
    analysis_start_year = analysis_start.year
    analysis_end_year = analysis_end.year
    
    # =========================================================================
    # BUFFER LOOKBACK
    # =========================================================================
    def _max_int_in_grid(key: str) -> int:
        vals = param_grid.get(key, [])
        ints = [int(v) for v in vals if v is not None and np.isfinite(v)]
        return max(ints) if ints else 0
    
    max_mom_lb = _max_int_in_grid("momentum_lookback_days")
    max_rp_lb = _max_int_in_grid("riskparity_lookback_days")
    max_ema = _max_int_in_grid("ema_span")
    max_vol = _max_int_in_grid("vol_window")
    
    max_lookback = max(max_mom_lb, max_rp_lb, max_ema, max_vol, 0)
    buffer_days = int(max_lookback * 2 + 30) if max_lookback > 0 else 60
    
    # =========================================================================
    # TEST PERIODS
    # =========================================================================
    first_test_year = analysis_start_year + train_years
    last_test_year = analysis_end_year - (test_years - 1)
    
    if first_test_year > last_test_year:
        raise ValueError(
            f"Finestra insufficiente: con train={train_years} e test={test_years}, "
            f"non ci sono anni OOS tra {analysis_start_year} e {analysis_end_year}."
        )
    
    test_periods = list(range(first_test_year, last_test_year + 1, test_years))
    
    if force_next_year_params and test_periods:
        test_periods.append(test_periods[-1] + test_years)
    
    # =========================================================================
    # GRIGLIA PARAMETRI
    # =========================================================================
    param_keys = list(param_grid.keys())
    all_combos = list(product(*param_grid.values()))
    n_combo = len(all_combos)
    
    # =========================================================================
    # HELPER FUNCTIONS
    # =========================================================================
    
    def _compute_score_from_returns(ret: pd.Series) -> Dict[str, float]:
        """Calcola metriche da serie returns."""
        ret = ret.dropna()
        if ret.empty:
            return {"Sharpe Ratio": np.nan, "CAGR": np.nan, "Calmar": np.nan}
        
        ann_ret = (1 + ret.mean()) ** 252 - 1
        ann_vol = ret.std(ddof=0) * np.sqrt(252)
        sharpe = ann_ret / ann_vol if ann_vol and np.isfinite(ann_vol) else np.nan
        
        cagr = (1 + ret).prod() ** (252 / max(len(ret), 1)) - 1
        
        eq = (1 + ret).cumprod()
        dd = (eq / eq.cummax() - 1).min()
        calmar = cagr / abs(dd) if dd != 0 else np.nan
        
        return {"Sharpe Ratio": sharpe, "CAGR": cagr, "Calmar": calmar}
    
    def _slice_with_buffer(start: str, end: str) -> pd.DataFrame:
        """Slice stocks_data con buffer lookback."""
        start_ts = pd.Timestamp(start)
        end_ts = pd.Timestamp(end)
        calc_start = start_ts - pd.Timedelta(days=buffer_days)
        return stocks_data.loc[calc_start:end_ts]
    
    def _slice_bench_with_buffer(start: str, end: str) -> pd.Series:
        """Slice benchmark_data con buffer lookback."""
        start_ts = pd.Timestamp(start)
        end_ts = pd.Timestamp(end)
        calc_start = start_ts - pd.Timedelta(days=buffer_days)
        return benchmark_data.loc[calc_start:end_ts]
    
    # =========================================================================
    # FUNZIONE OTTIMIZZAZIONE SINGOLA FINESTRA con PROGRESS BAR INTERNA
    # =========================================================================
    
    def _optimize_single_window(
        test_start_year: int,
        window_idx: int,
        n_windows: int,
        vol_cache: dict | None,
        show_inner_progress: bool = False,  # ← Default FALSE (solo sequential lo attiva)
    ) -> Dict[str, Any]:
        """
        Ottimizza griglia su singola finestra WF con progress bar interna opzionale.
        
        Questa funzione è ISOLATA e può essere eseguita in parallelo.
        La progress bar interna è DISABILITATA di default per evitare confusione in parallel mode.
        """
        
        train_start = f"{test_start_year - train_years}-01-01"
        train_end = f"{test_start_year - 1}-12-31"
        test_start = f"{test_start_year}-01-01"
        test_end = f"{test_start_year + test_years - 1}-12-31"
        
        # Verifica train non vuoto
        train_eval = stocks_data.loc[train_start:train_end]
        if train_eval.empty:
            return None
        
        # Dataset con buffer
        train_calc_stocks = _slice_with_buffer(train_start, train_end)
        train_calc_bench = _slice_bench_with_buffer(train_start, train_end)
        
        if train_calc_stocks.empty:
            return None
        
        # =====================================================================
        # GRID SEARCH con PROGRESS BAR INTERNA opzionale
        # =====================================================================
        best_score, best_params = -np.inf, None
        
        # Progress bar SOLO se richiesta (sequential mode)
        pbar_inner = None
        if show_inner_progress:
            pbar_inner = tqdm(
                total=n_combo,
                desc=f"  Grid [{test_start[:4]}-{test_end[:4]}]",
                position=1,  # sempre position 1 per sequential
                leave=False,
                bar_format='{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}, {rate_fmt}]'
            )
        
        for combo in all_combos:
            params = dict(zip(param_keys, combo))
            try:
                pf_train, *_ = build_rotational_portfolios_vbt(
                    stocks_data=train_calc_stocks,
                    benchmark_data=train_calc_bench,
                    portfolio_name="Train",
                    plot=False,
                    vol_cache=vol_cache,
                    **params,
                )
                
                ret_train = pf_train.returns().loc[train_start:train_end]
                score = _compute_score_from_returns(ret_train).get(metric, np.nan)
                
                if np.isfinite(score) and score > best_score:
                    best_score, best_params = score, params
            
            except Exception:
                pass
            
            # Update progress bar se attiva
            if pbar_inner is not None:
                pbar_inner.update(1)
        
        # Chiudi progress bar se attiva
        if pbar_inner is not None:
            pbar_inner.close()
        
        if best_params is None:
            return None
        
        # TEST con best params
        test_score = np.nan
        test_eval = stocks_data.loc[test_start:test_end]
        
        if not test_eval.empty:
            try:
                test_calc_stocks = _slice_with_buffer(test_start, test_end)
                test_calc_bench = _slice_bench_with_buffer(test_start, test_end)
                
                pf_test, *_ = build_rotational_portfolios_vbt(
                    stocks_data=test_calc_stocks,
                    benchmark_data=test_calc_bench,
                    portfolio_name="Test",
                    plot=False,
                    vol_cache=vol_cache,
                    **best_params,
                )
                
                ret_test = pf_test.returns().loc[test_start:test_end]
                test_score = _compute_score_from_returns(ret_test).get(metric, np.nan)
            
            except Exception:
                test_score = np.nan
        
        return {
            **best_params,
            "Window": f"{test_start}→{test_end}",
            "TrainScore": best_score,
            "TestScore": test_score,
        }
    
    # =========================================================================
    # ESECUZIONE PARALLELA con PROGRESS BARS
    # =========================================================================
    
    n_windows = len(test_periods)
    total_iterations = n_windows * n_combo
    
    # Header info
    print()
    print("="*80)
    print("WALK-FORWARD OPTIMIZATION - PARALLEL")
    print("="*80)
    print(f"Windows: {n_windows}")
    print(f"Combinations per window: {n_combo:,}")
    print(f"Total iterations: {total_iterations:,}")
    if n_jobs == 1:
        parallel_msg = "SEQUENTIAL (n_jobs=1)"
    else:
        cores = n_jobs if n_jobs > 0 else "ALL cores"
        parallel_msg = f"PARALLEL ({cores}, backend={backend})"
    print(f"Parallelization: {parallel_msg}")
    # print(f"Parallelization: {'SEQUENTIAL (n_jobs=1)' if n_jobs == 1 else f'PARALLEL ({n_jobs if n_jobs > 0 else 'ALL cores'}, backend={backend})'}")
    if vol_cache is not None:
        print(f"Vol cache: ENABLED (windows: {list(vol_cache.keys())})")
    print("="*80)
    print()
    
    if n_jobs == 1:
        # =====================================================================
        # MODALITÀ SEQUENZIALE con PROGRESS BAR NESTED
        # =====================================================================
        results = []
        
        # Progress bar per finestre (outer)
        pbar_windows = tqdm(
            test_periods,
            desc="WFO Windows",
            position=0,
            leave=True,
            bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} windows [{elapsed}<{remaining}]'
        )
        
        for wi, test_year in enumerate(pbar_windows, start=1):
            # Update descrizione finestra con train/test years
            train_start = f"{test_year - train_years}-01-01"
            train_end = f"{test_year - 1}-12-31"
            test_start = f"{test_year}-01-01"
            test_end = f"{test_year + test_years - 1}-12-31"
            
            pbar_windows.set_description(
                f"Window [{wi}/{n_windows}] Train:{train_start[:4]}-{train_end[:4]} Test:{test_start[:4]}-{test_end[:4]}"
            )
            
            # Ottimizza finestra (con progress bar interna)
            result = _optimize_single_window(
                test_year, 
                wi, 
                n_windows, 
                vol_cache,
                show_inner_progress=True  # ← NUOVO: abilita progress bar interna
            )
            
            if result:
                results.append(result)
        
        pbar_windows.close()
        
    else:
        # =====================================================================
        # MODALITÀ PARALLELA con PROGRESS BAR PRINCIPALE + WORKER BARS
        # =====================================================================
        
        # Progress bar principale (tracks finestre completate)
        pbar_main = tqdm(
            total=n_windows,
            desc="WFO Parallel",
            position=0,
            leave=True,
            bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} windows [{elapsed}<{remaining}, {rate_fmt}]'
        )
        
        # Parallel execution con update incrementale
        # Progress bar INTERNA DISABILITATA per evitare confusione
        results = []
        
        with Parallel(n_jobs=n_jobs, backend=backend, verbose=0) as parallel:
            for result in parallel(
                delayed(_optimize_single_window)(
                    test_year, 
                    wi, 
                    n_windows, 
                    vol_cache,
                    show_inner_progress=False  # ← DISABILITATA in parallel
                )
                for wi, test_year in enumerate(test_periods, start=1)
            ):
                results.append(result)
                pbar_main.update(1)  # Update nel main thread dopo ricezione result
        
        pbar_main.close()
        
        # Filtra None (finestre fallite)
        results = [r for r in results if r is not None]
    
    # =========================================================================
    # SUMMARY FINALE
    # =========================================================================
    print()
    print("="*80)
    print(f"✅ WFO COMPLETED: {len(results)}/{n_windows} windows successful")
    print("="*80)
    print()
    
    if not results:
        raise ValueError("Nessuna combinazione valida trovata in nessuna finestra")
    
    return pd.DataFrame(results).set_index("Window")
    
def build_rotational_portfolios_vbt_R2(
    stocks_data: pd.DataFrame,
    benchmark_data: pd.Series | None = None,
    rebalance_frequency: str = "ME",
    momentum_lookback_days: int = 126,
    riskparity_lookback_days: int = 20,
    n_top: int = 5,
    momentum_weight: float = 0.7,
    filter_ema: bool = False,
    filter_volatility: bool = False,
    filter_min_momentum: bool = False,
    ema_span: int = 200,
    volatility_quantile: float = 0.75,
    min_momentum_threshold: float = 1.0,
    init_cash: float = 100_000,
    portfolio_name: str = "Portafoglio rotazionale",
    plot: bool = True,
    start_date: str | None = None,
    bottom_tickers: bool = False,
    use_acceleration: bool = False,    
    debug: bool = False,
    build_other_portfolios: bool = False,
    vol_cache: dict | None = None,
    vbt_plot_width: int | None = 1000,   # <-- FIX: era usato ma non presente in signature
):
    """
    Costruisce portafogli rotazionali con ribilanciamento trading-aligned.

    FIX CRITICO (2026-03):
    - Corretto il filtro "ultimo periodo incompleto" (YE/QE/ME/W*) con conteggio giorni robusto
      (evita bug Pandas Period freq mismatch che eliminava la rebal_date finale, es. 2023-12-29).

    Policy selezioni:
    - sel_tickers_rot_w è DENSE: 1 riga per ogni rebal_date.
    - se selezione vuota dopo filtri -> carry-forward ultima selezione valida (se esiste).
    - se rebal_dates vuoto -> fallback cash (pesi zero) e selezioni vuote.

    Output principale (sempre, stessa arità):
      - pf_rot_w
      - pf_bh   (None se benchmark_data=None)
      - sel_tickers_rot_w
      - rankings_df

    Output aggiuntivi (solo se build_other_portfolios=True):
      - pf_mom
      - pf_rp

    bottom_tickers=True aggiunge:
      - sel_tickers_rot_bottom
    """
    import numpy as np
    import pandas as pd
    import vectorbt as vbt

    def _dbg(msg: str):
        if debug:
            print(msg)

    if not (0.0 <= float(momentum_weight) <= 1.0):
        raise ValueError("momentum_weight deve essere compreso tra 0.0 e 1.0")

    # ==========================================================
    # 1) DATI BASE (PREZZI)
    # ==========================================================
    prices = stocks_data.dropna(axis=1, how="all").ffill().bfill().copy()
    prices.index = pd.to_datetime(prices.index).normalize()
    prices = prices.sort_index()

    cols = prices.columns
    idx_full = prices.index.unique()

    bench_px = None
    if benchmark_data is not None:
        bench_px = benchmark_data.copy()
        bench_px.index = pd.to_datetime(bench_px.index).normalize()
        bench_px = bench_px.sort_index()

    # returns
    rets = prices.pct_change().fillna(0.0)

    # ==========================================================
    # 2) REBALANCE DATES (TRADING-ALIGNED) + filtro "periodo incompleto" ROBUSTO
    # ==========================================================
    freq = str(rebalance_frequency).upper().strip()

    def _count_days_in_period(trading_idx: pd.DatetimeIndex, freq_code: str, ref_date: pd.Timestamp) -> int:
        """Conteggio giorni nel periodo corrente in modo robusto (no confronto Period ambiguo)."""
        if len(trading_idx) == 0:
            return 0

        f = str(freq_code).upper().strip()
        ref_date = pd.Timestamp(ref_date).normalize()

        if f in {"YE", "Y", "A", "ANNUAL", "YEARLY"}:
            return int((trading_idx.year == ref_date.year).sum())

        if f in {"QE", "Q", "QUARTER", "QUARTERLY"}:
            ref_q = ref_date.to_period("Q")
            return int((trading_idx.to_period("Q") == ref_q).sum())

        if f in {"ME", "M", "MONTH", "MONTHLY"}:
            ref_m = ref_date.to_period("M")
            return int((trading_idx.to_period("M") == ref_m).sum())

        if f == "W":
            ref_w = ref_date.to_period("W-FRI")
            return int((trading_idx.to_period("W-FRI") == ref_w).sum())

        if f.startswith("W-"):
            ref_w = ref_date.to_period(f)
            return int((trading_idx.to_period(f) == ref_w).sum())

        # daily / unknown
        return 999999

    def _is_incomplete_last_period(trading_idx: pd.DatetimeIndex, freq_code: str) -> bool:
        """
        True se l'ULTIMO periodo è incompleto.
        FIX: conteggio giorni robusto (evita bug che eliminava la rebal_date finale in YE).
        """
        if len(trading_idx) == 0:
            return True

        f = str(freq_code).upper().strip()
        last_date = pd.Timestamp(trading_idx.max()).normalize()

        MIN_DAYS_MONTH = 10
        MIN_DAYS_QUARTER = 30
        MIN_DAYS_WEEK = 3
        MIN_DAYS_YEAR = 180

        if f in {"ME", "M", "MONTH", "MONTHLY"}:
            return _count_days_in_period(trading_idx, f, last_date) < MIN_DAYS_MONTH

        if f in {"QE", "Q", "QUARTER", "QUARTERLY"}:
            # richiede anche che siamo nell'ultimo mese del trimestre
            last_q = last_date.to_period("Q")
            end_month = last_q.quarter * 3
            if last_date.month != end_month:
                return True
            return _count_days_in_period(trading_idx, f, last_date) < MIN_DAYS_QUARTER

        if f in {"YE", "Y", "A", "ANNUAL", "YEARLY"}:
            return _count_days_in_period(trading_idx, f, last_date) < MIN_DAYS_YEAR

        if f == "W" or f.startswith("W-"):
            return _count_days_in_period(trading_idx, f, last_date) < MIN_DAYS_WEEK

        if f in {"D", "DAY", "DAILY"}:
            return False

        return False

    def compute_rebal_dates_trading(trading_idx: pd.DatetimeIndex, freq_code: str) -> pd.DatetimeIndex:
        """
        Calcola rebal_dates come ultimo giorno di trading di ciascun periodo.
        Poi applica filtro per evitare l'ultimo periodo "incompleto" (data-driven, robusto).
        """
        if len(trading_idx) == 0:
            return pd.DatetimeIndex([])

        f = str(freq_code).upper().strip()
        last_date = pd.Timestamp(trading_idx.max()).normalize()

        if f in {"D", "DAY", "DAILY"}:
            dates = pd.DatetimeIndex(trading_idx)
            return dates.sort_values().unique()

        if f == "W":
            anchor = "FRI"
            periods = trading_idx.to_period(f"W-{anchor}")
            dates = pd.DatetimeIndex(pd.Series(trading_idx).groupby(periods).max().values)

        elif f.startswith("W-"):
            periods = trading_idx.to_period(f)
            dates = pd.DatetimeIndex(pd.Series(trading_idx).groupby(periods).max().values)

        elif f in {"ME", "M", "MONTH", "MONTHLY"}:
            periods = trading_idx.to_period("M")
            dates = pd.DatetimeIndex(pd.Series(trading_idx).groupby(periods).max().values)

        elif f in {"QE", "Q", "QUARTER", "QUARTERLY"}:
            periods = trading_idx.to_period("Q")
            dates = pd.DatetimeIndex(pd.Series(trading_idx).groupby(periods).max().values)

        elif f in {"YE", "Y", "A", "ANNUAL", "YEARLY"}:
            periods = trading_idx.to_period("Y")
            dates = pd.DatetimeIndex(pd.Series(trading_idx).groupby(periods).max().values)

        else:
            # fallback: prova resample, altrimenti mensile
            try:
                _ = pd.Series(1, index=trading_idx).resample(freq_code).last().dropna()
                periods = trading_idx.to_period("M")
                dates = pd.DatetimeIndex(pd.Series(trading_idx).groupby(periods).max().values)
            except Exception:
                dates = pd.DatetimeIndex(trading_idx)

        dates = pd.DatetimeIndex(dates).sort_values().unique()

        # filtro ultimo periodo incompleto SOLO se l'ultima date coincide con last_date
        if len(dates) and pd.Timestamp(dates.max()).normalize() == last_date:
            if _is_incomplete_last_period(trading_idx, freq_code):
                dates = dates[:-1]

        return dates

    rebal_dates = compute_rebal_dates_trading(idx_full, freq)
    rebal_dates = pd.DatetimeIndex([d for d in rebal_dates if d in idx_full]).sort_values().unique()

    _dbg(f"[DEBUG] rebalance_frequency={rebalance_frequency} (norm={freq})")
    _dbg(f"[DEBUG] prices idx: {idx_full.min().date()} -> {idx_full.max().date()}  (n={len(idx_full)})")
    _dbg(f"[DEBUG] rebal_dates: n={len(rebal_dates)} tail={list(rebal_dates[-6:]) if len(rebal_dates) else []}")

    # ==========================================================
    # 3) PRECOMPUTE: MOMENTUM / VOL / EMA / ACCEL
    # ==========================================================
    mom_df = prices / prices.shift(int(momentum_lookback_days))

    # volatility con cache support
    if vol_cache is not None and int(riskparity_lookback_days) in vol_cache:
        _dbg(f"[DEBUG] Using cached volatility for window={riskparity_lookback_days}")
        vol_df = vol_cache[int(riskparity_lookback_days)]
        if not vol_df.index.equals(rets.index):
            vol_df = vol_df.reindex(rets.index)
    else:
        if vol_cache is not None:
            _dbg(f"[DEBUG] Window {riskparity_lookback_days} not in vol_cache, computing on-demand")
        vol_df = rets.rolling(int(riskparity_lookback_days), min_periods=1).std(ddof=0)

    inv_vol_df = 1.0 / vol_df.replace(0.0, np.nan)

    rm_df = mom_df.rank(pct=True, axis=1, na_option="bottom")
    riv_df = inv_vol_df.rank(pct=True, axis=1, na_option="bottom")

    ema_df = None
    if filter_ema:
        ema_df = prices.ewm(span=int(ema_span), adjust=False, min_periods=1).mean()

    accel_df = None
    accel_rank_df = None
    if use_acceleration:
        _ACCEL_SMOOTH = 5
        _ACCEL_SHIFT = 10
        _ACCEL_WEIGHT = 0.20
        mom_sm_df = mom_df.ewm(span=_ACCEL_SMOOTH, adjust=False, min_periods=1).mean()
        accel_df = mom_sm_df - mom_sm_df.shift(_ACCEL_SHIFT)
        accel_rank_df = accel_df.rank(pct=True, axis=1, na_option="bottom")

    # ==========================================================
    # 4) LOOP rebalance: pesi + selezioni (DENSE + CARRY-FWD)
    # ==========================================================
    w_rot: dict[pd.Timestamp, pd.Series] = {}
    sel_dates: list[pd.Timestamp] = []
    sel_values: list[list[str]] = []
    rank_dict: dict[pd.Timestamp, pd.Series] = {}

    w_mom = {} if build_other_portfolios else None
    w_rp = {} if build_other_portfolios else None

    bottom_values: list[list[str]] = [] if bottom_tickers else []

    prev_top: list[str] | None = None
    prev_bottom: list[str] | None = None

    # --- Guardrail CRITICO ---
    if len(rebal_dates) == 0:
        _dbg("[DEBUG] rebal_dates vuoto -> fallback pesi a zero (cash) e selezioni vuote")

        w_rot_sh = pd.DataFrame(0.0, index=prices.index, columns=cols)

        if start_date is not None:
            start_date_ts = pd.to_datetime(start_date).normalize()
            prices = prices.loc[start_date_ts:]
            w_rot_sh = w_rot_sh.loc[start_date_ts:]
            if bench_px is not None:
                bench_px = bench_px.loc[start_date_ts:]

        pf_rot_w = vbt.Portfolio.from_orders(
            close=prices,
            size=w_rot_sh,
            size_type="targetpercent",
            init_cash=init_cash,
            cash_sharing=True,
            freq="D",
        )

        pf_bh = None
        if bench_px is not None:
            bench_px2 = bench_px.dropna().rename("Benchmark")
            pf_bh = vbt.Portfolio.from_holding(
                close=bench_px2.to_frame(),
                init_cash=init_cash,
                cash_sharing=True,
                freq="D",
            )

        sel_tickers_rot_w = pd.DataFrame(columns=["Top_Tickers"])
        sel_tickers_rot_w.index.name = "RebalanceDate"

        rankings_df = pd.DataFrame()
        rankings_df.index.name = "RebalanceDate"

        if bottom_tickers:
            sel_tickers_rot_bottom = pd.DataFrame(columns=["Bottom_Tickers"])
            sel_tickers_rot_bottom.index.name = "RebalanceDate"
            if build_other_portfolios:
                return pf_rot_w, None, None, pf_bh, sel_tickers_rot_w, rankings_df, sel_tickers_rot_bottom
            return pf_rot_w, pf_bh, sel_tickers_rot_w, rankings_df, sel_tickers_rot_bottom

        if build_other_portfolios:
            return pf_rot_w, None, None, pf_bh, sel_tickers_rot_w, rankings_df

        return pf_rot_w, pf_bh, sel_tickers_rot_w, rankings_df

    # --- Caso normale: abbiamo rebal_dates validi ---
    for d in rebal_dates:
        d = pd.Timestamp(d).normalize()

        mom = mom_df.loc[d]
        vol = vol_df.loc[d]
        inv_vol = inv_vol_df.loc[d]
        rm = rm_df.loc[d]
        riv = riv_df.loc[d]

        base_mask = mom.notna() & inv_vol.notna()
        mask = base_mask.copy()
        n_base = int(base_mask.sum())

        if filter_ema:
            ema = ema_df.loc[d]
            m_ema = prices.loc[d] > ema
            mask &= m_ema
            n_ema = int(mask.sum())
        else:
            n_ema = n_base

        if filter_volatility:
            q = vol.quantile(float(volatility_quantile))
            m_vol = vol < q
            mask &= m_vol
            n_vol = int(mask.sum())
        else:
            n_vol = n_ema

        if filter_min_momentum:
            m_minmom = mom > float(min_momentum_threshold)
            mask &= m_minmom
            n_minmom = int(mask.sum())
        else:
            n_minmom = n_vol

        combo = float(momentum_weight) * rm + (1.0 - float(momentum_weight)) * riv

        if use_acceleration:
            accel_today = accel_df.loc[d]
            m_accel = accel_today > 0
            mask &= m_accel
            n_accel = int(mask.sum())

            accel_rank = accel_rank_df.loc[d]
            combo = combo + 0.20 * accel_rank
        else:
            n_accel = n_minmom

        combo = combo.where(mask)
        rank_dict[d] = combo.sort_values(ascending=False)

        # --- selezione TOP
        top_list: list[str] = []
        if not combo.dropna().empty:
            top_list = list(combo.nlargest(int(n_top)).index)

        carried = False
        if len(top_list) == 0 and prev_top is not None:
            top_list = prev_top.copy()
            carried = True

        sel_dates.append(d)
        sel_values.append(top_list)

        if len(top_list) > 0:
            prev_top = top_list

        if debug:
            if carried:
                print(
                    f"[DEBUG] CARRY-FORWARD | {d.date()} | "
                    f"reason=empty_selection_after_filters | "
                    f"mask_counts: base={n_base}, ema={n_ema}, vol={n_vol}, min_mom={n_minmom}, accel={n_accel} | "
                    f"carried_n={len(top_list)} carried_list={top_list}"
                )
            else:
                print(
                    f"[DEBUG] SELECTION OK | {d.date()} | "
                    f"mask_counts: base={n_base}, ema={n_ema}, vol={n_vol}, min_mom={n_minmom}, accel={n_accel} | "
                    f"selected_n={len(top_list)}"
                )

        # --- bottom (opzionale)
        if bottom_tickers:
            bottom_list: list[str] = []
            if not combo.dropna().empty:
                bottom_list = list(combo.nsmallest(int(n_top)).index)

            bottom_carried = False
            if len(bottom_list) == 0 and prev_bottom is not None:
                bottom_list = prev_bottom.copy()
                bottom_carried = True

            bottom_values.append(bottom_list)
            if len(bottom_list) > 0:
                prev_bottom = bottom_list

            if debug and bottom_carried:
                print(f"[DEBUG] CARRY-FORWARD BOTTOM | {d.date()} | carried_n={len(bottom_list)}")

        # --- pesi equal-weight
        w3 = pd.Series(0.0, index=cols)
        if len(top_list) > 0:
            top_in_universe = [t for t in top_list if t in cols]
            if len(top_in_universe) > 0:
                w3[top_in_universe] = 1.0 / len(top_in_universe)
        w_rot[d] = w3

        # --- altri portafogli
        if build_other_portfolios:
            w1 = mom.where(mask, 0.0)
            s1 = float(w1.sum())
            w1 = w1.div(s1) if np.isfinite(s1) and s1 else w1 * 0.0
            w_mom[d] = w1

            w2 = inv_vol.where(mask, 0.0)
            s2 = float(w2.sum())
            w2 = w2.div(s2) if np.isfinite(s2) and s2 else w2 * 0.0
            w_rp[d] = w2

    # ==========================================================
    # 5) WEIGHTS FULL INDEX + SHIFT (orders il giorno dopo)
    # ==========================================================
    w_rot_df = pd.DataFrame(w_rot).T.reindex(idx_full).ffill().fillna(0.0)
    w_rot_sh = w_rot_df.shift(1).ffill().fillna(0.0)

    if build_other_portfolios:
        w_mom_df = pd.DataFrame(w_mom).T.reindex(idx_full).ffill().fillna(0.0)
        w_rp_df = pd.DataFrame(w_rp).T.reindex(idx_full).ffill().fillna(0.0)
        w_mom_sh = w_mom_df.shift(1).ffill().fillna(0.0)
        w_rp_sh = w_rp_df.shift(1).ffill().fillna(0.0)

    # ==========================================================
    # 6) START DATE CUT
    # ==========================================================
    if start_date is not None:
        start_date_ts = pd.to_datetime(start_date).normalize()
        w_rot_sh = w_rot_sh.loc[start_date_ts:]
        prices = prices.loc[start_date_ts:]
        if bench_px is not None:
            bench_px = bench_px.loc[start_date_ts:]
        if build_other_portfolios:
            w_mom_sh = w_mom_sh.loc[start_date_ts:]
            w_rp_sh = w_rp_sh.loc[start_date_ts:]

    # ==========================================================
    # 7) PORTFOLIOS
    # ==========================================================
    pf_rot_w = vbt.Portfolio.from_orders(
        close=prices,
        size=w_rot_sh,
        size_type="targetpercent",
        init_cash=init_cash,
        cash_sharing=True,
        freq="D",
    )

    pf_bh = None
    if bench_px is not None:
        bench_px2 = bench_px.dropna().rename("Benchmark")
        pf_bh = vbt.Portfolio.from_holding(
            close=bench_px2.to_frame(),
            init_cash=init_cash,
            cash_sharing=True,
            freq="D",
        )

    pf_mom = None
    pf_rp = None
    if build_other_portfolios:
        pf_mom = vbt.Portfolio.from_orders(
            close=prices,
            size=w_mom_sh,
            size_type="targetpercent",
            init_cash=init_cash,
            cash_sharing=True,
            freq="D",
        )
        pf_rp = vbt.Portfolio.from_orders(
            close=prices,
            size=w_rp_sh,
            size_type="targetpercent",
            init_cash=init_cash,
            cash_sharing=True,
            freq="D",
        )

    # ==========================================================
    # 8) PLOT
    # ==========================================================
    if plot:
        import plotly.graph_objects as go

        fig = go.Figure()
        cum_rot_w = pf_rot_w.value() / init_cash
        fig.add_trace(go.Scatter(x=cum_rot_w.index, y=cum_rot_w, mode="lines", name="Rotational Weighted"))

        if build_other_portfolios:
            cum_mom = pf_mom.value() / init_cash
            cum_rp = pf_rp.value() / init_cash
            fig.add_trace(go.Scatter(x=cum_mom.index, y=cum_mom, mode="lines", name="Momentum"))
            fig.add_trace(go.Scatter(x=cum_rp.index, y=cum_rp, mode="lines", name="RiskParity"))

        if pf_bh is not None:
            cum_bh = pf_bh.value() / init_cash
            fig.add_trace(go.Scatter(
                x=cum_bh.index, y=cum_bh, mode="lines", name="Benchmark",
                line=dict(color="gray", width=2), opacity=0.8
            ))

        fig.update_layout(
            title=f"{portfolio_name} – Rendimenti cumulati",
            yaxis_tickformat=".0%",
            width=vbt_plot_width,
            height=600,
            template="plotly_white",
            xaxis=dict(
                rangeselector=dict(buttons=[
                    dict(count=1, label="1M", step="month", stepmode="backward"),
                    dict(count=3, label="3M", step="month", stepmode="backward"),
                    dict(count=6, label="6M", step="month", stepmode="backward"),
                    dict(count=1, label="YTD", step="year", stepmode="todate"),
                    dict(step="all"),
                ]),
                rangeslider=dict(visible=True),
                type="date",
            ),
        )
        fig.show()

    # ==========================================================
    # 9) SELECTION DF + RANKINGS DF
    # ==========================================================
    sel_tickers_rot_w = pd.DataFrame(
        {"Top_Tickers": sel_values},
        index=pd.DatetimeIndex(sel_dates, name="RebalanceDate"),
    )

    rankings_df = pd.DataFrame(rank_dict).T
    rankings_df.index.name = "RebalanceDate"

    # ==========================================================
    # 10) RETURN ARITY (stabile)
    # ==========================================================
    if bottom_tickers:
        sel_tickers_rot_bottom = pd.DataFrame(
            {"Bottom_Tickers": bottom_values},
            index=pd.DatetimeIndex(sel_dates, name="RebalanceDate"),
        )
        sel_tickers_rot_bottom.index.name = "RebalanceDate"

        if build_other_portfolios:
            return pf_rot_w, pf_mom, pf_rp, pf_bh, sel_tickers_rot_w, rankings_df, sel_tickers_rot_bottom
        return pf_rot_w, pf_bh, sel_tickers_rot_w, rankings_df, sel_tickers_rot_bottom

    if build_other_portfolios:
        return pf_rot_w, pf_mom, pf_rp, pf_bh, sel_tickers_rot_w, rankings_df

    return pf_rot_w, pf_bh, sel_tickers_rot_w, rankings_df


In [ ]:
"""
build_rotational_portfolios_vbt.py
===================================
Versione migliorata di build_rotational_portfolios_vbt.

MIGLIORAMENTI RISPETTO ALL'ORIGINALE
-------------------------------------
1. ARITÀ RETURN STABILE
   - Ritorna sempre un RotationalVbtResult (dataclass).
   - Nessuna più tuple 4/5/6/7 dipendente da flag booleani.
   - pf_mom, pf_rp, sel_bottom sono None se non richiesti.

2. LOGICA REBAL-DATES DEDUPLICATA
   - Usa compute_rebal_dates() da rotational_engine.py (unica fonte di verità).
   - Rimosse: compute_rebal_dates_trading(), _count_days_in_period(),
     _is_incomplete_last_period() (erano duplicati interni alla funzione).

3. VALIDAZIONE PARAMETRI COMPLETA
   - n_top, ema_span, volatility_quantile, min_momentum_threshold,
     riskparity_lookback_days, momentum_lookback_days tutti validati.

4. DEBUG STRUTTURATO E COMPLETO
   - DebugLogger: livelli INFO / DETAIL / TRACE.
   - Log su: rebal_dates, ogni selezione, pesi calcolati, shift, start_date cut.
   - n_accel loggato correttamente anche quando use_acceleration=False.
   - carried loggato sempre (non solo quando bottom_carried).

5. CARRIED FLAG IN OUTPUT
   - sel_tickers_rot_w ha colonna 'carried' (bool) oltre a 'Top_Tickers'.
   - Informazione prima persa, ora disponibile per audit e reporting.

6. FALLBACK REBAL_DATES VUOTO NON DUPLICA CODICE
   - Un solo punto di costruzione VBT (helper _build_vbt_portfolio).

7. DOCSTRING OPERATIVA
   - Parametri tutti documentati con tipo, default e effetto.
   - Arità return esplicitata per ogni combinazione di flag.
   - vol_cache documentato.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Optional

# import numpy as np
# import pandas as pd


# # ─── import dalla single source of truth ────────────────────────────────────
# # compute_rebal_dates è definita in rotational_engine.py.
# # Se questo file è nello stesso package, l'import è diretto.
# # Se usato standalone nel JN, assicurarsi che rotational_engine sia in scope.
# try:
#     from rotational_engine import compute_rebal_dates
# except ImportError:
#     # fallback: la funzione deve essere disponibile nel namespace del JN
#     compute_rebal_dates = None  # type: ignore


# ─────────────────────────────────────────────────────────────────────────────
# OUTPUT DATACLASS  (arità stabile)
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class RotationalVbtResult:
    """
    Output strutturato di build_rotational_portfolios_vbt.

    Attributi sempre presenti
    -------------------------
    pf_rot      : vbt.Portfolio   Portafoglio rotazionale equal-weight.
    pf_bh       : vbt.Portfolio | None   Buy-and-hold benchmark (None se benchmark_data=None).
    selections  : pd.DataFrame    Index=RebalanceDate. Colonne: Top_Tickers (list), carried (bool).
    rankings    : pd.DataFrame    Combo-score per tutti i ticker a ogni rebal_date.
    rebal_dates : pd.DatetimeIndex  Date di ribilanciamento effettivamente usate.

    Attributi opzionali (None se non richiesti)
    --------------------------------------------
    pf_mom      : vbt.Portfolio | None   Solo se build_other_portfolios=True.
    pf_rp       : vbt.Portfolio | None   Solo se build_other_portfolios=True.
    sel_bottom  : pd.DataFrame | None    Solo se bottom_tickers=True.

    RETROCOMPATIBILITA' TUPLE
    -------------------------
    L'oggetto e' subscriptable e iterabile, replicando l'arità dell'API legacy:

      default (4)      : pf_rot, pf_bh, selections, rankings
      with_others (6)  : pf_rot, pf_mom, pf_rp, pf_bh, selections, rankings
      with_bottom (5)  : pf_rot, pf_bh, selections, rankings, sel_bottom
      full (7)         : pf_rot, pf_mom, pf_rp, pf_bh, selections, rankings, sel_bottom

    Esempi:
        pf_rot_w, pf_bh, sel, rankings = result   # unpacking
        pf_rot_w = result[0]                       # subscript
        pf_rot_w = result.pf_rot                   # attributo (preferito)
    """
    pf_rot:      object
    pf_bh:       Optional[object]
    selections:  pd.DataFrame
    rankings:    pd.DataFrame
    rebal_dates: pd.DatetimeIndex
    pf_mom:      Optional[object] = None
    pf_rp:       Optional[object] = None
    sel_bottom:  Optional[pd.DataFrame] = None

    def _as_tuple(self) -> tuple:
        """Tuple nell'arità corretta in base ai campi opzionali attivi."""
        has_others = self.pf_mom is not None or self.pf_rp is not None
        has_bottom = self.sel_bottom is not None
        if has_others and has_bottom:
            return (self.pf_rot, self.pf_mom, self.pf_rp, self.pf_bh,
                    self.selections, self.rankings, self.sel_bottom)
        if has_others:
            return (self.pf_rot, self.pf_mom, self.pf_rp, self.pf_bh,
                    self.selections, self.rankings)
        if has_bottom:
            return (self.pf_rot, self.pf_bh,
                    self.selections, self.rankings, self.sel_bottom)
        return (self.pf_rot, self.pf_bh, self.selections, self.rankings)

    def __iter__(self):
        return iter(self._as_tuple())

    def __getitem__(self, idx):
        return self._as_tuple()[idx]

    def __len__(self):
        return len(self._as_tuple())


# ─────────────────────────────────────────────────────────────────────────────
# DEBUG LOGGER INTERNO
# ─────────────────────────────────────────────────────────────────────────────

class _DebugLogger:
    """
    Logger strutturato a 3 livelli per debug interno.

    Livelli
    -------
    0 = off
    1 = INFO   : eventi principali (rebal_dates, selezioni, start_date cut)
    2 = DETAIL : dettagli per ogni rebal_date (mask counts, pesi)
    3 = TRACE  : tutto (shift matrix, fallback steps)
    """
    def __init__(self, level: int):
        self.level = int(level)

    def info(self, msg: str):
        if self.level >= 1:
            print(f"[INFO]   {msg}")

    def detail(self, msg: str):
        if self.level >= 2:
            print(f"[DETAIL] {msg}")

    def trace(self, msg: str):
        if self.level >= 3:
            print(f"[TRACE]  {msg}")


# ─────────────────────────────────────────────────────────────────────────────
# HELPER VBT
# ─────────────────────────────────────────────────────────────────────────────

def _build_vbt_portfolio(prices: pd.DataFrame, weights: pd.DataFrame, init_cash: float):
    """Costruisce un vbt.Portfolio.from_orders con i parametri standard."""
    import vectorbt as vbt
    return vbt.Portfolio.from_orders(
        close=prices,
        size=weights,
        size_type="targetpercent",
        init_cash=init_cash,
        cash_sharing=True,
        freq="D",
    )


def _build_vbt_bh(bench_px: pd.Series, prices_index: pd.DatetimeIndex, init_cash: float):
    """Costruisce il benchmark buy-and-hold allineato all'indice dei prezzi."""
    import vectorbt as vbt
    bench = bench_px.reindex(prices_index).ffill().dropna().rename("Benchmark")
    return vbt.Portfolio.from_holding(
        close=bench.to_frame(),
        init_cash=init_cash,
        cash_sharing=True,
        freq="D",
    )


# ─────────────────────────────────────────────────────────────────────────────
# FUNZIONE PRINCIPALE
# ─────────────────────────────────────────────────────────────────────────────

def build_rotational_portfolios_vbt(
    stocks_data: pd.DataFrame,
    benchmark_data: pd.Series | None = None,
    # ── frequenza e lookback ─────────────────────────────────────────────────
    rebalance_frequency: str = "ME",
    momentum_lookback_days: int = 126,
    riskparity_lookback_days: int = 20,
    # ── selezione ────────────────────────────────────────────────────────────
    n_top: int = 5,
    momentum_weight: float = 0.7,
    # ── filtri opzionali ─────────────────────────────────────────────────────
    filter_ema: bool = False,
    filter_volatility: bool = False,
    filter_min_momentum: bool = False,
    ema_span: int = 200,
    volatility_quantile: float = 0.75,
    min_momentum_threshold: float = 1.0,
    # ── accelerazione ────────────────────────────────────────────────────────
    use_acceleration: bool = False,
    # ── portafoglio ──────────────────────────────────────────────────────────
    init_cash: float = 100_000,
    start_date: str | pd.Timestamp | None = None,
    # ── output aggiuntivi ────────────────────────────────────────────────────
    build_other_portfolios: bool = False,
    bottom_tickers: bool = False,
    # ── performance ──────────────────────────────────────────────────────────
    vol_cache: dict | None = None,
    # ── plot ─────────────────────────────────────────────────────────────────
    plot: bool = True,
    portfolio_name: str = "Portafoglio rotazionale",
    vbt_plot_width: int = 1000,
    # ── debug ────────────────────────────────────────────────────────────────
    debug: bool = False,
    debug_level: int = 1,
) -> RotationalVbtResult:
    """
    Costruisce portafogli rotazionali equal-weight con ribilanciamento trading-aligned.

    MECCANISMO CORE
    ---------------
    1. Calcola le rebal_dates come ultimo trading-day di ciascun periodo completo
       (delegato a compute_rebal_dates(), unica fonte di verità nel progetto).
    2. A ogni rebal_date applica filtri opzionali (EMA, volatilità, momentum minimo,
       accelerazione) e seleziona i top-N ticker per combo-score
       (momentum_weight * rank_mom + (1-momentum_weight) * rank_inv_vol).
    3. Se la selezione è vuota dopo i filtri, esegue carry-forward dell'ultima
       selezione valida (segnalato nella colonna `carried` dell'output).
    4. Costruisce la matrice dei pesi equal-weight e la shifta di 1 giorno
       (gli ordini vengono eseguiti il giorno di trading successivo alla rebal_date).

    PARAMETRI
    ---------
    stocks_data : pd.DataFrame
        Prezzi giornalieri. Index = DatetimeIndex (trading days only).
        Colonne = ticker. NaN interni vengono ffill/bfill.

    benchmark_data : pd.Series | None
        Prezzi del benchmark (es. ETF su indice). Se None, pf_bh=None in output.

    rebalance_frequency : str  default='ME'
        Frequenza di ribilanciamento. Valori supportati:
        'ME'/'M' (mensile), 'QE'/'Q' (trimestrale), 'YE'/'Y' (annuale),
        'W'/'W-FRI' (settimanale), 'D' (giornaliero).

    momentum_lookback_days : int  default=126
        Finestra in giorni per il calcolo del momentum (price ratio).
        Deve essere > 0 e < len(stocks_data).

    riskparity_lookback_days : int  default=20
        Finestra in giorni per il calcolo della volatilità (rolling std dei ritorni).
        Deve essere > 0.

    n_top : int  default=5
        Numero di ticker da selezionare a ogni ribilanciamento. Deve essere >= 1.

    momentum_weight : float  default=0.7
        Peso del rank di momentum nel combo-score [0.0, 1.0].
        Il peso del rank inv-volatilità è (1 - momentum_weight).

    filter_ema : bool  default=False
        Se True, esclude i ticker il cui prezzo è < EMA(ema_span).

    filter_volatility : bool  default=False
        Se True, esclude i ticker nel quantile superiore di volatilità
        (soglia = volatility_quantile).

    filter_min_momentum : bool  default=False
        Se True, esclude i ticker con momentum < min_momentum_threshold.
        Un threshold di 1.0 significa: escludi chi ha prezzo attuale < prezzo N giorni fa.

    ema_span : int  default=200
        Span per il calcolo dell'EMA (usato solo se filter_ema=True). Deve essere > 0.

    volatility_quantile : float  default=0.75
        Quantile di esclusione per volatilità (usato solo se filter_volatility=True).
        Deve essere in (0, 1).

    min_momentum_threshold : float  default=1.0
        Soglia minima di momentum (usato solo se filter_min_momentum=True).

    use_acceleration : bool  default=False
        Se True, aggiunge al combo-score un termine di accelerazione del momentum
        (EWM a breve su EWM a lungo). I ticker con accelerazione negativa vengono esclusi.

    init_cash : float  default=100_000
        Capitale iniziale per i portafogli VBT.

    start_date : str | pd.Timestamp | None  default=None
        Se fornito, i portafogli vengono tagliati a partire da questa data.
        I pesi vengono costruiti sull'intero range (per lookback corretto)
        e poi tagliati. Deve essere una data presente o successiva al primo
        trading day disponibile.

    build_other_portfolios : bool  default=False
        Se True, costruisce anche:
        - pf_mom : portafoglio momentum puro (pesi proporzionali al momentum)
        - pf_rp  : portafoglio risk-parity puro (pesi proporzionali a 1/vol)
        Disponibili in result.pf_mom e result.pf_rp.

    bottom_tickers : bool  default=False  [SPERIMENTALE]
        Se True, calcola anche la selezione dei bottom-N ticker (peggiori per score).
        Disponibile in result.sel_bottom.

    vol_cache : dict | None  default=None
        Cache delle volatilità precalcolate. Formato: {lookback_days: vol_df}.
        vol_df deve avere stesso indice di stocks_data.
        Utile in WFO dove la stessa finestra viene ricalcolata più volte.

    plot : bool  default=True
        Se True, mostra il grafico Plotly dei rendimenti cumulati.

    portfolio_name : str  default='Portafoglio rotazionale'
        Titolo del grafico.

    vbt_plot_width : int  default=1000
        Larghezza in pixel del grafico Plotly.

    debug : bool  default=False
        Abilita il logging di debug (equivalente a debug_level=1 se True, 0 se False).
        Se debug_level è specificato esplicitamente, ha precedenza.

    debug_level : int  default=1
        Livello di dettaglio del debug (attivo solo se debug=True):
        1 = INFO   : eventi principali (rebal_dates, selezioni, start_date cut)
        2 = DETAIL : dettaglio per ogni rebal_date (mask counts, pesi effettivi)
        3 = TRACE  : tutto (shift matrix, valori intermedi)

    RETURN
    ------
    RotationalVbtResult con i seguenti campi:

    Sempre presenti:
      .pf_rot      vbt.Portfolio   Portafoglio rotazionale.
      .pf_bh       vbt.Portfolio | None   Benchmark B&H. None se benchmark_data=None.
      .selections  pd.DataFrame   Index=RebalanceDate. Colonne:
                     Top_Tickers (list[str]) : ticker selezionati
                     carried (bool)          : True se selezione da carry-forward
      .rankings    pd.DataFrame   Combo-score completo per ogni rebal_date.
      .rebal_dates pd.DatetimeIndex  Date di ribilanciamento usate.

    Opzionali (None se flag non attivo):
      .pf_mom      vbt.Portfolio | None   Solo se build_other_portfolios=True.
      .pf_rp       vbt.Portfolio | None   Solo se build_other_portfolios=True.
      .sel_bottom  pd.DataFrame | None    Solo se bottom_tickers=True.

    RAISES
    ------
    ValueError
        Se i parametri numerici sono fuori range.
    TypeError
        Se stocks_data non è un pd.DataFrame.
    RuntimeError
        Se compute_rebal_dates non è disponibile nel namespace.
    """

    # ── 0) Risolvi debug_level ────────────────────────────────────────────────
    effective_level = debug_level if debug else 0
    log = _DebugLogger(effective_level)

    # ── 1) Validazione parametri ──────────────────────────────────────────────
    if not isinstance(stocks_data, pd.DataFrame):
        raise TypeError("stocks_data deve essere un pd.DataFrame")
    if not 0.0 <= momentum_weight <= 1.0:
        raise ValueError(f"momentum_weight={momentum_weight} non in [0, 1]")
    if n_top < 1:
        raise ValueError(f"n_top={n_top} deve essere >= 1")
    if momentum_lookback_days < 1:
        raise ValueError(f"momentum_lookback_days={momentum_lookback_days} deve essere >= 1")
    if riskparity_lookback_days < 1:
        raise ValueError(f"riskparity_lookback_days={riskparity_lookback_days} deve essere >= 1")
    if filter_ema and ema_span < 1:
        raise ValueError(f"ema_span={ema_span} deve essere >= 1")
    if filter_volatility and not 0.0 < volatility_quantile < 1.0:
        raise ValueError(f"volatility_quantile={volatility_quantile} deve essere in (0, 1)")

    _compute_rebal = compute_rebal_dates
    if _compute_rebal is None:
        raise RuntimeError(
            "compute_rebal_dates non disponibile. "
            "Assicurarsi che rotational_engine.py sia importato nel namespace."
        )

    # ── 2) Normalizzazione dati ───────────────────────────────────────────────
    prices = stocks_data.dropna(axis=1, how="all").ffill().bfill().copy()
    prices.index = pd.to_datetime(prices.index).normalize()
    prices = prices.sort_index()
    cols = prices.columns
    idx_full = prices.index.unique()

    bench_px = None
    if benchmark_data is not None:
        bench_px = benchmark_data.copy()
        bench_px.index = pd.to_datetime(bench_px.index).normalize()
        bench_px = bench_px.sort_index()

    rets = prices.pct_change().fillna(0.0)

    log.info(
        f"freq={rebalance_frequency}  "
        f"prices={idx_full.min().date()}→{idx_full.max().date()}  "
        f"n_assets={prices.shape[1]}  "
        f"n_top={n_top}  mom_lb={momentum_lookback_days}  rp_lb={riskparity_lookback_days}"
    )

    # ── 3) Rebal dates (unica fonte di verità) ────────────────────────────────
    freq = str(rebalance_frequency).upper().strip()
    rebal_dates = _compute_rebal(idx_full, freq)
    rebal_dates = pd.DatetimeIndex(
        [d for d in rebal_dates if d in idx_full]
    ).sort_values().unique()

    log.info(
        f"rebal_dates: n={len(rebal_dates)}  "
        f"first={rebal_dates[0].date() if len(rebal_dates) else 'N/A'}  "
        f"last={rebal_dates[-1].date() if len(rebal_dates) else 'N/A'}"
    )
    log.detail(f"rebal_dates tail: {[str(d.date()) for d in rebal_dates[-6:]]}")

    # ── 4) Precomputa indicatori ──────────────────────────────────────────────
    mom_df = prices / prices.shift(momentum_lookback_days)

    if vol_cache is not None and riskparity_lookback_days in vol_cache:
        log.detail(f"vol_cache HIT: window={riskparity_lookback_days}")
        vol_df = vol_cache[riskparity_lookback_days]
        if not vol_df.index.equals(rets.index):
            vol_df = vol_df.reindex(rets.index)
    else:
        if vol_cache is not None:
            log.detail(f"vol_cache MISS: window={riskparity_lookback_days}, calcolo on-demand")
        vol_df = rets.rolling(riskparity_lookback_days, min_periods=1).std(ddof=0)

    inv_vol_df = 1.0 / vol_df.replace(0.0, np.nan)
    rank_mom_df  = mom_df.rank(pct=True, axis=1, na_option="bottom")
    rank_ivol_df = inv_vol_df.rank(pct=True, axis=1, na_option="bottom")

    ema_df = None
    if filter_ema:
        ema_df = prices.ewm(span=ema_span, adjust=False, min_periods=1).mean()

    accel_df = accel_rank_df = None
    _ACCEL_WEIGHT = 0.20
    if use_acceleration:
        mom_sm  = mom_df.ewm(span=5, adjust=False, min_periods=1).mean()
        accel_df = mom_sm - mom_sm.shift(10)
        accel_rank_df = accel_df.rank(pct=True, axis=1, na_option="bottom")

    # ── 5) Loop di selezione ──────────────────────────────────────────────────
    w_rot: dict[pd.Timestamp, pd.Series] = {}
    w_mom: dict[pd.Timestamp, pd.Series] = {} if build_other_portfolios else None
    w_rp:  dict[pd.Timestamp, pd.Series] = {} if build_other_portfolios else None

    sel_dates:    list[pd.Timestamp] = []
    sel_tickers:  list[list[str]]    = []
    sel_carried:  list[bool]         = []
    rank_records: dict[pd.Timestamp, pd.Series] = {}

    bottom_sel:     list[list[str]] = []
    prev_top:       list[str] | None = None
    prev_bottom:    list[str] | None = None

    # ── fallback: nessuna rebal_date ──────────────────────────────────────────
    if len(rebal_dates) == 0:
        log.info("WARN: nessuna rebal_date valida → portafoglio in cash (pesi zero)")
        return _build_empty_result(
            prices=prices,
            bench_px=bench_px,
            cols=cols,
            init_cash=init_cash,
            start_date=start_date,
            build_other_portfolios=build_other_portfolios,
            bottom_tickers=bottom_tickers,
        )

    for d in rebal_dates:
        d = pd.Timestamp(d).normalize()

        mom     = mom_df.loc[d]
        vol     = vol_df.loc[d]
        inv_vol = inv_vol_df.loc[d]
        rm      = rank_mom_df.loc[d]
        riv     = rank_ivol_df.loc[d]

        # mask progressiva con conteggi per debug
        mask = mom.notna() & inv_vol.notna()
        n_base = int(mask.sum())

        n_ema = n_vol = n_minmom = n_accel = n_base  # default se filtro non attivo

        if filter_ema:
            mask &= prices.loc[d] > ema_df.loc[d]
            n_ema = int(mask.sum())

        if filter_volatility:
            mask &= vol < vol.quantile(volatility_quantile)
            n_vol = int(mask.sum())

        if filter_min_momentum:
            mask &= mom > min_momentum_threshold
            n_minmom = int(mask.sum())

        combo = momentum_weight * rm + (1.0 - momentum_weight) * riv

        if use_acceleration:
            mask &= accel_df.loc[d] > 0
            combo = combo + _ACCEL_WEIGHT * accel_rank_df.loc[d]
            n_accel = int(mask.sum())

        combo_masked = combo.where(mask)
        rank_records[d] = combo_masked.sort_values(ascending=False)

        # selezione top-N
        valid = combo_masked.dropna()
        if not valid.empty:
            top_list = list(valid.nlargest(n_top).index)
            carried  = False
        elif prev_top is not None:
            top_list = list(prev_top)
            carried  = True
        else:
            top_list = []
            carried  = False

        sel_dates.append(d)
        sel_tickers.append(top_list)
        sel_carried.append(carried)

        if top_list and not carried:
            prev_top = top_list

        # log per ogni rebal_date
        status = "CARRY-FWD" if carried else ("EMPTY    " if not top_list else "OK       ")
        log.detail(
            f"{status} | {d.date()} | "
            f"base={n_base} ema={n_ema} vol={n_vol} mom={n_minmom} accel={n_accel} | "
            f"selected={top_list}"
        )

        # pesi equal-weight
        w = pd.Series(0.0, index=cols)
        valid_top = [t for t in top_list if t in cols]
        if valid_top:
            w[valid_top] = 1.0 / len(valid_top)
        w_rot[d] = w

        log.trace(f"  w_rot[{d.date()}] non-zero: { {k: round(v,4) for k,v in w.items() if v > 0} }")

        # portafogli aggiuntivi
        if build_other_portfolios:
            w1 = mom.where(mask, 0.0);  s1 = float(w1.sum())
            w_mom[d] = w1.div(s1) if np.isfinite(s1) and s1 else w1 * 0.0

            w2 = inv_vol.where(mask, 0.0); s2 = float(w2.sum())
            w_rp[d]  = w2.div(s2) if np.isfinite(s2) and s2 else w2 * 0.0

        # bottom tickers (sperimentale)
        if bottom_tickers:
            if not valid.empty:
                bot_list = list(valid.nsmallest(n_top).index)
                prev_bottom = bot_list
            elif prev_bottom is not None:
                bot_list = list(prev_bottom)
                log.detail(f"  BOTTOM CARRY-FWD | {d.date()} | {bot_list}")
            else:
                bot_list = []
            bottom_sel.append(bot_list)

    # ── 6) Matrice pesi → shift(1) ────────────────────────────────────────────
    def _to_shifted(w_dict: dict) -> pd.DataFrame:
        df = pd.DataFrame(w_dict).T.reindex(idx_full).ffill().fillna(0.0)
        shifted = df.shift(1).ffill().fillna(0.0)
        log.trace(
            f"  weight matrix: shape={shifted.shape}  "
            f"non-zero rows={(shifted.sum(axis=1) > 0).sum()}"
        )
        return shifted

    w_rot_sh = _to_shifted(w_rot)

    w_mom_sh = _to_shifted(w_mom) if build_other_portfolios else None
    w_rp_sh  = _to_shifted(w_rp)  if build_other_portfolios else None

    # ── 7) Taglio start_date ──────────────────────────────────────────────────
    if start_date is not None:
        s_ts = pd.Timestamp(start_date).normalize()
        n_before = len(prices)
        w_rot_sh = w_rot_sh.loc[s_ts:]
        prices   = prices.loc[s_ts:]
        if bench_px is not None:
            bench_px = bench_px.loc[s_ts:]
        if build_other_portfolios:
            w_mom_sh = w_mom_sh.loc[s_ts:]
            w_rp_sh  = w_rp_sh.loc[s_ts:]
        log.info(f"start_date cut: {s_ts.date()}  ({n_before - len(prices)} righe rimosse)")

    log.info(
        f"Done: selections={len(sel_dates)}  "
        f"carried={sum(sel_carried)}  "
        f"empty={sum(1 for t in sel_tickers if not t)}"
    )

    # ── 8) Costruzione portafogli VBT ─────────────────────────────────────────
    pf_rot = _build_vbt_portfolio(prices, w_rot_sh, init_cash)

    pf_bh = None
    if bench_px is not None:
        pf_bh = _build_vbt_bh(bench_px, prices.index, init_cash)

    pf_mom = pf_rp = None
    if build_other_portfolios:
        pf_mom = _build_vbt_portfolio(prices, w_mom_sh, init_cash)
        pf_rp  = _build_vbt_portfolio(prices, w_rp_sh,  init_cash)

    # ── 9) Plot ───────────────────────────────────────────────────────────────
    if plot:
        _plot_results(
            pf_rot=pf_rot,
            pf_bh=pf_bh,
            pf_mom=pf_mom,
            pf_rp=pf_rp,
            init_cash=init_cash,
            portfolio_name=portfolio_name,
            width=vbt_plot_width,
        )

    # ── 10) Output DataFrames ─────────────────────────────────────────────────
    selections = pd.DataFrame(
        {"Top_Tickers": sel_tickers, "carried": sel_carried},
        index=pd.DatetimeIndex(sel_dates, name="RebalanceDate"),
    )

    rankings = pd.DataFrame(rank_records).T
    rankings.index.name = "RebalanceDate"

    sel_bottom_df = None
    if bottom_tickers:
        sel_bottom_df = pd.DataFrame(
            {"Bottom_Tickers": bottom_sel},
            index=pd.DatetimeIndex(sel_dates, name="RebalanceDate"),
        )

    return RotationalVbtResult(
        pf_rot=pf_rot,
        pf_bh=pf_bh,
        selections=selections,
        rankings=rankings,
        rebal_dates=rebal_dates,
        pf_mom=pf_mom,
        pf_rp=pf_rp,
        sel_bottom=sel_bottom_df,
    )


# ─────────────────────────────────────────────────────────────────────────────
# FALLBACK: rebal_dates vuoto
# ─────────────────────────────────────────────────────────────────────────────

def _build_empty_result(
    prices: pd.DataFrame,
    bench_px: pd.Series | None,
    cols: pd.Index,
    init_cash: float,
    start_date,
    build_other_portfolios: bool,
    bottom_tickers: bool,
) -> RotationalVbtResult:
    """
    Costruisce un RotationalVbtResult con pesi zero (portafoglio in cash)
    quando rebal_dates è vuoto. Un solo punto di costruzione VBT.
    """
    if start_date is not None:
        s_ts = pd.Timestamp(start_date).normalize()
        prices = prices.loc[s_ts:]
        if bench_px is not None:
            bench_px = bench_px.loc[s_ts:]

    w_zero = pd.DataFrame(0.0, index=prices.index, columns=cols)

    pf_rot = _build_vbt_portfolio(prices, w_zero, init_cash)
    pf_bh  = _build_vbt_bh(bench_px, prices.index, init_cash) if bench_px is not None else None

    empty_sel = pd.DataFrame(
        {"Top_Tickers": pd.Series(dtype=object), "carried": pd.Series(dtype=bool)},
        index=pd.DatetimeIndex([], name="RebalanceDate"),
    )
    empty_rank = pd.DataFrame(index=pd.DatetimeIndex([], name="RebalanceDate"), columns=cols)
    empty_dates = pd.DatetimeIndex([])

    return RotationalVbtResult(
        pf_rot=pf_rot,
        pf_bh=pf_bh,
        selections=empty_sel,
        rankings=empty_rank,
        rebal_dates=empty_dates,
        pf_mom=_build_vbt_portfolio(prices, w_zero, init_cash) if build_other_portfolios else None,
        pf_rp=_build_vbt_portfolio(prices, w_zero, init_cash)  if build_other_portfolios else None,
        sel_bottom=pd.DataFrame(
            {"Bottom_Tickers": pd.Series(dtype=object)},
            index=pd.DatetimeIndex([], name="RebalanceDate"),
        ) if bottom_tickers else None,
    )


# ─────────────────────────────────────────────────────────────────────────────
# PLOT
# ─────────────────────────────────────────────────────────────────────────────

def _plot_results(pf_rot, pf_bh, pf_mom, pf_rp, init_cash, portfolio_name, width):
    import plotly.graph_objects as go

    fig = go.Figure()

    def _add(pf, name, color=None, opacity=1.0):
        y = pf.value() / init_cash
        kw = dict(x=y.index, y=y, mode="lines", name=name, opacity=opacity)
        if color:
            kw["line"] = dict(color=color, width=2)
        fig.add_trace(go.Scatter(**kw))

    _add(pf_rot, "Rotational")
    if pf_mom is not None:
        _add(pf_mom, "Momentum puro")
    if pf_rp is not None:
        _add(pf_rp,  "Risk-Parity puro")
    if pf_bh is not None:
        _add(pf_bh, "Benchmark", color="gray", opacity=0.8)

    fig.update_layout(
        title=f"{portfolio_name} – Rendimenti cumulati",
        yaxis_tickformat=".0%",
        width=width,
        height=600,
        template="plotly_white",
        xaxis=dict(
            rangeselector=dict(buttons=[
                dict(count=1,  label="1M",  step="month", stepmode="backward"),
                dict(count=3,  label="3M",  step="month", stepmode="backward"),
                dict(count=6,  label="6M",  step="month", stepmode="backward"),
                dict(count=1,  label="YTD", step="year",  stepmode="todate"),
                dict(step="all"),
            ]),
            rangeslider=dict(visible=True),
            type="date",
        ),
    )
    fig.show()


# ─────────────────────────────────────────────────────────────────────────────
# RETROCOMPATIBILITÀ: unpacking del vecchio formato tuple
# ─────────────────────────────────────────────────────────────────────────────

def unpack_vbt_result(result: RotationalVbtResult, mode: str = "default") -> tuple:
    """
    Converte un RotationalVbtResult nel formato tuple dell'API precedente.
    Utile per codice legacy che fa unpacking diretto.

    mode='default'
        → (pf_rot, pf_bh, sel_tickers_rot_w, rankings_df)
    mode='with_others'
        → (pf_rot, pf_mom, pf_rp, pf_bh, sel_tickers_rot_w, rankings_df)
    mode='with_bottom'
        → (pf_rot, pf_bh, sel_tickers_rot_w, rankings_df, sel_bottom)
    mode='full'
        → (pf_rot, pf_mom, pf_rp, pf_bh, sel_tickers_rot_w, rankings_df, sel_bottom)
    """
    modes = {
        "default":     (result.pf_rot, result.pf_bh, result.selections, result.rankings),
        "with_others": (result.pf_rot, result.pf_mom, result.pf_rp, result.pf_bh, result.selections, result.rankings),
        "with_bottom": (result.pf_rot, result.pf_bh, result.selections, result.rankings, result.sel_bottom),
        "full":        (result.pf_rot, result.pf_mom, result.pf_rp, result.pf_bh, result.selections, result.rankings, result.sel_bottom),
    }
    if mode not in modes:
        raise ValueError(f"mode='{mode}' non valido. Scegliere tra: {list(modes)}")
    return modes[mode]

In [ ]:
# Metrics (backtest non ottimizati)

def build_benchmark(benchmark_portfolio: dict, start_date="2018-01-01", end_date=None, auto_adjust: bool = True) -> pd.Series:
    """
    Crea un benchmark sintetico a partire da un dizionario {ticker: peso}.
    Scarica i dati da yfinance, li normalizza e li aggrega secondo i pesi.

    Parametri:
    -----------
    benchmark_portfolio : dict
        Dizionario dei ticker e dei pesi (es. {'SPY': 0.5, 'GLD': 0.3, 'TLT': 0.2})
    start_date : str
        Data di inizio in formato 'YYYY-MM-DD'
    end_date : str
        Data di fine. Se None, usa la data attuale.

    Ritorna:
    --------
    benchmark_series : pd.Series
        Serie storica dei prezzi del benchmark ponderato.
    """
    tickers = list(benchmark_portfolio.keys())
    weights = pd.Series(benchmark_portfolio)

    # Scarica i dati da yfinance
    data = yf.download(tickers, start=start_date, end=end_date,auto_adjust=auto_adjust)["Close"]
    data = data.dropna(how="all")  # rimuove righe completamente NaN

    # Normalizza ogni colonna a 1 all'inizio
    data_norm = data / data.iloc[0]

    # Allinea i pesi solo ai ticker presenti nei dati scaricati
    common_tickers = [ticker for ticker in weights.index if ticker in data_norm.columns]
    weights = weights[common_tickers]
    data_norm = data_norm[common_tickers]

    # Calcolo benchmark come media ponderata
    benchmark_series = (data_norm * weights).sum(axis=1)

    return benchmark_series.dropna()


def analyze_portfolio_metrics(
    port_cumrets: pd.DataFrame,
    portfolio_name = "Portafoglio Rotazionale",
    benchmark_cumret: pd.DataFrame = None,
    freq: str = "D",
    sort_by: str = "CAGR (%)",
    ascending: bool = False,
    plot_radar: bool = False,
    radar_metrics: Union[str, List[str]] = "all",
    highlight_best: bool = True
) -> pd.DataFrame:
    def compute_metrics_from_cum(cum_series: pd.Series) -> Dict[str, float]:
        """Calcola le metriche partendo da una serie di rendimenti cumulativi (>=1)."""
        # rets = cum_series.pct_change().dropna()
        rets = cum_series.pct_change(fill_method=None).dropna()
        if rets.empty or len(rets) < 2:
            return {col: np.nan for col in [
                "Cumulative Return (%)", "Annualized Return (%)", "CAGR (%)",
                "Annualized Volatility (%)", "Sharpe Ratio", "Sortino Ratio",
                "Max Drawdown (%)", "Calmar Ratio", "Win Rate (%)",
                "Avg Daily Return (%)", "Median Daily Return (%)"
            ]}

        # fattore di annualizzazione in base alla frequenza
        ann_factor = {"D": 252, "W": 52, "ME": 12}.get(freq.upper(), 252)

        # --------------------- metriche principali ---------------------
        total_ret = cum_series.iloc[-1] - 1
        duration_days = (cum_series.index[-1] - cum_series.index[0]).days
        total_years = duration_days / 365.25 if duration_days > 0 else np.nan

        # CAGR (compounded)
        cagr = (1 + total_ret) ** (1 / total_years) - 1 if total_years else np.nan

        # Annualized arithmetic return
        ann_return = rets.mean() * ann_factor

        # Volatilità annualizzata
        ann_vol = rets.std(ddof=0) * np.sqrt(ann_factor)

        # Sharpe e Sortino (using arithmetic return)
        sharpe = ann_return / ann_vol if ann_vol else np.nan
        downside_std = rets[rets < 0].std(ddof=0) * np.sqrt(ann_factor)
        sortino = ann_return / downside_std if downside_std else np.nan

        # Drawdown & Calmar
        max_dd = (cum_series / cum_series.cummax() - 1).min()
        calmar = cagr / abs(max_dd) if max_dd else np.nan

        # Win rate
        win_rate = (rets > 0).mean() * 100

        return {
            "Cumulative Return (%)": total_ret * 100,
            "Annualized Return (%)": ann_return * 100,
            "CAGR (%)": cagr * 100,
            "Annualized Volatility (%)": ann_vol * 100,
            "Sharpe Ratio": sharpe,
            "Sortino Ratio": sortino,
            "Max Drawdown (%)": -max_dd * 100,
            "Calmar Ratio": calmar,
            "Win Rate (%)": win_rate,
            "Avg Daily Return (%)": rets.mean() * 100,
            # "Median Daily Return (%)": rets.median() * 100
        }

    def normalize_by_absolute_ranges(df: pd.DataFrame,
                                     ranges: Dict[str, Tuple[float, float]]) -> pd.DataFrame:
        """Normalizza ciascuna colonna di df secondo range assoluti predefiniti."""
        df_norm = pd.DataFrame(index=df.index)
        for col in df.columns:
            if col in ranges:
                min_val, max_val = ranges[col]
                df_norm[col] = (df[col] - min_val) / (max_val - min_val)
            else:
                df_norm[col] = df[col]
        return df_norm.clip(0, 1)

    print(f'{Emoji.SETUP} Analisi Portfolio Rotazionale ({BOLD}{portfolio_name}{RESET}):\n')
    
    # -----------------------------------------------------------------
    # Prepara dataframe input
    # -----------------------------------------------------------------
    if isinstance(port_cumrets, pd.Series):
        port_cumrets = port_cumrets.to_frame("Rotational")

    metrics = {name: compute_metrics_from_cum(series)
               for name, series in port_cumrets.items()}

    if benchmark_cumret is not None:
        bench_series = benchmark_cumret.squeeze() if isinstance(benchmark_cumret, pd.DataFrame) else benchmark_cumret
        metrics["Benchmark"] = compute_metrics_from_cum(bench_series)

    metrics_df = pd.DataFrame(metrics).T

    # -----------------------------------------------------------------
    # Ordinamento & styling opzionale
    # -----------------------------------------------------------------
    if sort_by in metrics_df.columns:
        metrics_df = metrics_df.sort_values(by=sort_by, ascending=ascending)

    if highlight_best:
        styled = metrics_df.style
        for col in metrics_df.select_dtypes(include=[np.number]).columns:
            reverse = col in ["Annualized Volatility (%)", "Max Drawdown (%)"]
            styled = styled.background_gradient(
                subset=[col],
                cmap="RdYlGn_r" if reverse else "RdYlGn",
                low=0, high=0, axis=0
            )
        display(styled.format("{:.2f}"))

    # -----------------------------------------------------------------
    # Radar chart opzionale
    # -----------------------------------------------------------------
    if plot_radar and len(metrics_df) <= 5:
        if radar_metrics == "basic":
            radar_cols = ["CAGR (%)", "Sharpe Ratio", "Max Drawdown (%)", "Win Rate (%)"]
        elif radar_metrics == "all":
            radar_cols = list(metrics_df.columns)
        elif isinstance(radar_metrics, list):
            radar_cols = radar_metrics
        else:
            radar_cols = ["CAGR (%)", "Sharpe Ratio", "Max Drawdown (%)", "Win Rate (%)"]

        abs_ranges = {
            "CAGR (%)": (0, 20),
            "Annualized Return (%)": (0, 20),
            "Sharpe Ratio": (0, 2),
            "Sortino Ratio": (0, 3),
            "Max Drawdown (%)": (0, 50),
            "Win Rate (%)": (0, 100),
            "Annualized Volatility (%)": (0, 30),
            "Calmar Ratio": (0, 2),
            "Avg Daily Return (%)": (0, 0.3),
            "Median Daily Return (%)": (0, 0.3),
            "Cumulative Return (%)": (0, 200)
        }

        radar_df = normalize_by_absolute_ranges(
            metrics_df[radar_cols].fillna(0), abs_ranges
        )

        fig = go.Figure()
        for idx, row in radar_df.iterrows():
            fig.add_trace(go.Scatterpolar(
                r=row.tolist(),
                theta=radar_cols,
                fill='toself',
                name=idx
            ))

        fig.update_layout(
            polar=dict(radialaxis=dict(visible=True, tickformat=".1f", range=[0, 1])),
            title="Radar Chart normalizzato su range assoluti",
            height=750, width=900,
            template="plotly_white",
            showlegend=True
        )
        fig.show()

    return metrics_df.round(2)

def pf_stats_aligned(
    pf,
    benchmark: pd.Series | None = None,
    analysis_start_date: str | pd.Timestamp | None = None,
    analysis_end_date: str | pd.Timestamp | None = None,
    rebase_to: float = 100_000.0,
    annualization: int = 252,
    label_pf: str = "Portfolio",
    label_bench: str = "Benchmark",
    return_series: bool = False,
) -> dict:
    """
    Calcola metriche equity-based ALLINEATE su una finestra di analisi comune,
    indipendentemente dal fatto che pf sia stato costruito con buffer (warm-up).

    Policy:
      - Finestra = [analysis_start_date, analysis_end_date] se forniti.
      - Se analysis_end_date è None -> usa ultimo giorno comune tra equity pf e benchmark (se presente),
        altrimenti ultimo giorno equity pf.
      - Benchmark viene riallineato sul calendario della equity del pf via reindex+ffill+bfill.
      - Equity pf e benchmark vengono REBASED a `rebase_to` sul primo giorno della finestra.
      - Metriche: Total Return, CAGR, Vol Ann, Sharpe (rf=0), Max DD.

    Ritorna un dict:
      {
        "pf": metrics_pf,
        "benchmark": metrics_bench or None,
        "window": (start_ts, end_ts),
        "series": {"equity_pf":..., "equity_bench":...}   # solo se return_series=True
      }
    """
    def _to_ts(x):
        if x is None:
            return None
        return pd.Timestamp(x).normalize()

    def _rebase(s: pd.Series, base: float) -> pd.Series:
        first = float(s.iloc[0])
        if first == 0 or not np.isfinite(first):
            raise ValueError("Rebase impossibile: primo valore nullo/non finito.")
        return (s / first) * float(base)

    def _equity_metrics(eq: pd.Series) -> dict:
        eq = eq.dropna().astype(float)
        if len(eq) < 3:
            raise ValueError("Equity troppo corta per metriche.")

        total_ret = (eq.iloc[-1] / eq.iloc[0]) - 1.0
        r = eq.pct_change().dropna()

        n_days = (eq.index[-1] - eq.index[0]).days
        years = n_days / 365.25 if n_days > 0 else np.nan
        cagr = (eq.iloc[-1] / eq.iloc[0]) ** (1.0 / years) - 1.0 if years and np.isfinite(years) and years > 0 else np.nan

        vol_ann = r.std(ddof=0) * np.sqrt(annualization)
        sharpe = (r.mean() * annualization) / vol_ann if vol_ann and np.isfinite(vol_ann) and vol_ann > 0 else np.nan

        peak = eq.cummax()
        dd = (eq / peak) - 1.0

        return {
            "Start": eq.index[0],
            "End": eq.index[-1],
            "Days": int((eq.index[-1] - eq.index[0]).days),
            "Start Value": float(eq.iloc[0]),
            "End Value": float(eq.iloc[-1]),
            "Total Return %": 100.0 * float(total_ret),
            "CAGR %": 100.0 * float(cagr) if np.isfinite(cagr) else np.nan,
            "Vol Ann %": 100.0 * float(vol_ann) if np.isfinite(vol_ann) else np.nan,
            "Sharpe": float(sharpe) if np.isfinite(sharpe) else np.nan,
            "Max DD %": 100.0 * float(dd.min()),
        }

    # ------------------------------------------------------------
    # 1) Equity pf
    # ------------------------------------------------------------
    eq_pf = pf.value().copy()
    eq_pf.index = pd.to_datetime(eq_pf.index).normalize()
    eq_pf = eq_pf.sort_index()

    # ------------------------------------------------------------
    # 2) Benchmark (opzionale)
    # ------------------------------------------------------------
    eq_bench = None
    if benchmark is not None:
        b = benchmark.copy()
        b.index = pd.to_datetime(b.index).normalize()
        b = b.sort_index()
        eq_bench = b

    # ------------------------------------------------------------
    # 3) Definisci finestra di analisi
    # ------------------------------------------------------------
    start = _to_ts(analysis_start_date) or pd.Timestamp(eq_pf.index.min()).normalize()

    if analysis_end_date is not None:
        end = _to_ts(analysis_end_date)
    else:
        if eq_bench is not None:
            end = min(pd.Timestamp(eq_pf.index.max()).normalize(),
                      pd.Timestamp(eq_bench.index.max()).normalize())
        else:
            end = pd.Timestamp(eq_pf.index.max()).normalize()

    if start > end:
        start, end = end, start

    # clip pf
    eq_pf_c = eq_pf.loc[(eq_pf.index >= start) & (eq_pf.index <= end)]
    if eq_pf_c.empty:
        raise ValueError("Finestra analisi non interseca l'equity del portafoglio.")

    # clip + align benchmark sul calendario del pf
    eq_bench_c = None
    if eq_bench is not None:
        eq_bench_c = eq_bench.loc[(eq_bench.index >= start) & (eq_bench.index <= end)]
        # riallinea su calendario pf
        eq_bench_c = eq_bench_c.reindex(eq_pf_c.index).ffill().bfill()
        if eq_bench_c.empty:
            eq_bench_c = None

    # ------------------------------------------------------------
    # 4) Rebase (100k di default)
    # ------------------------------------------------------------
    eq_pf_c = _rebase(eq_pf_c, rebase_to)
    if eq_bench_c is not None:
        eq_bench_c = _rebase(eq_bench_c, rebase_to)

    # ------------------------------------------------------------
    # 5) Metriche
    # ------------------------------------------------------------
    m_pf = _equity_metrics(eq_pf_c)
    m_b = _equity_metrics(eq_bench_c) if eq_bench_c is not None else None

    out = {
        "window": (start, end),
        "pf": {"label": label_pf, "metrics": m_pf},
        "benchmark": {"label": label_bench, "metrics": m_b} if m_b is not None else None,
    }

    if return_series:
        out["series"] = {"equity_pf": eq_pf_c, "equity_bench": eq_bench_c}

    return out

def calc_vbt_internal_benchmark_buyhold_equal_cash(
    prices_wide: pd.DataFrame,
    init_cash: float = 100000.0,
    start: str | pd.Timestamp | None = None,
    end: str | pd.Timestamp | None = None,
    ffill_prices: bool = True,
):
    """
    Calcola MANUALMENTE il "benchmark interno" che VectorBT riporta come
    `Benchmark Return [%]` quando il Portfolio è multi-ticker (multi-colonna).

    REGOLA CHIAVE (da fissare nel framework):
    - Il benchmark interno NON è un equal-weight ribilanciato giornalmente.
    - Il benchmark interno è un BUY & HOLD "equal-cash" (paniere statico):
        1) alla data iniziale si investe `init_cash` in parti uguali tra gli asset disponibili;
        2) si acquistano quote (shares) fisse e si tengono fino alla fine del periodo;
        3) nessun ribilanciamento in corso d'opera;
        4) equity(t) = somma_i( shares_i * price_i(t) ).

    Perché è diverso dal tuo primo tentativo (media dei rendimenti):
    - fare la media dei rendimenti giornalieri equivale a un portafoglio
      equal-weight RIBILANCIATO OGNI GIORNO (daily rebalanced),
      che spesso produce un TR diverso (nel tuo caso più alto).
    - vbt invece usa un benchmark che rappresenta "compra e tieni l'universo"
      senza rotazione e senza ribilanciamento.

    Parametri
    ----------
    prices_wide : pd.DataFrame
        DataFrame prezzi in formato wide:
        - index = Date (DatetimeIndex o convertibile)
        - columns = tickers
        - valori = prezzi (nel tuo framework: Close già adjusted/total return)
        Può contenere NaN (es. titoli non quotati all’inizio o buchi dati).

    init_cash : float
        Capitale iniziale del benchmark.

    start, end : str | pd.Timestamp | None
        Periodo di calcolo. Se None, usa tutto l'indice disponibile in prices_wide.
        Nota: le "date effettive" usate sono le prime/ultime righe risultanti
        dopo lo slicing, e vengono ritornate in output.

    ffill_prices : bool
        Se True, applica forward-fill ai prezzi DOPO lo slicing temporale.
        Serve a gestire buchi sporadici (missing data) senza interrompere l'equity.
        Non "crea" prezzi prima della prima osservazione: i NaN iniziali restano NaN,
        quindi un ticker senza prezzo alla data iniziale viene escluso dal paniere.

    Output (dict)
    -------------
    start_used : Timestamp
        Prima data effettivamente presente dopo slicing.

    end_used : Timestamp
        Ultima data effettivamente presente dopo slicing.

    n_assets : int
        Numero di asset inclusi nel paniere (quelli con prezzo valido alla start).

    assets : list[str]
        Elenco ticker effettivamente inclusi (prezzo valido alla start).

    shares : pd.Series
        Shares fissi comprati alla start per ciascun ticker incluso.

    equity : pd.Series
        Equity line del benchmark buy&hold equal-cash.

    end_value : float
        Valore finale del benchmark.

    total_return_pct : float
        Total Return (%) = (end_value / init_cash - 1) * 100.
        Questo è il valore che deve coincidere con `pf_rot.stats()["Benchmark Return [%]"]`
        quando `prices_wide` è lo stesso close usato per costruire pf_rot.
    """
    if not isinstance(prices_wide, pd.DataFrame):
        raise TypeError("prices_wide deve essere un DataFrame wide (Date index, tickers columns).")

    # --- Normalizza indice data e conserva solo colonne numeriche ---
    df = prices_wide.copy()
    df.index = pd.to_datetime(df.index)
    df = df.select_dtypes(include=[np.number])

    # --- Slicing temporale richiesto (usa i dati presenti in df) ---
    if start is not None:
        df = df.loc[df.index >= pd.to_datetime(start)]
    if end is not None:
        df = df.loc[df.index <= pd.to_datetime(end)]

    if df.empty:
        raise ValueError("DataFrame vuoto dopo slicing per date.")

    # --- Date effettive usate ---
    start_used = df.index[0]
    end_used = df.index[-1]

    # --- Gestione buchi dati: forward-fill interno al periodo ---
    # Nota: NON risolve NaN iniziali (prima osservazione). Quelli restano NaN.
    if ffill_prices:
        df = df.ffill()

    # --- Selezione asset inclusi: devono avere prezzo valido alla data iniziale ---
    # I ticker senza prezzo a start non possono essere comprati (shares non definibili).
    p0 = df.iloc[0]
    valid_cols = p0.dropna().index.tolist()

    if len(valid_cols) == 0:
        raise ValueError("Nessun ticker con prezzo valido alla start (dopo ffill se attivo).")

    dfv = df[valid_cols]
    p0v = dfv.iloc[0]

    # --- Regola equal-cash: investe init_cash/N in ogni asset incluso ---
    cash_per_asset = init_cash / len(valid_cols)

    # --- Shares fissi comprati alla start e poi mantenuti (NO rebalance) ---
    shares = cash_per_asset / p0v

    # --- Equity line: somma del valore delle posizioni buy&hold ---
    equity = (dfv * shares).sum(axis=1)

    # --- Total Return rispetto a init_cash (non rispetto a equity.iloc[0]) ---
    total_return_pct = (equity.iloc[-1] / init_cash - 1.0) * 100.0

    return {
        "start_used": start_used,
        "end_used": end_used,
        "n_assets": len(valid_cols),
        "assets": valid_cols,
        "shares": shares,
        "equity": equity,
        "end_value": float(equity.iloc[-1]),
        "total_return_pct": float(total_return_pct),
    }


In [ ]:
# Run time

def r_run_portfolio(
    portfolio: dict,
    year: int | None = None,
    *,
    # --- dry-run ---
    dry_run: bool = False,
    # --- download ---
    start_date=None,            # default: now() se None
    end_date=None,
    lookback_buffer_days: int = 365 * 2,
    show_progress: bool = False,
    wfo_results_dir: str = "WFO_R_RESULTS",
    # --- WFO summary ---
    wfo_file_save: str | None = None,   # default: f"{portfolio_title}_{year}.wfo_summary.csv"
    # --- report ---
    report_end_date=None,                 # None oppure 'YYYY-MM-DD'
    sender_email: str = "",
    sender_password: str = "",
    recipient_email: str = "",
    subject=None,
    verbose: bool = False,
    debug: bool = False,
):
    """
    Esegue la pipeline operativa del portafoglio rotazionale a partire
    da una struttura portfolio predefinita.

    Struttura attesa:
    portfolio = {
        "Title": "Germany Plan",
        "tickers": [...]
    }

    Modalità dry-run:
      - stampa le azioni che verrebbero eseguite
      - non scarica dati
      - non legge file
      - non invia email

    Dipendenze attese già disponibili nel progetto:
      - now()
      - download_data(...)
      - collect_selections_from_summary(...)
      - extract_operational_params_from_summary(...)
      - send_rotational_portfolio_report(...)
    """

    # --- Validazione minima portfolio ---
    if not isinstance(portfolio, dict):
        raise TypeError("portfolio deve essere un dict, es. {'Title': '...', 'tickers': [...]}")

    if "Title" not in portfolio or "tickers" not in portfolio:
        raise KeyError("portfolio deve contenere le chiavi obbligatorie: 'Title' e 'tickers'")

    portfolio_title = portfolio["Title"]
    tickers = portfolio["tickers"]

    # Supporto a indici: se tickers e' una stringa (ossia non e' una lista predefinita) creo la lista di tickers:
    tickers = (
        extract_tickers_from_wikipedia(tickers,exclude=["GOOG"],rename={"BRK.B": "BRK-B"})
        if isinstance(tickers, str)
        else list(tickers)
    )

    if not isinstance(tickers, (list, tuple)) or len(tickers) == 0:
        raise ValueError("portfolio['tickers'] deve essere una lista/tupla non vuota di ticker")

    # --- Year default: anno corrente ---
    if year is None:
        year = int(pd.Timestamp.now().year)

    
    # --- WFO summary filename ---
    if wfo_file_save is None:
        wfo_file_save = f"{portfolio_title}_{year}.wfo_summary.csv"

    wfo_file_save=f"{wfo_results_dir}/{wfo_file_save}"
    
    # --- Start date default ---
    if start_date is None:
        start_date = now()

    # --- Download start (buffer) ---
    download_start_date = (
        pd.to_datetime(start_date) - timedelta(days=int(lookback_buffer_days))
    ).strftime("%Y-%m-%d")

    # --- DRY RUN: stampa piano azioni e termina ---
    if dry_run:
        print("[DRY-RUN] r_run_portfolio")
        print(f"  - portfolio_title      : {portfolio_title}")
        print(f"  - year                 : {year}")
        print(f"  - tickers              : {len(tickers)} tickers")
        print(f"  - start_date (input)   : {start_date}")
        print(f"  - lookback_buffer_days : {lookback_buffer_days}")
        print(f"  - download_start_date  : {download_start_date}")
        print(f"  - end_date             : {end_date}")
        print(f"  - show_progress        : {show_progress}")
        print(f"  - wfo_file_save        : {wfo_file_save}")
        print(f"  - report_end_date                : {report_end_date}")
        print("  - Azioni che verrebbero eseguite:")
        print("    1) download_data(tickers, download_start_date, end_date, show_progress=...)")
        print("    2) pd.read_csv(wfo_file_save, index_col='Window')")
        print("    3) collect_selections_from_summary(summary_df, stocks_data, debug=...)")
        print("    4) extract_operational_params_from_summary(summary_df, report_end_date)")
        print("    5) send_rotational_portfolio_report(..., rebalance_frequency, n_top)")
        print("    6) pd.write_csv(sel_tickers_file = {portfolio_title}_{current_year}_sel_tickers_current_year.csv)")

        print("  - Email params:")
        print(f"    sender_email     : {sender_email}")
        print(f"    recipient_email  : {recipient_email}")
        print(f"    subject          : {subject}")
        print(f"    verbose          : {verbose}")
        return {
            "dry_run": True,
            "portfolio": portfolio_title,
            "year": year,
            "wfo_file_save": wfo_file_save,
            "download_start_date": download_start_date,
            "end_date": end_date,
            # richiesti: presenti ma non calcolabili in dry-run
            "sel_tickers": None,
            "summary_df": None,
        }

    # --- Download dati ---
    stocks_data = download_data(
        tickers,
        download_start_date,
        end_date,
        show_progress=show_progress,
        auto_adjust=False
    )

    # --- Carica summary WFO ---
    summary_df=load_wfo_summary(wfo_file_save)
    
    # --- Selezioni tickers dal summary ---
    sel_tickers = collect_selections_from_summary(
        summary_df=summary_df,
        stocks_data=stocks_data,
        debug=debug
    )
    # ------------------------------------------------------------
    # Salvataggio selezioni ticker dell'anno corrente
    # ------------------------------------------------------------
    current_year = now().year

    sel_tickers_current_year = sel_tickers[
        sel_tickers.index.year == current_year
    ]

    sel_tickers_file = f"{portfolio_title}_{current_year}_sel_tickers_current_year.csv"
    sel_tickers_file = f"{wfo_results_dir}/{sel_tickers_file}"


    if not sel_tickers_current_year.empty:
        sel_tickers_current_year.to_csv(sel_tickers_file)
        if verbose:
            print(f"[INFO] Selezioni anno {current_year} salvate in: {sel_tickers_file}")
    else:
        if verbose:
            print(f"[INFO] Nessuna selezione disponibile per l'anno {current_year}")

    # --- Parametri operativi ---
    params_ops = extract_operational_params_from_summary(summary_df, report_end_date)
    rebalance_frequency = params_ops["rebalance_frequency"]
    n_top = params_ops["n_top"]

    # --- Invio report ---
    send_rotational_portfolio_report(
        sel_tickers,
        portfolio_title,
        report_end_date,
        sender_email,
        sender_password,
        recipient_email,
        subject,
        verbose,
        rebalance_frequency,
        n_top,
        trading_index=stocks_data.index
    )


    return {
        "dry_run": False,
        "portfolio": portfolio_title,
        "year": year,
        "wfo_file_save": wfo_file_save,
        "download_start_date": download_start_date,
        "end_date": end_date,
        # richiesti: aggiunti in output
        "sel_tickers": sel_tickers,
        "summary_df": summary_df,
        # resto invariato
        "params_ops": params_ops,
        "rebalance_frequency": rebalance_frequency,
        "n_top": n_top,
        "stocks_data": stocks_data
    }
def extract_operational_params_from_summary(
    summary_df: pd.DataFrame,
    today: str | pd.Timestamp | None = None
) -> dict:
    """
    Estrae i parametri operativi dal risultato WFO per l'ANNO di riferimento.

    Regola:
    - se today=None -> usa l'anno corrente (pd.Timestamp.today()).
    - se today è valorizzato -> usa l'anno di today (accetta str o Timestamp).

    Se summary_df è indicizzato per "Window" nel formato:
        'YYYY-MM-DD→YYYY-MM-DD'
    allora seleziona la riga la cui finestra INIZIA nell'anno target (tipico: 2026-01-01→2026-12-31).

    Ritorna (minimo indispensabile):
    - rebalance_frequency (str)
    - n_top (int)
    """

    if summary_df is None or summary_df.empty:
        raise ValueError("summary_df è vuoto: impossibile estrarre parametri operativi")

    # --- anno target ---
    if today is None:
        target_year = pd.Timestamp.today().year
    else:
        target_year = pd.to_datetime(today).year

    # --- ricava lo start-year da index "Window" (formato 'start→end') ---
    idx = summary_df.index.astype(str)

    def _start_year(window_str: str) -> int | None:
        try:
            start_str = window_str.split("→", 1)[0]
            return pd.to_datetime(start_str).year
        except Exception:
            return None

    start_years = pd.Series([_start_year(x) for x in idx], index=summary_df.index)

    # --- seleziona la riga per l'anno target ---
    mask = start_years == int(target_year)
    if not mask.any():
        available_years = sorted({y for y in start_years.dropna().astype(int).tolist()})
        raise ValueError(
            f"Nessuna finestra WFO trovata per l'anno target {target_year}. "
            f"Anni disponibili in summary_df: {available_years}"
        )

    # Se ci sono più righe per lo stesso anno (caso raro), prendi l'ultima occorrenza
    row = summary_df.loc[mask].iloc[-1]

    rebalance_frequency = row.get("rebalance_frequency")
    n_top = row.get("n_top")

    if pd.isna(rebalance_frequency):
        raise ValueError(f"rebalance_frequency mancante per l'anno {target_year} in summary_df")

    if pd.isna(n_top):
        raise ValueError(f"n_top mancante per l'anno {target_year} in summary_df")

    return {
        "rebalance_frequency": str(rebalance_frequency),
        "n_top": int(n_top),
        "year": int(target_year),
    }

def generate_rotational_portfolio_report(
    sel_tickers,
    portfolio_name,
    today,
    sender_email="",
    sender_password="",
    recipient_email="",
    subject=None,
    verbose=False,
    # --- parametri operativi espliciti (da WFO) ---
    rebalance_frequency: str | None = None,
    n_top: int | None = None,
    attachments=None,
    # --- NEW: calendario di borsa reale (es. stocks_data.index) ---
    trading_index=None,
):
    """
    Genera e stampa/invia un report HTML per un portafoglio rotazionale.

    Regole (robuste, una volta per tutte):
    - target di calendario: fine mese (ME) / fine trimestre (QE)
    - effective rebalance date: ultima seduta <= target (snap usando trading_index)
    - execution day: prima seduta > effective (sempre da trading_index)
    - il report viene prodotto SEMPRE:
        * se today == execution day -> mostra azioni operative (buy/sell/keep)
        * altrimenti -> mostra stato, holdings correnti, prossime date chiave
    """
    import datetime
    import pandas as pd

    # Normalizza today per subject/label
    if today is None:
        today_ts = pd.Timestamp.today().normalize()
    else:
        today_ts = pd.to_datetime(today).normalize()

    today_str = today_ts.strftime("%Y-%m-%d")

    html_report = analyze_rebalance_actions_for_report(
        sel_tickers=sel_tickers,
        today=today_ts,
        rebalance_frequency=rebalance_frequency,
        n_top=n_top,
        verbose=verbose,
        trading_index=trading_index,
    )

    # Stampa o invia report
    if recipient_email:
        if subject is None:
            subject = f"[TS_LAB] Report di Portafoglio {portfolio_name} ({today_str})"

        send_email_report(
            sender_email,
            sender_password,
            recipient_email,
            subject,
            html_report,
            attachments
        )
    else:
        if verbose:
            print("\n*** Nessun destinatario specificato, non spedisco l'email. Report HTML: ***\n")
        print(html_report)

send_rotational_portfolio_report = generate_rotational_portfolio_report

def analyze_rebalance_actions_for_report(
    sel_tickers: pd.DataFrame,
    today: str | pd.Timestamp | None = None,
    rebalance_frequency: str | None = None,
    n_top: int | None = None,
    verbose: bool = False,
    trading_index=None,
) -> str:
    import pandas as pd
    from typing import List
    
    """
    Analizza e descrive le azioni di ribilanciamento di un portafoglio rotazionale,
    secondo una logica **puramente calendariale**, progettata per un job automatico
    eseguito **tutti i giorni alle 08:00**.
    
    LOGICA DI TRIGGER (CALENDARIO, NON MERCATO)
    ------------------------------------------
    Il ribilanciamento NON dipende dalla disponibilità dei dati di chiusura del giorno,
    ma esclusivamente dal calendario:
    
    - ME (Mensile):        ribilanciamento il **1° giorno del mese**
    - QE (Trimestrale):    ribilanciamento il **1° giorno del trimestre**
                           (gennaio, aprile, luglio, ottobre)
    - WE (Settimanale):    ribilanciamento il **lunedì**
    
    Se oggi soddisfa la regola di calendario per la frequenza scelta, allora
    oggi è considerato "rebalance day", indipendentemente da:
    - orario di esecuzione,
    - fatto che oggi sia trading day,
    - weekend o festività.
    
    FIXING OPERATIVO (ASOF)
    ----------------------
    Il fixing dei segnali e delle selezioni avviene sempre sull’ultima seduta
    di mercato **effettivamente disponibile**:
    
        ASOF = ultima data in trading_index ≤ today
    
    Questo garantisce che:
    - alle 08:00 si usino solo dati già disponibili,
    - nei weekend/festivi si utilizzi automaticamente l’ultima seduta precedente,
    - non vi sia dipendenza dal fatto che il close del giorno corrente sia già noto.
    
    EXECUTION
    ---------
    Le operazioni NON vengono eseguite su ASOF, ma sono pianificate per:
    
        execution = prima seduta di mercato successiva ad ASOF
    
    Nel report viene esplicitamente indicato che:
    - oggi è il giorno di pianificazione del ribilanciamento,
    - gli ordini vanno eseguiti alla prossima seduta utile.
    
    COMPORTAMENTO IN GIORNI NON DI RIBILANCIAMENTO
    ----------------------------------------------
    Se oggi NON è un giorno di ribilanciamento calendario:
    - la funzione mostra lo stato corrente del portafoglio,
    - riporta l’ultima selezione disponibile,
    - indica chiaramente se oggi è trading day o non trading day.
    
    OBIETTIVO DELLA FUNZIONE
    -----------------------
    Questa funzione è pensata per:
    - reporting operativo,
    - controllo quotidiano delle azioni di rotazione,
    - utilizzo in pipeline automatiche e deterministiche.
    
    La funzione NON:
    - esegue operazioni,
    - scarica dati,
    - assume che i prezzi di oggi siano disponibili.
    
    Restituisce esclusivamente una descrizione HTML delle azioni da intraprendere,
    coerente, ripetibile e indipendente dall’orario di esecuzione.
    """

    # =========================
    # Normalize today (job alle 08:00 → date-only)
    # =========================
    if today is None:
        today_ts = pd.Timestamp.today().normalize()
    else:
        today_ts = pd.to_datetime(today).normalize()

    # =========================
    # Normalize selections index
    # =========================
    sel_df = sel_tickers.copy()
    sel_df.index = pd.to_datetime(sel_df.index).normalize()
    sel_idx = sel_df.index.sort_values().unique()

    if len(sel_idx) == 0:
        return "<h2>⚠️ Nessun dato di selezione disponibile</h2>"

    # =========================
    # Normalize trading calendar
    # =========================
    if trading_index is None:
        return "<h2>⚠️ trading_index non fornito</h2>"

    t_idx = pd.DatetimeIndex(pd.to_datetime(trading_index)).normalize().sort_values().unique()
    if len(t_idx) == 0:
        return "<h2>⚠️ trading_index vuoto</h2>"

    # =========================
    # Helpers
    # =========================
    def _snap_prev(idx, d):
        pos = idx.searchsorted(d, side="right") - 1
        return None if pos < 0 else pd.Timestamp(idx[pos]).normalize()

    def _next(idx, d):
        pos = idx.searchsorted(d, side="right")
        return None if pos >= len(idx) else pd.Timestamp(idx[pos]).normalize()

    def _to_set(tickers_value) -> set:
        if tickers_value is None:
            return set()
        if isinstance(tickers_value, (list, tuple, set)):
            return {t for t in tickers_value if isinstance(t, str) and t}
        if isinstance(tickers_value, pd.Series):
            non_null = tickers_value.dropna()
            if len(non_null) == 0:
                return set()
            return _to_set(non_null.iloc[0])
        return set()
        
    def _get_tickers(df: pd.DataFrame, date) -> set:
        """Accesso robusto alla colonna ticker per una data. Guardrail su NaT e date mancanti."""
        if date is None:
            return set()
        try:
            date = pd.Timestamp(date)
        except Exception:
            return set()
        if pd.isna(date):
            return set()
        col = "Top_Tickers" if "Top_Tickers" in df.columns else "tickers"
        if date not in df.index:
            return set()
        return _to_set(df.at[date, col])

    
    def _market_status(d, idx, asof=None):
        # Weekend: qui possiamo essere certi
        if d.weekday() >= 5:
            return "<p>🏛️ Stato mercato: <b>Weekend</b></p>"
    
        # Giorno feriale: è una seduta potenziale (non diciamo 'aperto')
        if d in idx:
            return "<p>🏛️ Stato mercato: <b>Seduta di trading</b> (dati aggiornati)</p>"
    
        # Feriale ma non presente nei dati: tipico pre-market / provider non aggiornato
        if asof is not None:
            return (
                "<p>🏛️ Stato mercato: <b>Seduta di trading</b> "
                f"(dati non ancora aggiornati; ASOF={pd.Timestamp(asof).date()})</p>"
            )
    
        return "<p>🏛️ Stato mercato: <b>Seduta di trading</b> (dati non ancora aggiornati)</p>"
          
    def _market_status_R2(d, idx, asof=None):
        """
        idx = trading_index basato sui dati (non su calendario exchange).
        d  = today normalized
        asof = ultima data disponibile <= d (se già calcolata)
        """
        # 1) Se oggi è già nei dati: è sicuramente trading day (data disponibile)
        if d in idx:
            return "<p>🏛️ Stato mercato: <b>Trading day</b> (data disponibile)</p>"
    
        # 2) Se oggi NON è nei dati, potrebbe essere:
        #    - mercato chiuso (pre-market, oppure dati non aggiornati)
        #    - oppure weekend
        #    Usiamo un fallback robusto e NON lo chiamiamo mai "weekend/festivo"
        #    a meno che sia davvero weekend.
        if d.weekday() >= 5:  # 5=Saturday, 6=Sunday
            return "<p>🏛️ Stato mercato: <b>Non trading day</b> (weekend)</p>"
    
        # Giorno feriale ma non presente in idx: tipicamente USA alle 08:00
        # (o data provider non ancora aggiornato). Usiamo ASOF per esplicitarlo.
        if asof is not None:
            return (
                "<p>🏛️ Stato mercato: <b>Trading day</b> "
                f"(mercato chiuso / dati non ancora disponibili; ASOF={pd.Timestamp(asof).date()})</p>"
            )
    
        return "<p>🏛️ Stato mercato: <b>Trading day</b> (mercato chiuso / dati non ancora disponibili)</p>"

    def _market_status_R1(d, idx):
        if d in idx:
            return "<p>🏛️ Stato mercato: <b>Trading day</b></p>"
        return "<p>🏛️ Stato mercato: <b>Non trading day</b> (weekend/festivo)</p>"

    def _is_rebalance_day(d, freq):
        f = str(freq).upper()
        if f == "ME":
            return d.day == 1
        if f == "QE":
            return d.day == 1 and d.month in (1, 4, 7, 10)
        if f == "WE":
            return d.weekday() == 0  # Monday
        return False

    def _format_freq(freq):
        return {
            "ME": "Mensile (1° giorno del mese)",
            "QE": "Trimestrale (1° giorno del trimestre)",
            "WE": "Settimanale (lunedì)",
        }.get(freq, "non specificata")

    # =========================
    # Labels
    # =========================
    freq_code = str(rebalance_frequency).upper() if rebalance_frequency else ""
    freq_label = _format_freq(freq_code)
    n_top_label = "non specificato" if n_top is None else f"Top {int(n_top)}"

    # =========================
    # Trigger calendario
    # =========================
    is_rebalance_today = _is_rebalance_day(today_ts, freq_code)

    # =========================
    # ASOF & execution (sempre calendario trading)
    # =========================
    asof = _snap_prev(t_idx, today_ts)
    if asof is None:
        return "<h2>⚠️ Nessuna seduta disponibile ≤ today</h2>"

    execution = _next(t_idx, asof)
    execution_str = execution.date() if execution is not None else "N/D"

    # =========================
    # NON rebalance day → solo stato
    # =========================
    if not is_rebalance_today:
        # last_sel = sel_idx[sel_idx <= today_ts].max()
        # holdings = sorted(_to_set(sel_df.loc[last_sel]))
        
        eligible = sel_idx[sel_idx <= today_ts]
        if len(eligible) == 0:
            return "<h2>⚠️ Nessuna selezione disponibile ≤ oggi</h2>"
        last_sel = eligible.max()
        holdings = sorted(_get_tickers(sel_df, last_sel))

        html = f"<h2>🕒 Oggi NON è data di ribilanciamento ({today_ts.date()})</h2>"
        html += _market_status(today_ts, t_idx)
        html += (
            f"<p>🗓️ Frequenza: <b>{freq_label}</b></p>"
            f"<p>🎯 Target: <b>{n_top_label}</b></p>"
            f"<p>📌 ASOF: <b>{asof.date()}</b></p>"
            f"<p>Ultima selezione: <b>{last_sel.date()}</b></p>"
        )

        if holdings:
            html += "<h3>📌 Portafoglio attuale:</h3><ul>"
            company_data = build_company_df_with_cache(holdings)
            for t in holdings:
                company = company_data.at[t, "Company"] if t in company_data.index else ""
                html += f"<li>{t} – {company}</li>"
            html += "</ul>"

        return html

    # =========================
    # REBALANCE DAY
    # =========================
    eligible_sel = sel_idx[sel_idx <= asof]
    curr_sel = eligible_sel.max()
    prev_sel = eligible_sel[-2] if len(eligible_sel) > 1 else None

    # curr_set = _to_set(sel_df.loc[curr_sel])
    # prev_set = _to_set(sel_df.loc[prev_sel]) if prev_sel is not None else set()
    
    curr_set = _get_tickers(sel_df, curr_sel)
    prev_set = _get_tickers(sel_df, prev_sel)   # _get_tickers gestisce già None → set()

    to_keep = sorted(prev_set & curr_set)
    to_sell = sorted(prev_set - curr_set)
    to_buy  = sorted(curr_set - prev_set)

    company_data = build_company_df_with_cache(to_sell + to_buy + to_keep)

    html = f"<h2>🔄 OGGI È data di ribilanciamento – {today_ts.date()}</h2>"
    html += _market_status(today_ts, t_idx)
    html += (
        f"<p>🗓️ Frequenza: <b>{freq_label}</b></p>"
        f"<p>🎯 Target: <b>{n_top_label}</b></p>"
        f"<p>📌 ASOF (fine periodo precedente): <b>{asof.date()}</b></p>"
        f"<p>🗓️ Execution (prossima seduta): <b>{execution_str}</b></p>"
        f"<p><i>Selezione usata</i>: <b>{curr_sel.date()}</b></p>"
        f"<p style='color:#8a6d3b'>Ordini da eseguire alla prossima seduta utile.</p>"
    )

    def _fmt(lst, title, icon):
        if not lst:
            return ""
        s = f"<h3>{icon} {title} ({len(lst)}):</h3><ul>"
        for t in lst:
            company = company_data.at[t, "Company"] if t in company_data.index else ""
            s += f"<li><b>{t}</b> – {company}</li>"
        return s + "</ul>"

    html += _fmt(to_keep, "Da mantenere", "📌")
    html += _fmt(to_sell, "Da vendere", "❌")
    html += _fmt(to_buy,  "Da acquistare", "✅")

    return html


In [ ]:
"""
Monte Carlo Block Bootstrap per Portfolios Rotazionali
=======================================================

Implementazione completa pronta all'uso per analisi robustezza
portafogli rotazionali tramite block bootstrap.

Autore: Analisi Framework Rotazionale (Claude Code)
Data: Febbraio 2026
"""


# =============================================================================
# CORE: Block Bootstrap Engine
# =============================================================================
def monte_carlo_block_bootstrap_rotational(
    portfolio: vbt.Portfolio,
    sel_tickers_df: pd.DataFrame,
    stocks_data: pd.DataFrame,
    n_simulations: int = 10_000,
    block_size: int = 20,
    init_cash: float = 100_000,
    preserve_mean: bool = False,
    random_seed: Optional[int] = None,
    show_progress: bool = True
) -> pd.DataFrame:

    if random_seed is not None:
        np.random.seed(random_seed)

    # ------------------------------------------------------------
    # FIX: allinea il periodo prezzi al periodo effettivo di selezione
    # ------------------------------------------------------------
    sel_tickers_df = sel_tickers_df.copy()
    sel_tickers_df.index = pd.DatetimeIndex(sel_tickers_df.index)

    start_dt = sel_tickers_df.index.min()
    end_dt   = sel_tickers_df.index.max()

    # Slice coerente: usiamo solo il range in cui esistono selezioni
    stocks_data = stocks_data.loc[start_dt:end_dt].copy()

    # Se per qualche motivo lo slicing svuota, fail-fast
    if stocks_data.empty:
        raise ValueError(
            f"stocks_data vuoto dopo slicing su [{start_dt.date()} - {end_dt.date()}]. "
            "Verifica che stocks_data e sel_tickers_df condividano lo stesso calendario/date."
        )

    # Estrai info dal periodo allineato
    returns = stocks_data.pct_change().fillna(0).infer_objects(copy=False)
    dates = returns.index
    all_tickers = list(stocks_data.columns)
    n_days = len(dates)

    # Rebalance dates: tieni solo quelle presenti nel calendario dei prezzi
    rebal_dates = pd.DatetimeIndex(sel_tickers_df.index)
    rebal_dates = rebal_dates[(rebal_dates >= dates[0]) & (rebal_dates <= dates[-1])]

    if len(rebal_dates) == 0:
        raise ValueError(
            "Nessuna rebal_date di sel_tickers_df cade dentro l'intervallo di stocks_data (dopo slicing)."
        )

    # Pre-allocazione matrice risultati
    sim_equity_curves = np.zeros((n_days, n_simulations))

    # Setup progress bar
    pbar = tqdm(total=n_simulations, desc="MC Bootstrap", disable=not show_progress)

    for sim_idx in range(n_simulations):
        # 1. Bootstrap returns per ogni ticker (blocchi)
        sim_returns_dict = {}

        for ticker in all_tickers:
            ticker_rets = returns[ticker].values
            bootstrapped_rets = _block_bootstrap_single_series(
                ticker_rets,
                block_size=block_size,
                preserve_mean=preserve_mean
            )
            sim_returns_dict[ticker] = bootstrapped_rets

        # 2. Converti in DataFrame
        sim_returns_df = pd.DataFrame(sim_returns_dict, index=dates)

        # 3. Ricostruisci prezzi da returns bootstrappati (base = primo giorno del periodo allineato)
        sim_prices = (1 + sim_returns_df).cumprod() * stocks_data.iloc[0]

        # 4. Simula equity usando STESSE selezioni (coerenti con l'intervallo)
        sim_equity = _simulate_rotational_equity(
            sim_prices,
            sel_tickers_df,
            rebal_dates,
            init_cash
        )

        sim_equity_curves[:, sim_idx] = sim_equity
        pbar.update(1)

    pbar.close()

    # Converti in DataFrame
    result_df = pd.DataFrame(
        sim_equity_curves,
        index=dates,
        columns=[f"sim_{i}" for i in range(n_simulations)]
    )

    return result_df
    


def _block_bootstrap_single_series(
    series: np.ndarray,
    block_size: int,
    preserve_mean: bool = False
) -> np.ndarray:
    """
    Block bootstrap di una singola serie temporale.
    
    Preserva autocorrelazione campionando blocchi contigui.
    """
    n = len(series)
    n_blocks_needed = int(np.ceil(n / block_size))
    
    bootstrapped = []
    
    for _ in range(n_blocks_needed):
        # Sample random starting point
        max_start = max(0, n - block_size)
        start_idx = np.random.randint(0, max_start + 1)
        
        # Extract block
        block = series[start_idx:start_idx + block_size]
        
        # Center block se preserve_mean
        if preserve_mean:
            block = block - block.mean() + series.mean()
        
        bootstrapped.extend(block)
    
    # Tronca alla lunghezza originale
    return np.array(bootstrapped[:n])


def _simulate_rotational_equity(
    prices: pd.DataFrame,
    selections: pd.DataFrame,
    rebal_dates: pd.DatetimeIndex,
    init_cash: float
) -> np.ndarray:
    """
    Simula equity curve dato prezzi e selezioni.
    
    Implementazione efficiente usando numpy per velocità.
    """
    equity = np.zeros(len(prices))
    equity[0] = init_cash
    
    # Current holdings: {ticker: n_shares}
    holdings = {}
    holdings_value = 0.0
    cash = init_cash
    
    # Pre-compute price ratios per velocità
    price_ratios = (prices / prices.shift(1).fillna(prices.iloc[0])).values
    
    for i in range(1, len(prices)):
        current_date = prices.index[i]
        prev_date = prices.index[i-1]
        
        # Check se oggi è rebalance
        is_rebalance = current_date in rebal_dates
        
        if is_rebalance:
            # Liquida holdings correnti
            if holdings:
                total_value = sum(
                    n_shares * prices.loc[prev_date, ticker]
                    for ticker, n_shares in holdings.items()
                    if ticker in prices.columns
                )
            else:
                total_value = cash
            
            # Nuova selezione
            try:
                selected_tickers = selections.loc[current_date, 'Top_Tickers']
                if isinstance(selected_tickers, str):
                    # Caso singolo ticker (non dovrebbe accadere)
                    selected_tickers = [selected_tickers]
                elif not isinstance(selected_tickers, list):
                    # Può essere una Series o altro
                    selected_tickers = list(selected_tickers)
            except:
                selected_tickers = []
            
            # Riallocazione equal-weight
            n_selected = len(selected_tickers)
            if n_selected > 0:
                target_per_ticker = total_value / n_selected
                holdings = {}
                
                for ticker in selected_tickers:
                    if ticker in prices.columns:
                        price_at_rebal = prices.loc[current_date, ticker]
                        if price_at_rebal > 0:
                            n_shares = target_per_ticker / price_at_rebal
                            holdings[ticker] = n_shares
                
                cash = 0.0
            else:
                # Nessuna selezione → 100% cash
                holdings = {}
                cash = total_value
        
        # Aggiorna valore holdings
        if holdings:
            portfolio_value = 0.0
            for ticker, n_shares in holdings.items():
                if ticker in prices.columns:
                    portfolio_value += n_shares * prices.iloc[i][ticker]
            equity[i] = portfolio_value + cash
        else:
            equity[i] = cash
    
    return equity


# =============================================================================
# ANALISI RISULTATI
# =============================================================================

def analyze_mc_results(
    mc_equity_curves: pd.DataFrame,
    portfolio_actual: vbt.Portfolio,
    confidence_level: float = 0.90,
    print_report: bool = True
) -> Dict:
    """
    Analizza distribuzione risultati Monte Carlo.
    
    Parametri
    ----------
    mc_equity_curves : pd.DataFrame
        Output di monte_carlo_block_bootstrap_rotational
    portfolio_actual : vbt.Portfolio
        Portfolio reale per confronto
    confidence_level : float, default=0.90
        Livello confidence interval (0.90 = 90%)
    print_report : bool, default=True
        Stampa report testuale
        
    Returns
    -------
    dict
        Dizionario con tutte le metriche calcolate
    """
    
    n_sims = mc_equity_curves.shape[1]
    init_value = mc_equity_curves.iloc[0, 0]
    
    # =========================================================================
    # 1. FINAL RETURNS
    # =========================================================================
    final_values = mc_equity_curves.iloc[-1]
    final_returns = (final_values / init_value) - 1
    
    actual_final_return = (portfolio_actual.value().iloc[-1] / portfolio_actual.init_cash) - 1
    
    alpha_lower = (1 - confidence_level) / 2
    alpha_upper = 1 - alpha_lower
    
    final_return_stats = {
        'actual': actual_final_return,
        'mc_mean': final_returns.mean(),
        'mc_median': final_returns.median(),
        'mc_std': final_returns.std(),
        'ci_lower': final_returns.quantile(alpha_lower),
        'ci_upper': final_returns.quantile(alpha_upper),
        'percentile_of_actual': (final_returns < actual_final_return).mean()
    }
    
    # =========================================================================
    # 2. CAGR
    # =========================================================================
    n_years = len(mc_equity_curves) / 252
    
    mc_cagrs = ((final_values / init_value) ** (1 / n_years)) - 1
    actual_cagr = ((portfolio_actual.value().iloc[-1] / portfolio_actual.init_cash) ** (1 / n_years)) - 1
    
    cagr_stats = {
        'actual': actual_cagr,
        'mc_mean': mc_cagrs.mean(),
        'mc_median': mc_cagrs.median(),
        'mc_std': mc_cagrs.std(),
        'ci_lower': mc_cagrs.quantile(alpha_lower),
        'ci_upper': mc_cagrs.quantile(alpha_upper),
        'percentile_of_actual': (mc_cagrs < actual_cagr).mean()
    }
    
    # =========================================================================
    # 3. MAX DRAWDOWN
    # =========================================================================
    def compute_max_dd(equity_series):
        cummax = equity_series.cummax()
        dd = (equity_series - cummax) / cummax
        return dd.min()
    
    mc_drawdowns = mc_equity_curves.apply(compute_max_dd, axis=0)
    actual_dd = compute_max_dd(portfolio_actual.value())
    
    dd_stats = {
        'actual': actual_dd,
        'mc_mean': mc_drawdowns.mean(),
        'mc_median': mc_drawdowns.median(),
        'mc_std': mc_drawdowns.std(),
        'ci_lower': mc_drawdowns.quantile(alpha_lower),  # less negative
        'ci_upper': mc_drawdowns.quantile(alpha_upper),  # more negative (worse)
        'worst_5pct': mc_drawdowns.quantile(0.95),  # 95th percentile = worst
        'percentile_of_actual': (mc_drawdowns < actual_dd).mean()  # <0, so < = worse
    }
    
    # =========================================================================
    # 4. SHARPE RATIO
    # =========================================================================
    mc_returns = mc_equity_curves.pct_change().fillna(0)
    mc_sharpes = (mc_returns.mean() / mc_returns.std()) * np.sqrt(252)
    
    actual_sharpe = portfolio_actual.sharpe_ratio()
    
    sharpe_stats = {
        'actual': actual_sharpe,
        'mc_mean': mc_sharpes.mean(),
        'mc_median': mc_sharpes.median(),
        'mc_std': mc_sharpes.std(),
        'ci_lower': mc_sharpes.quantile(alpha_lower),
        'ci_upper': mc_sharpes.quantile(alpha_upper),
        'percentile_of_actual': (mc_sharpes < actual_sharpe).mean()
    }
    
    # =========================================================================
    # 5. CALMAR RATIO
    # =========================================================================
    mc_calmars = mc_cagrs / mc_drawdowns.abs()
    mc_calmars = mc_calmars.replace([np.inf, -np.inf], np.nan).dropna()
    
    actual_calmar = portfolio_actual.calmar_ratio()
    
    calmar_stats = {
        'actual': actual_calmar,
        'mc_mean': mc_calmars.mean(),
        'mc_median': mc_calmars.median(),
        'mc_std': mc_calmars.std(),
        'ci_lower': mc_calmars.quantile(alpha_lower),
        'ci_upper': mc_calmars.quantile(alpha_upper),
        'percentile_of_actual': (mc_calmars < actual_calmar).mean() if len(mc_calmars) > 0 else np.nan
    }
    
    # =========================================================================
    # PRINT REPORT
    # =========================================================================
    
    if print_report:
        print("=" * 80)
        print(f"MONTE CARLO BLOCK BOOTSTRAP ANALYSIS ({n_sims:,} simulations)")
        print("=" * 80)
        print(f"Confidence Level: {confidence_level:.0%}")
        print(f"Period: {mc_equity_curves.index[0].date()} → {mc_equity_curves.index[-1].date()}")
        print(f"Duration: {n_years:.2f} years")
        print()
        
        print("─" * 80)
        print("FINAL RETURN")
        print("─" * 80)
        _print_stat("Actual", final_return_stats['actual'], is_pct=True)
        _print_stat("MC Mean", final_return_stats['mc_mean'], is_pct=True)
        _print_stat("MC Median", final_return_stats['mc_median'], is_pct=True)
        _print_stat("MC Std", final_return_stats['mc_std'], is_pct=True)
        print(f"  {confidence_level:.0%} CI: [{final_return_stats['ci_lower']:.1%}, {final_return_stats['ci_upper']:.1%}]")
        print(f"  Actual Percentile: {final_return_stats['percentile_of_actual']:.1%}")
        _print_flag(final_return_stats['percentile_of_actual'])
        print()
        
        print("─" * 80)
        print("CAGR (Compound Annual Growth Rate)")
        print("─" * 80)
        _print_stat("Actual", cagr_stats['actual'], is_pct=True)
        _print_stat("MC Mean", cagr_stats['mc_mean'], is_pct=True)
        _print_stat("MC Median", cagr_stats['mc_median'], is_pct=True)
        print(f"  {confidence_level:.0%} CI: [{cagr_stats['ci_lower']:.1%}, {cagr_stats['ci_upper']:.1%}]")
        print(f"  Actual Percentile: {cagr_stats['percentile_of_actual']:.1%}")
        _print_flag(cagr_stats['percentile_of_actual'])
        print()
        
        print("─" * 80)
        print("MAX DRAWDOWN")
        print("─" * 80)
        _print_stat("Actual", dd_stats['actual'], is_pct=True)
        _print_stat("MC Mean", dd_stats['mc_mean'], is_pct=True)
        _print_stat("MC Median", dd_stats['mc_median'], is_pct=True)
        print(f"  {confidence_level:.0%} CI: [{dd_stats['ci_lower']:.1%}, {dd_stats['ci_upper']:.1%}]")
        print(f"  Worst 5%: {dd_stats['worst_5pct']:.1%}")
        print(f"  Actual Percentile: {dd_stats['percentile_of_actual']:.1%} (lower = better)")
        _print_flag_dd(dd_stats['percentile_of_actual'])
        print()
        
        print("─" * 80)
        print("SHARPE RATIO")
        print("─" * 80)
        _print_stat("Actual", sharpe_stats['actual'], is_pct=False, decimals=3)
        _print_stat("MC Mean", sharpe_stats['mc_mean'], is_pct=False, decimals=3)
        _print_stat("MC Median", sharpe_stats['mc_median'], is_pct=False, decimals=3)
        _print_stat("MC Std", sharpe_stats['mc_std'], is_pct=False, decimals=3)
        print(f"  {confidence_level:.0%} CI: [{sharpe_stats['ci_lower']:.3f}, {sharpe_stats['ci_upper']:.3f}]")
        print(f"  Actual Percentile: {sharpe_stats['percentile_of_actual']:.1%}")
        _print_flag(sharpe_stats['percentile_of_actual'])
        print()
        
        print("─" * 80)
        print("CALMAR RATIO")
        print("─" * 80)
        _print_stat("Actual", calmar_stats['actual'], is_pct=False, decimals=3)
        _print_stat("MC Mean", calmar_stats['mc_mean'], is_pct=False, decimals=3)
        _print_stat("MC Median", calmar_stats['mc_median'], is_pct=False, decimals=3)
        if not np.isnan(calmar_stats['ci_lower']):
            print(f"  {confidence_level:.0%} CI: [{calmar_stats['ci_lower']:.3f}, {calmar_stats['ci_upper']:.3f}]")
        print()
        
        print("=" * 80)
        print("INTERPRETATION")
        print("=" * 80)
        _interpret_results(final_return_stats, cagr_stats, dd_stats, sharpe_stats)
        print("=" * 80)
    
    # Return completo
    return {
        'final_return': final_return_stats,
        'cagr': cagr_stats,
        'max_drawdown': dd_stats,
        'sharpe': sharpe_stats,
        'calmar': calmar_stats,
        'n_simulations': n_sims,
        'confidence_level': confidence_level
    }


def _print_stat(label, value, is_pct=True, decimals=1):
    """Helper per stampare statistiche formattate."""
    if is_pct:
        print(f"  {label:12s}: {value:+.{decimals}%}")
    else:
        print(f"  {label:12s}: {value:+.{decimals}f}")


def _print_flag(percentile):
    """Stampa flag interpretazione basato su percentile."""
    if percentile > 0.95:
        print("  🚩 WARNING: Actual > 95th percentile → Likely overfitting or luck")
    elif percentile > 0.75:
        print("  🟡 CAUTION: Actual > 75th percentile → Above average, monitor")
    elif percentile < 0.25:
        print("  ⚠️  CONCERN: Actual < 25th percentile → Below average")
    else:
        print("  ✅ NORMAL: Actual within expected range")


def _print_flag_dd(percentile):
    """Stampa flag per drawdown (logica inversa: basso = buono)."""
    if percentile < 0.05:
        print("  🟢 EXCELLENT: Actual DD better than 95% of simulations")
    elif percentile < 0.25:
        print("  ✅ GOOD: Actual DD better than average")
    elif percentile > 0.75:
        print("  ⚠️  CONCERN: Actual DD worse than 75% of simulations")
    else:
        print("  🟡 NORMAL: Actual DD within expected range")


def _interpret_results(final_ret, cagr, dd, sharpe):
    """Interpretazione automatica risultati."""
    
    # Check overfitting
    overfitting_score = 0
    if final_ret['percentile_of_actual'] > 0.95:
        overfitting_score += 2
    elif final_ret['percentile_of_actual'] > 0.85:
        overfitting_score += 1
    
    if sharpe['percentile_of_actual'] > 0.95:
        overfitting_score += 2
    elif sharpe['percentile_of_actual'] > 0.85:
        overfitting_score += 1
    
    # Check lucky DD
    lucky_dd = dd['percentile_of_actual'] < 0.15
    
    # Check robustezza
    robust = (
        0.30 < final_ret['percentile_of_actual'] < 0.70 and
        0.30 < sharpe['percentile_of_actual'] < 0.70
    )
    
    # Interpretazioni
    if overfitting_score >= 3:
        print("🚩 HIGH OVERFITTING RISK")
        print("   Your actual performance is in the extreme tail of MC distribution.")
        print("   This suggests parameter overfitting or exceptional luck.")
        print("   → Expect mean reversion in future OOS performance.")
        print()
    
    if lucky_dd and overfitting_score >= 2:
        print("🚩 LUCKY SCENARIO")
        print("   Both high returns AND low drawdown vs MC distribution.")
        print("   → Unusually favorable market conditions or overfitting.")
        print()
    
    if robust:
        print("✅ ROBUST PARAMETERS")
        print("   Actual performance near center of MC distribution.")
        print("   → Parameters appear stable and not overfit.")
        print()
    
    # Risk assessment
    worst_dd = dd['worst_5pct']
    if worst_dd < -0.40:
        print("⚠️  TAIL RISK ALERT")
        print(f"   Worst 5% scenarios show DD up to {worst_dd:.1%}")
        print("   → Consider position sizing or hedging strategies.")
        print()
    
    # Confidence assessment
    ci_width_ret = final_ret['ci_upper'] - final_ret['ci_lower']
    if ci_width_ret > 1.0:  # >100% range
        print("⚠️  HIGH UNCERTAINTY")
        print(f"   90% CI width: {ci_width_ret:.1%}")
        print("   → Highly variable outcomes, difficult to forecast.")
        print()


# =============================================================================
# VISUALIZZAZIONI
# =============================================================================

def plot_mc_distribution(
    mc_equity_curves: pd.DataFrame,
    portfolio_actual: vbt.Portfolio,
    save_path: Optional[str] = None,
    figsize: Tuple[int, int] = (16, 12)
):
    """
    Crea visualizzazione completa distribuzione Monte Carlo.
    
    4 plot:
    - Equity curves con percentili
    - Distribuzione final returns
    - Distribuzione max drawdown  
    - Distribuzione Sharpe
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    sns.set_style("whitegrid")
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    
    # =========================================================================
    # PLOT 1: Equity Curves con Percentili
    # =========================================================================
    ax = axes[0, 0]
    
    # Percentili MC
    pct_5 = mc_equity_curves.quantile(0.05, axis=1)
    pct_25 = mc_equity_curves.quantile(0.25, axis=1)
    pct_50 = mc_equity_curves.quantile(0.50, axis=1)
    pct_75 = mc_equity_curves.quantile(0.75, axis=1)
    pct_95 = mc_equity_curves.quantile(0.95, axis=1)
    
    # Actual
    actual_equity = portfolio_actual.value()
    
    # Plot
    ax.fill_between(mc_equity_curves.index, pct_5, pct_95, 
                     alpha=0.2, color='blue', label='5th-95th percentile')
    ax.fill_between(mc_equity_curves.index, pct_25, pct_75,
                     alpha=0.3, color='blue', label='25th-75th percentile')
    ax.plot(mc_equity_curves.index, pct_50, 'b-', linewidth=2, label='MC Median')
    ax.plot(actual_equity.index, actual_equity.values, 'r-', linewidth=2, label='Actual')
    
    ax.set_title('Monte Carlo Equity Curves Distribution', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Portfolio Value ($)')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    
    # =========================================================================
    # PLOT 2: Final Returns Distribution
    # =========================================================================
    ax = axes[0, 1]
    
    init_value = mc_equity_curves.iloc[0, 0]
    final_returns = (mc_equity_curves.iloc[-1] / init_value - 1) * 100
    actual_final_ret = (actual_equity.iloc[-1] / portfolio_actual.init_cash - 1) * 100
    
    ax.hist(final_returns, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
    ax.axvline(actual_final_ret, color='red', linestyle='--', linewidth=2, label=f'Actual: {actual_final_ret:.1f}%')
    ax.axvline(final_returns.median(), color='blue', linestyle='--', linewidth=2, label=f'MC Median: {final_returns.median():.1f}%')
    
    ax.set_title('Final Return Distribution', fontsize=14, fontweight='bold')
    ax.set_xlabel('Final Return (%)')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # =========================================================================
    # PLOT 3: Max Drawdown Distribution
    # =========================================================================
    ax = axes[1, 0]
    
    def compute_max_dd(equity_series):
        cummax = equity_series.cummax()
        dd = (equity_series - cummax) / cummax
        return dd.min() * 100
    
    mc_drawdowns = mc_equity_curves.apply(compute_max_dd, axis=0)
    actual_dd = compute_max_dd(actual_equity) 
    
    ax.hist(mc_drawdowns, bins=50, alpha=0.7, color='salmon', edgecolor='black')
    ax.axvline(actual_dd, color='red', linestyle='--', linewidth=2, label=f'Actual: {actual_dd:.1f}%')
    ax.axvline(mc_drawdowns.median(), color='blue', linestyle='--', linewidth=2, label=f'MC Median: {mc_drawdowns.median():.1f}%')
    
    ax.set_title('Max Drawdown Distribution', fontsize=14, fontweight='bold')
    ax.set_xlabel('Max Drawdown (%)')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # =========================================================================
    # PLOT 4: Sharpe Ratio Distribution
    # =========================================================================
    ax = axes[1, 1]
    
    mc_returns = mc_equity_curves.pct_change().fillna(0)
    mc_sharpes = (mc_returns.mean() / mc_returns.std()) * np.sqrt(252)
    actual_sharpe = portfolio_actual.sharpe_ratio()
    
    ax.hist(mc_sharpes, bins=50, alpha=0.7, color='lightgreen', edgecolor='black')
    ax.axvline(actual_sharpe, color='red', linestyle='--', linewidth=2, label=f'Actual: {actual_sharpe:.2f}')
    ax.axvline(mc_sharpes.median(), color='blue', linestyle='--', linewidth=2, label=f'MC Median: {mc_sharpes.median():.2f}')
    
    ax.set_title('Sharpe Ratio Distribution', fontsize=14, fontweight='bold')
    ax.set_xlabel('Sharpe Ratio')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Plot saved to: {save_path}")
    
    plt.show()
    
    return fig


# =============================================================================
# UTILITY: Export Results
# =============================================================================

def export_mc_results(
    mc_equity_curves: pd.DataFrame,
    analysis_dict: Dict,
    output_dir: str = "./mc_results"
):
    """
    Esporta risultati Monte Carlo in vari formati.
    
    Crea:
    - mc_equity_curves.csv (tutte le simulazioni)
    - mc_summary.csv (statistiche)
    - mc_report.txt (report testuale)
    """
    import os
    
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. Equity curves complete
    mc_equity_curves.to_csv(f"{output_dir}/mc_equity_curves.csv")
    print(f"✅ Equity curves saved: {output_dir}/mc_equity_curves.csv")
    
    # 2. Summary statistiche
    summary_data = []
    for metric_name, metric_stats in analysis_dict.items():
        if isinstance(metric_stats, dict) and 'actual' in metric_stats:
            summary_data.append({
                'Metric': metric_name,
                'Actual': metric_stats.get('actual'),
                'MC_Mean': metric_stats.get('mc_mean'),
                'MC_Median': metric_stats.get('mc_median'),
                'MC_Std': metric_stats.get('mc_std'),
                'CI_Lower': metric_stats.get('ci_lower'),
                'CI_Upper': metric_stats.get('ci_upper'),
                'Percentile': metric_stats.get('percentile_of_actual')
            })
    
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_csv(f"{output_dir}/mc_summary.csv", index=False)
    print(f"✅ Summary saved: {output_dir}/mc_summary.csv")
    
    print(f"\n📁 All results in: {output_dir}/")


# =============================================================================
# ESEMPIO USO
# =============================================================================

# # Esegui Monte Carlo
# mc_results = monte_carlo_block_bootstrap_rotational(
#     portfolio=pf_rot,
#     sel_tickers_df=sel_tickers,
#     stocks_data=stocks_data,
#     n_simulations=10_000,
#     block_size=20,
#     random_seed=42
# )

# # Analizza risultati
# analysis = analyze_mc_results(
#     mc_results,
#     pf_rot,
#     confidence_level=0.90,
#     print_report=True
# )

# # Visualizza
# fig = plot_mc_distribution(mc_results, pf_rot)

# # Esporta
# export_mc_results(mc_results, analysis, output_dir="./mc_results")

def monte_carlo_ranking_noise(
    stocks_data: pd.DataFrame,
    benchmark_data: Optional[pd.Series],
    params: Dict,
    n_simulations: int = 1_000,
    noise_std: float = 0.05,
    noise_type: str = "rank",
    init_cash: float = 100_000,
    random_seed: Optional[int] = None,
    show_progress: bool = True,
    build_portfolio_func: Optional[Callable] = None
) -> Dict:
    """
    Monte Carlo Ranking Noise Test per portfolios rotazionali.
    
    Aggiunge noise gaussiano al ranking, poi verifica:
    1. Quanto cambiano le selezioni (selection stability)
    2. Quanto cambia la performance (performance robustness)
    
    Parametri
    ----------
    stocks_data : pd.DataFrame
        Prezzi storici (stesso formato di build_rotational_portfolios_vbt)
    benchmark_data : pd.Series, optional
        Benchmark per confronto
    params : dict
        Parametri portfolio (da WFO o fissi)
        Es: {'rebalance_frequency': 'QE', 'n_top': 5, ...}
    n_simulations : int, default=1_000
        Numero simulazioni (1k tipicamente sufficiente, meno computation-heavy)
    noise_std : float, default=0.05
        Deviazione standard noise (0.05 = 5% rank shift)
        - 0.01-0.03: noise lieve (test sensibilità minima)
        - 0.05-0.10: noise moderato (raccomandato)
        - 0.10-0.20: noise forte (worst-case test)
    noise_type : str, default="rank"
        Tipo di noise:
        - "rank": noise sui rank percentili [0,1]
        - "score": noise sui combo score diretti
    init_cash : float, default=100_000
        Capitale iniziale
    random_seed : int, optional
        Seed per riproducibilità
    show_progress : bool, default=True
        Progress bar
    build_portfolio_func : Callable, optional
        Funzione custom per build portfolio.
        Se None, usa build_rotational_portfolios_vbt di default.
        
    Returns
    -------
    dict
        Risultati completi:
        - 'baseline': Portfolio baseline (no noise)
        - 'simulations': Lista portfolios con noise
        - 'selections_baseline': Selezioni baseline
        - 'selections_sims': Lista selezioni simulate
        - 'metrics': Dict metriche comparative
        - 'stability': Metriche stabilità selezioni
        
    Esempio
    -------
    >>> results = monte_carlo_ranking_noise(
    ...     stocks_data,
    ...     benchmark_data,
    ...     params={'rebalance_frequency': 'QE', 'n_top': 5, ...},
    ...     n_simulations=1_000,
    ...     noise_std=0.05
    ... )
    >>> analyze_ranking_noise_results(results)
    """
    
    if random_seed is not None:
        np.random.seed(random_seed)
    
    # Set build function (assume disponibile)
    if build_portfolio_func is None:
            build_portfolio_func = build_rotational_portfolios_vbt

    
    # =========================================================================
    # 1. BASELINE (no noise)
    # =========================================================================
    print("Building baseline portfolio (no noise)...")
    
    pf_baseline, pf_bench, sel_baseline, rankings_baseline = build_portfolio_func(
        stocks_data=stocks_data,
        benchmark_data=benchmark_data,
        portfolio_name="Baseline",
        init_cash=init_cash,
        plot=False,
        **params
    )
    
    baseline_sharpe = pf_baseline.sharpe_ratio()
    baseline_cagr = pf_baseline.annualized_return()
    
    print(f"Baseline Sharpe: {baseline_sharpe:.3f}")
    print(f"Baseline CAGR: {baseline_cagr:.2%}")
    print()
    
    # =========================================================================
    # 2. SIMULAZIONI CON NOISE
    # =========================================================================
    print(f"Running {n_simulations} simulations with noise_std={noise_std}...")
    
    sim_results = []
    sim_sharpes = []
    sim_cagrs = []
    sim_selections = []
    
    pbar = tqdm(total=n_simulations, desc="Noise Sims", disable=not show_progress)
    
    for sim_idx in range(n_simulations):
        # Crea versione con noise della funzione build
        pf_noisy, _, sel_noisy, _ = _build_portfolio_with_ranking_noise(
            build_func=build_portfolio_func,
            stocks_data=stocks_data,
            benchmark_data=benchmark_data,
            params=params,
            noise_std=noise_std,
            noise_type=noise_type,
            init_cash=init_cash,
            sim_seed=sim_idx if random_seed else None
        )
        
        try:
            sim_sharpe = pf_noisy.sharpe_ratio()
            sim_cagr = pf_noisy.annualized_return()
        except:
            sim_sharpe = np.nan
            sim_cagr = np.nan
        
        sim_results.append({
            'portfolio': pf_noisy,
            'selections': sel_noisy,
            'sharpe': sim_sharpe,
            'cagr': sim_cagr,
            'sim_idx': sim_idx
        })
        
        sim_sharpes.append(sim_sharpe)
        sim_cagrs.append(sim_cagr)
        sim_selections.append(sel_noisy)
        
        pbar.update(1)
    
    pbar.close()
    
    # =========================================================================
    # 3. ANALISI STABILITÀ SELEZIONI
    # =========================================================================
    print("\nAnalyzing selection stability...")
    
    stability_metrics = _compute_selection_stability(
        baseline_selections=sel_baseline,
        simulated_selections=sim_selections
    )
    
    # =========================================================================
    # 4. ANALISI PERFORMANCE ROBUSTNESS
    # =========================================================================
    print("Analyzing performance robustness...")
    
    sim_sharpes_clean = [s for s in sim_sharpes if np.isfinite(s)]
    sim_cagrs_clean = [c for c in sim_cagrs if np.isfinite(c)]
    
    performance_metrics = {
        'baseline_sharpe': baseline_sharpe,
        'baseline_cagr': baseline_cagr,
        'mc_sharpe_mean': np.mean(sim_sharpes_clean) if sim_sharpes_clean else np.nan,
        'mc_sharpe_std': np.std(sim_sharpes_clean) if sim_sharpes_clean else np.nan,
        'mc_sharpe_median': np.median(sim_sharpes_clean) if sim_sharpes_clean else np.nan,
        'mc_cagr_mean': np.mean(sim_cagrs_clean) if sim_cagrs_clean else np.nan,
        'mc_cagr_std': np.std(sim_cagrs_clean) if sim_cagrs_clean else np.nan,
        'sharpe_degradation': baseline_sharpe - np.mean(sim_sharpes_clean) if sim_sharpes_clean else np.nan,
        'sharpe_degradation_pct': ((baseline_sharpe - np.mean(sim_sharpes_clean)) / baseline_sharpe * 100) 
                                   if sim_sharpes_clean and baseline_sharpe != 0 else np.nan,
        'n_failed_sims': n_simulations - len(sim_sharpes_clean)
    }
    
    # =========================================================================
    # RETURN COMPLETO
    # =========================================================================
    return {
        'baseline': {
            'portfolio': pf_baseline,
            'benchmark': pf_bench,
            'selections': sel_baseline,
            'rankings': rankings_baseline,
            'sharpe': baseline_sharpe,
            'cagr': baseline_cagr
        },
        'simulations': sim_results,
        'selections_baseline': sel_baseline,
        'selections_sims': sim_selections,
        'metrics': performance_metrics,
        'stability': stability_metrics,
        'params': params,
        'noise_std': noise_std,
        'n_simulations': n_simulations
    }

def _build_portfolio_with_ranking_noise(
    build_func: Callable,
    stocks_data: pd.DataFrame,
    benchmark_data: Optional[pd.Series],
    params: Dict,
    noise_std: float,
    noise_type: str,
    init_cash: float,
    sim_seed: Optional[int] = None
):
    """
    Wrapper che aggiunge noise al ranking internamente.
    
    STRATEGIA: Monkey-patch temporaneo della funzione rank di pandas
    per aggiungere noise ai rank percentili.
    """
    
    # Salva funzione rank originale
    original_rank = pd.Series.rank
    
    # Crea versione con noise
    def rank_with_noise(self, *args, **kwargs):
        # Rank normale
        ranked = original_rank(self, *args, **kwargs)
        
        # Aggiungi noise gaussiano
        if sim_seed is not None:
            np.random.seed(sim_seed + hash(str(self.index[0])) % 10000)
        
        noise = np.random.normal(0, noise_std, size=len(ranked))
        ranked_noisy = ranked + noise * len(ranked)  # scale by n per mantenere range
        
        # Re-rank per garantire ordine valido
        ranked_noisy = pd.Series(ranked_noisy, index=ranked.index).rank(
            method='average', na_option='keep'
        )
        
        return ranked_noisy
    
    # Applica monkey patch
    pd.Series.rank = rank_with_noise
    
    try:
        # Build portfolio con rank modificato
        pf, pf_bench, sel, rankings = build_func(
            stocks_data=stocks_data,
            benchmark_data=benchmark_data,
            portfolio_name=f"Noise_{sim_seed}",
            init_cash=init_cash,
            plot=False,
            **params
        )
    finally:
        # SEMPRE ripristina funzione originale
        pd.Series.rank = original_rank
    
    return pf, pf_bench, sel, rankings

def _compute_selection_stability(
    baseline_selections: pd.DataFrame,
    simulated_selections: List[pd.DataFrame]
) -> Dict:
    """
    Calcola metriche di stabilità selezioni tra baseline e simulazioni.
    
    Metriche:
    - Avg overlap: % ticker comuni tra baseline e sim
    - Min overlap: worst-case scenario
    - Stability score: weighted metric
    - Churn rate: % ticker changed on average
    """
    
    if baseline_selections.empty or not simulated_selections:
        return {
            'avg_overlap': 0.0,
            'min_overlap': 0.0,
            'max_overlap': 0.0,
            'std_overlap': 0.0,
            'stability_score': 0.0,
            'avg_churn_rate': 1.0,
            'n_dates': 0
        }
    
    rebal_dates = baseline_selections.index
    
    overlaps = []
    churn_rates = []
    
    for date in rebal_dates:
        if date not in baseline_selections.index:
            continue
        
        baseline_set = set(baseline_selections.loc[date, 'tickers'])
        n_baseline = len(baseline_set)
        
        if n_baseline == 0:
            continue
        
        date_overlaps = []
        
        for sim_sel in simulated_selections:
            if date not in sim_sel.index:
                continue
            
            sim_set = set(sim_sel.loc[date, 'tickers'])
            
            # Overlap
            common = len(baseline_set & sim_set)
            overlap = common / n_baseline if n_baseline > 0 else 0.0
            date_overlaps.append(overlap)
            
            # Churn
            changed = len(baseline_set ^ sim_set)  # symmetric difference
            total = len(baseline_set | sim_set)
            churn = changed / total if total > 0 else 0.0
            churn_rates.append(churn)
        
        if date_overlaps:
            overlaps.extend(date_overlaps)
    
    if not overlaps:
        return {
            'avg_overlap': 0.0,
            'min_overlap': 0.0,
            'max_overlap': 0.0,
            'std_overlap': 0.0,
            'stability_score': 0.0,
            'avg_churn_rate': 1.0,
            'n_dates': 0
        }
    
    avg_overlap = np.mean(overlaps)
    min_overlap = np.min(overlaps)
    max_overlap = np.max(overlaps)
    std_overlap = np.std(overlaps)
    avg_churn = np.mean(churn_rates) if churn_rates else 1.0
    
    # Stability score: weighted combination
    # High overlap = good, Low std = good, Low churn = good
    stability_score = (
        0.5 * avg_overlap +
        0.3 * (1 - std_overlap) +
        0.2 * (1 - avg_churn)
    )
    
    return {
        'avg_overlap': avg_overlap,
        'min_overlap': min_overlap,
        'max_overlap': max_overlap,
        'std_overlap': std_overlap,
        'stability_score': stability_score,
        'avg_churn_rate': avg_churn,
        'n_dates': len(rebal_dates),
        'n_comparisons': len(overlaps)
    }

# =============================================================================
# ANALISI RISULTATI
# =============================================================================

def analyze_ranking_noise_results(
    results: Dict,
    print_report: bool = True
) -> Dict:
    """
    Analizza risultati Ranking Noise Test.
    
    Parametri
    ----------
    results : dict
        Output di monte_carlo_ranking_noise()
    print_report : bool
        Stampa report testuale
        
    Returns
    -------
    dict
        Analisi interpretata con raccomandazioni
    """
    
    baseline = results['baseline']
    metrics = results['metrics']
    stability = results['stability']
    noise_std = results['noise_std']
    n_sims = results['n_simulations']
    
    # =========================================================================
    # INTERPRETAZIONE
    # =========================================================================
    
    # Selection Stability
    if stability['avg_overlap'] > 0.80:
        stability_rating = "EXCELLENT"
        stability_color = "🟢"
        stability_msg = "Selezioni molto stabili, parametri robusti"
    elif stability['avg_overlap'] > 0.65:
        stability_rating = "GOOD"
        stability_color = "✅"
        stability_msg = "Selezioni ragionevolmente stabili"
    elif stability['avg_overlap'] > 0.50:
        stability_rating = "MODERATE"
        stability_color = "🟡"
        stability_msg = "Selezioni moderatamente stabili, attenzione"
    else:
        stability_rating = "FRAGILE"
        stability_color = "🔴"
        stability_msg = "Selezioni fragili, parametri sensibili a noise"
    
    # Performance Robustness
    sharpe_deg_pct = metrics['sharpe_degradation_pct']
    
    if np.isnan(sharpe_deg_pct):
        performance_rating = "UNKNOWN"
        performance_color = "⚪"
        performance_msg = "Impossibile calcolare degradation"
    elif sharpe_deg_pct < 5:
        performance_rating = "EXCELLENT"
        performance_color = "🟢"
        performance_msg = "Performance molto robusta al noise"
    elif sharpe_deg_pct < 15:
        performance_rating = "GOOD"
        performance_color = "✅"
        performance_msg = "Performance ragionevolmente robusta"
    elif sharpe_deg_pct < 30:
        performance_rating = "MODERATE"
        performance_color = "🟡"
        performance_msg = "Performance moderatamente sensibile"
    else:
        performance_rating = "WEAK"
        performance_color = "🔴"
        performance_msg = "Performance molto sensibile al noise"
    
    # Overall Assessment
    if stability_rating in ["EXCELLENT", "GOOD"] and performance_rating in ["EXCELLENT", "GOOD"]:
        overall_rating = "ROBUST"
        overall_color = "🟢"
        overall_msg = "Parametri robusti, deploy consigliato"
    elif "FRAGILE" in [stability_rating, performance_rating] or "WEAK" in [stability_rating, performance_rating]:
        overall_rating = "RISKY"
        overall_color = "🔴"
        overall_msg = "Parametri fragili, riconsiderare strategia"
    else:
        overall_rating = "ACCEPTABLE"
        overall_color = "🟡"
        overall_msg = "Parametri accettabili ma con cautela"
    
    interpretation = {
        'stability': {
            'rating': stability_rating,
            'color': stability_color,
            'message': stability_msg
        },
        'performance': {
            'rating': performance_rating,
            'color': performance_color,
            'message': performance_msg
        },
        'overall': {
            'rating': overall_rating,
            'color': overall_color,
            'message': overall_msg
        }
    }
    
    # =========================================================================
    # PRINT REPORT
    # =========================================================================
    
    if print_report:
        print("=" * 80)
        print(f"MONTE CARLO RANKING NOISE TEST ({n_sims:,} simulations)")
        print("=" * 80)
        print(f"Noise Level: {noise_std:.2%} rank std")
        print(f"Parameters: {results['params']}")
        print()
        
        print("─" * 80)
        print("BASELINE PERFORMANCE")
        print("─" * 80)
        print(f"  Sharpe Ratio: {baseline['sharpe']:.3f}")
        print(f"  CAGR:         {baseline['cagr']:.2%}")
        print()
        
        print("─" * 80)
        print("SELECTION STABILITY")
        print("─" * 80)
        print(f"  Avg Overlap:       {stability['avg_overlap']:.1%} {stability_color}")
        print(f"  Min Overlap:       {stability['min_overlap']:.1%}")
        print(f"  Max Overlap:       {stability['max_overlap']:.1%}")
        print(f"  Std Overlap:       {stability['std_overlap']:.3f}")
        print(f"  Avg Churn Rate:    {stability['avg_churn_rate']:.1%}")
        print(f"  Stability Score:   {stability['stability_score']:.3f}")
        print()
        print(f"  Rating: {stability_color} {stability_rating}")
        print(f"  → {stability_msg}")
        print()
        
        print("─" * 80)
        print("PERFORMANCE ROBUSTNESS")
        print("─" * 80)
        print(f"  Baseline Sharpe:   {metrics['baseline_sharpe']:.3f}")
        print(f"  MC Mean Sharpe:    {metrics['mc_sharpe_mean']:.3f}")
        print(f"  MC Std Sharpe:     {metrics['mc_sharpe_std']:.3f}")
        print(f"  Sharpe Degradation: {metrics['sharpe_degradation']:+.3f} ({sharpe_deg_pct:+.1f}%) {performance_color}")
        print()
        print(f"  Baseline CAGR:     {metrics['baseline_cagr']:.2%}")
        print(f"  MC Mean CAGR:      {metrics['mc_cagr_mean']:.2%}")
        print(f"  MC Std CAGR:       {metrics['mc_cagr_std']:.2%}")
        print()
        print(f"  Failed Sims:       {metrics['n_failed_sims']}/{n_sims}")
        print()
        print(f"  Rating: {performance_color} {performance_rating}")
        print(f"  → {performance_msg}")
        print()
        
        print("=" * 80)
        print("OVERALL ASSESSMENT")
        print("=" * 80)
        print(f"{overall_color} {overall_rating}: {overall_msg}")
        print()
        
        _print_recommendations(interpretation, stability, metrics)
        
        print("=" * 80)
    
    return {
        'interpretation': interpretation,
        'stability_metrics': stability,
        'performance_metrics': metrics
    }


def _print_recommendations(interpretation, stability, metrics):
    """Stampa raccomandazioni basate su risultati."""
    
    print("RECOMMENDATIONS")
    print("─" * 80)
    
    stability_rating = interpretation['stability']['rating']
    performance_rating = interpretation['performance']['rating']
    
    if stability_rating == "FRAGILE":
        print("🔴 CRITICAL - Selection Instability:")
        print("   → Increase lookback windows (es. 60→120 giorni)")
        print("   → Increase n_top (più ticker = meno sensibile a ranking noise)")
        print("   → Consider removing aggressive filters")
        print()
    
    if performance_rating == "WEAK":
        print("🔴 CRITICAL - Performance Degradation:")
        print(f"   → Sharpe drops by {metrics['sharpe_degradation_pct']:.1f}% with noise")
        print("   → Parameters likely overfit to specific rankings")
        print("   → Test with simpler strategy or longer lookbacks")
        print()
    
    if stability['avg_overlap'] < 0.60:
        print("⚠️  LOW OVERLAP WARNING:")
        print("   → Less than 60% selection overlap with noise")
        print("   → Consider momentum_weight closer to 1.0 (less risk-parity)")
        print("   → Test with acceleration=False")
        print()
    
    if stability['avg_churn_rate'] > 0.50:
        print("⚠️  HIGH CHURN WARNING:")
        print("   → More than 50% portfolio turnover with noise")
        print("   → Transaction costs will significantly impact live performance")
        print("   → Consider quarterly rebalance instead of monthly")
        print()
    
    if interpretation['overall']['rating'] == "ROBUST":
        print("✅ STRATEGY APPROVED:")
        print("   → Parameters appear robust to ranking perturbations")
        print("   → Safe to deploy with confidence")
        print("   → Continue monitoring with walk-forward OOS")
        print()
    elif interpretation['overall']['rating'] == "ACCEPTABLE":
        print("🟡 PROCEED WITH CAUTION:")
        print("   → Parameters acceptable but not ideal")
        print("   → Consider reducing position sizes by 20-30%")
        print("   → Monitor first 2-3 rebalances closely")
        print()


# =============================================================================
# VISUALIZZAZIONI
# =============================================================================

def plot_ranking_noise_analysis(
    results: Dict,
    save_path: Optional[str] = None,
    figsize: Tuple[int, int] = (16, 10)
):
    """
    Visualizzazione completa Ranking Noise Test.
    
    4 grafici:
    1. Selection Overlap Distribution
    2. Performance Distribution (Sharpe)
    3. Churn Rate over Time
    4. Equity Curve Comparison (Baseline vs MC mean/range)
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    sns.set_style("whitegrid")
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    
    baseline = results['baseline']
    sims = results['simulations']
    stability = results['stability']
    metrics = results['metrics']
    
    # =========================================================================
    # PLOT 1: Selection Overlap Distribution
    # =========================================================================
    ax = axes[0, 0]
    
    # Calcola overlap per ogni simulazione
    baseline_sel = results['selections_baseline']
    sim_selections = results['selections_sims']
    
    all_overlaps = []
    for date in baseline_sel.index:
        baseline_set = set(baseline_sel.loc[date, 'Top_Tickers'])
        n_baseline = len(baseline_set)
        
        if n_baseline == 0:
            continue
        
        for sim_sel in sim_selections:
            if date in sim_sel.index:
                sim_set = set(sim_sel.loc[date, 'tickers'])
                overlap = len(baseline_set & sim_set) / n_baseline
                all_overlaps.append(overlap * 100)
    
    if all_overlaps:
        ax.hist(all_overlaps, bins=30, alpha=0.7, color='skyblue', edgecolor='black')
        ax.axvline(stability['avg_overlap'] * 100, color='red', linestyle='--', 
                   linewidth=2, label=f"Mean: {stability['avg_overlap']:.1%}")
        ax.axvline(80, color='green', linestyle=':', linewidth=2, alpha=0.5, label='80% threshold')
    
    ax.set_title('Selection Overlap Distribution', fontsize=14, fontweight='bold')
    ax.set_xlabel('Overlap with Baseline (%)')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # =========================================================================
    # PLOT 2: Sharpe Distribution
    # =========================================================================
    ax = axes[0, 1]
    
    sim_sharpes = [s['sharpe'] for s in sims if np.isfinite(s['sharpe'])]
    
    if sim_sharpes:
        ax.hist(sim_sharpes, bins=30, alpha=0.7, color='lightgreen', edgecolor='black')
        ax.axvline(baseline['sharpe'], color='red', linestyle='--', linewidth=2,
                   label=f"Baseline: {baseline['sharpe']:.3f}")
        ax.axvline(np.mean(sim_sharpes), color='blue', linestyle='--', linewidth=2,
                   label=f"MC Mean: {np.mean(sim_sharpes):.3f}")
    
    ax.set_title('Sharpe Ratio Distribution (with Noise)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Sharpe Ratio')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # =========================================================================
    # PLOT 3: Churn Rate Over Time
    # =========================================================================
    ax = axes[1, 0]
    
    # Calcola churn per data
    churn_by_date = {}
    
    for date in baseline_sel.index:
        baseline_set = set(baseline_sel.loc[date, 'tickers'])
        
        date_churns = []
        for sim_sel in sim_selections:
            if date in sim_sel.index:
                sim_set = set(sim_sel.loc[date, 'Top_Tickers'])
                changed = len(baseline_set ^ sim_set)
                total = len(baseline_set | sim_set)
                churn = (changed / total * 100) if total > 0 else 0
                date_churns.append(churn)
        
        if date_churns:
            churn_by_date[date] = np.mean(date_churns)
    
    if churn_by_date:
        dates = list(churn_by_date.keys())
        churns = list(churn_by_date.values())
        
        ax.plot(dates, churns, 'o-', alpha=0.6, color='coral')
        ax.axhline(stability['avg_churn_rate'] * 100, color='red', linestyle='--',
                   label=f"Mean: {stability['avg_churn_rate']:.1%}")
        ax.axhline(50, color='orange', linestyle=':', alpha=0.5, label='50% threshold')
    
    ax.set_title('Churn Rate Over Time', fontsize=14, fontweight='bold')
    ax.set_xlabel('Rebalance Date')
    ax.set_ylabel('Churn Rate (%)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
    
    # =========================================================================
    # PLOT 4: Equity Curves (Baseline vs MC range)
    # =========================================================================
    ax = axes[1, 1]
    
    # Baseline equity
    baseline_equity = baseline['portfolio'].value()
    ax.plot(baseline_equity.index, baseline_equity.values, 'r-', linewidth=2,
            label='Baseline', zorder=10)
    
    # MC equity curves (sample 20 for visualization)
    n_plot = min(20, len(sims))
    for i in range(n_plot):
        sim_equity = sims[i]['portfolio'].value()
        ax.plot(sim_equity.index, sim_equity.values, 'gray', alpha=0.2, linewidth=1)
    
    # MC mean
    if sims:
        sim_equities = np.array([s['portfolio'].value().values for s in sims if hasattr(s['portfolio'], 'value')])
        if len(sim_equities) > 0:
            mc_mean = sim_equities.mean(axis=0)
            ax.plot(baseline_equity.index, mc_mean, 'b--', linewidth=2,
                    label='MC Mean', zorder=9)
    
    ax.set_title('Equity Curves: Baseline vs Noise Simulations', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Portfolio Value ($)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Plot saved: {save_path}")
    
    plt.show()
    
    return fig

# =============================================================================
# MC PER WFO: PER-WINDOW ANALYSIS
# =============================================================================

def monte_carlo_wfo_per_window(
    stocks_data: pd.DataFrame,
    benchmark_data: pd.Series,
    wfo_summary: pd.DataFrame,
    benchmark_title: str,
    n_simulations: int = 1_000,
    noise_std: float = 0.05,
    init_cash: float = 100_000,
    random_seed: Optional[int] = None,
    show_progress: bool = True
) -> pd.DataFrame:
    """
    Esegue MC Ranking Noise per OGNI finestra WFO separatamente.
    
    Testa robustezza parametri nella STESSA finestra OOS dove sono stati usati.
    
    Parametri
    ----------
    stocks_data : pd.DataFrame
        Prezzi storici completi
    benchmark_data : pd.Series
        Benchmark completo
    wfo_summary : pd.DataFrame
        Output di walk_forward_rotational() con colonne:
        - Index: Window string (es. "2020-01-01→2020-12-31 | 2021-01-01→2021-12-31")
        - Parametri: rebalance_frequency, n_top, momentum_lookback_days, etc.
        - TestStart, TestEnd: date OOS (se presenti, altrimenti parsate da Index)
    benchmark_title : str
        Nome benchmark (es. "^STOXX50E")
    n_simulations : int, default=1_000
        Numero simulazioni MC per finestra
    noise_std : float, default=0.05
        Deviazione standard noise (5% raccomandato)
    init_cash : float, default=100_000
        Capitale iniziale
    random_seed : int, optional
        Seed per riproducibilità
    show_progress : bool, default=True
        Progress bar
        
    Returns
    -------
    pd.DataFrame
        Una riga per finestra WFO con metriche:
        - Window, TestStart, TestEnd
        - Baseline_Sharpe, MC_Mean_Sharpe, Sharpe_Degradation_Pct
        - Avg_Overlap, Min_Overlap, Stability_Score
        - Overall_Rating
        - Parametri della finestra
        
    Esempio
    -------
    >>> wfo_summary = load_wfo_summary("Alpha Euro_2026.wfo_summary")
    >>> wfo_mc = monte_carlo_wfo_per_window(
    ...     stocks_data, benchmark_data, wfo_summary,
    ...     benchmark_title="^STOXX50E",
    ...     n_simulations=1_000
    ... )
    >>> print(wfo_mc[['Window', 'Avg_Overlap', 'Overall_Rating']])
    """
    wfo_summary = filter_testable_windows(
        wfo_summary,
        min_days_ago=30,
        verbose=True
    )

    if random_seed is not None:
        np.random.seed(random_seed)
    
    # # Import funzione (già disponibile nell'ambiente)
    # try:
    #     from r_functions import build_rotational_portfolios_from_wfo_result
    # except ImportError:
    #     raise ImportError(
    #         "build_rotational_portfolios_from_wfo_result non trovata. "
    #         "Assicurati di aver eseguito %run r_functions.ipynb"
    #     )
    
    results_per_window = []
    
    # =========================================================================
    # LOOP FINESTRE
    # =========================================================================
    
    pbar_windows = tqdm(
        total=len(wfo_summary),
        desc="MC per Finestra WFO",
        disable=not show_progress,
        position=0
    )
    
    for window_idx, (idx, row) in enumerate(wfo_summary.iterrows(), start=1):
        
        # ---------------------------------------------------------------------
        # 1. PARSE FINESTRA (formato: "TestStart→TestEnd")
        # ---------------------------------------------------------------------
        try:
            # Parse diretto da index: "2021-01-01→2021-12-31"
            parts = [x.strip() for x in str(idx).split("→", 1)]
            if len(parts) != 2:
                print(f"⚠️  Skipping window {idx}: invalid format (expected 'START→END')")
                pbar_windows.update(1)
                continue
            
            test_start = pd.Timestamp(parts[0])
            test_end = pd.Timestamp(parts[1])
        except Exception as e:
            print(f"⚠️  Skipping window {idx}: cannot parse dates ({e})")
            pbar_windows.update(1)
            continue
        
        window_str = f"{test_start.date()} → {test_end.date()}"
        pbar_windows.set_description(f"MC [{window_idx}/{len(wfo_summary)}] {window_str}")
        
        # ---------------------------------------------------------------------
        # 2. VERIFICA SE FINESTRA È TESTABILE
        # ---------------------------------------------------------------------
        
        today = pd.Timestamp.now().normalize()
        
        # CASO 1: Finestra completamente futura
        if test_start > today:
            if show_progress:
                print(f"⏭️  Skipping future window: {window_str}")
            pbar_windows.update(1)
            continue
        
        # CASO 2: Finestra parzialmente completata (test_end nel futuro o molto recente)
        # Per portafogli trimestrali, se siamo a Feb 2026 e test_end è Dic 2026,
        # non ci saranno rebalance dates ancora disponibili
        if test_end > today:
            if show_progress:
                print(f"⏭️  Skipping incomplete window (ends in future): {window_str}")
            pbar_windows.update(1)
            continue
        
        # CASO 3: Test_end è nel passato ma troppo recente (< 30 giorni)
        # Potrebbero mancare ancora dati completi
        days_since_end = (today - test_end).days
        if days_since_end < 30:
            if show_progress:
                print(f"⏭️  Skipping recent window (only {days_since_end} days old): {window_str}")
            pbar_windows.update(1)
            continue
        
        # CASO 4: Verifica dati disponibili
        available_data = stocks_data.loc[test_start:test_end]
        if available_data.empty:
            print(f"⚠️  Skipping {window_str}: no data available in period")
            pbar_windows.update(1)
            continue
        
        if len(available_data) < 10:  # almeno 10 giorni trading
            print(f"⚠️  Skipping {window_str}: insufficient data ({len(available_data)} days)")
            pbar_windows.update(1)
            continue
        
        # ---------------------------------------------------------------------
        # 3. BASELINE PORTFOLIO (da WFO result, no noise)
        # ---------------------------------------------------------------------
        
        # Limita summary_df a questa finestra
        summary_df_window = wfo_summary.loc[[idx]]
        
        try:
            pf_baseline, pf_bench, sel_baseline = build_rotational_portfolios_from_wfo_result(
                summary_df=summary_df_window,
                stocks_data=stocks_data,
                benchmark_data=benchmark_data,
                benchmark_title=benchmark_title,
                portfolio_name=f"Baseline_{window_str}",
                start_date=test_start,
                end_date=test_end,
                init_cash=init_cash,
                plot=False,
                debug=False,
                show_report=False
            )
        except Exception as e:
            error_msg = str(e).lower()
            
            # Errori comuni per finestre incomplete
            if any(keyword in error_msg for keyword in ['vuoto', 'empty', 'no selection', 'no data']):
                if show_progress:
                    print(f"⏭️  Skipping {window_str}: no valid selections (likely incomplete period)")
            else:
                print(f"⚠️  Error building baseline for {window_str}: {e}")
            
            pbar_windows.update(1)
            continue
        
        # Verifica esplicita che sel_baseline non sia vuoto
        if sel_baseline is None or sel_baseline.empty or len(sel_baseline) == 0:
            if show_progress:
                print(f"⏭️  Skipping {window_str}: no rebalance dates in period")
            pbar_windows.update(1)
            continue
        
        # Verifica che portfolio sia valido
        try:
            baseline_sharpe = pf_baseline.sharpe_ratio()
            baseline_cagr = pf_baseline.annualized_return()
            
            # Check se metriche sono valide
            if not np.isfinite(baseline_sharpe) or not np.isfinite(baseline_cagr):
                print(f"⚠️  Skipping {window_str}: invalid baseline metrics (Sharpe={baseline_sharpe}, CAGR={baseline_cagr})")
                pbar_windows.update(1)
                continue
                
        except Exception as e:
            print(f"⚠️  Skipping {window_str}: cannot compute baseline metrics ({e})")
            pbar_windows.update(1)
            continue
        
        # ---------------------------------------------------------------------
        # 3. MC SIMULAZIONI CON NOISE
        # ---------------------------------------------------------------------
        
        sim_sharpes = []
        sim_cagrs = []
        sim_selections = []
        
        pbar_sims = tqdm(
            total=n_simulations,
            desc=f"  Sims {window_str}",
            disable=not show_progress,
            position=1,
            leave=False
        )
        
        for sim_idx in range(n_simulations):
            try:
                # Build con noise usando monkey-patch interno
                # NOTA: build_rotational_portfolios_from_wfo_result chiama
                # build_rotational_portfolios_vbt internamente, quindi
                # il monkey-patch su pd.Series.rank funziona
                
                original_rank = pd.Series.rank
                
                def rank_with_noise(self, *args, **kwargs):
                    ranked = original_rank(self, *args, **kwargs)
                    if random_seed is not None:
                        np.random.seed(random_seed + sim_idx + hash(str(self.index[0])) % 10000)
                    noise = np.random.normal(0, noise_std, size=len(ranked))
                    ranked_noisy = ranked + noise * len(ranked)
                    ranked_noisy = pd.Series(ranked_noisy, index=ranked.index).rank(
                        method='average', na_option='keep'
                    )
                    return ranked_noisy
                
                pd.Series.rank = rank_with_noise
                
                pf_noisy, _, sel_noisy = build_rotational_portfolios_from_wfo_result(
                    summary_df=summary_df_window,
                    stocks_data=stocks_data,
                    benchmark_data=benchmark_data,
                    benchmark_title=benchmark_title,
                    portfolio_name=f"Noisy_{sim_idx}",
                    start_date=test_start,
                    end_date=test_end,
                    init_cash=init_cash,
                    plot=False,
                    debug=False,
                    show_report=False
                )
                
                pd.Series.rank = original_rank
                
                sim_sharpe = pf_noisy.sharpe_ratio()
                sim_cagr = pf_noisy.annualized_return()
                
                sim_sharpes.append(sim_sharpe)
                sim_cagrs.append(sim_cagr)
                sim_selections.append(sel_noisy)
                
            except Exception as e:
                # Ripristina sempre
                pd.Series.rank = original_rank
                # Continua con prossima sim
            finally:
                pbar_sims.update(1)
        
        pbar_sims.close()
        
        # ---------------------------------------------------------------------
        # 4. ANALISI STABILITÀ
        # ---------------------------------------------------------------------
        
        if not sim_selections:
            print(f"⚠️  No valid simulations for {window_str}")
            pbar_windows.update(1)
            continue
        
        stability = _compute_selection_stability(sel_baseline, sim_selections)
        
        # Performance metrics
        sim_sharpes_clean = [s for s in sim_sharpes if np.isfinite(s)]
        sim_cagrs_clean = [c for c in sim_cagrs if np.isfinite(c)]
        
        mc_sharpe_mean = np.mean(sim_sharpes_clean) if sim_sharpes_clean else np.nan
        mc_cagr_mean = np.mean(sim_cagrs_clean) if sim_cagrs_clean else np.nan
        
        sharpe_degradation = baseline_sharpe - mc_sharpe_mean
        sharpe_degradation_pct = (sharpe_degradation / baseline_sharpe * 100) if baseline_sharpe != 0 else np.nan
        
        # Rating
        if stability['avg_overlap'] > 0.80 and abs(sharpe_degradation_pct) < 5:
            rating = "ROBUST"
        elif stability['avg_overlap'] < 0.50 or abs(sharpe_degradation_pct) > 30:
            rating = "RISKY"
        else:
            rating = "ACCEPTABLE"
        
        # ---------------------------------------------------------------------
        # 5. STORE RESULTS
        # ---------------------------------------------------------------------
        
        # Estrai parametri della finestra
        param_cols = [
            'rebalance_frequency', 'momentum_lookback_days',
            'riskparity_lookback_days', 'n_top', 'use_acceleration',
            'momentum_weight', 'filter_ema', 'filter_volatility',
            'filter_min_momentum'
        ]
        
        params_dict = {col: row.get(col, np.nan) for col in param_cols}
        
        results_per_window.append({
            'Window': window_str,
            'TestStart': test_start,
            'TestEnd': test_end,
            'Baseline_Sharpe': baseline_sharpe,
            'Baseline_CAGR': baseline_cagr,
            'MC_Mean_Sharpe': mc_sharpe_mean,
            'MC_Mean_CAGR': mc_cagr_mean,
            'Sharpe_Degradation': sharpe_degradation,
            'Sharpe_Degradation_Pct': sharpe_degradation_pct,
            'Avg_Overlap': stability['avg_overlap'],
            'Min_Overlap': stability['min_overlap'],
            'Max_Overlap': stability['max_overlap'],
            'Stability_Score': stability['stability_score'],
            'Avg_Churn_Rate': stability['avg_churn_rate'],
            'Overall_Rating': rating,
            'N_Valid_Sims': len(sim_sharpes_clean),
            **params_dict
        })
        
        pbar_windows.update(1)
    
    pbar_windows.close()
    
    # =========================================================================
    # RETURN DATAFRAME
    # =========================================================================
    
    results_df = pd.DataFrame(results_per_window)
    
    return results_df


def filter_testable_windows(
    wfo_summary: pd.DataFrame,
    min_days_ago: int = 30,
    verbose: bool = True
) -> pd.DataFrame:
    """
    Filtra wfo_summary per includere solo finestre testabili con MC.
    
    Rimuove:
    - Finestre future (TestEnd > oggi)
    - Finestre incomplete (TestEnd < 30 giorni fa)
    
    Parametri
    ----------
    wfo_summary : pd.DataFrame
        WFO summary completo (index format: "TestStart→TestEnd")
    min_days_ago : int, default=30
        Minimo giorni tra TestEnd e oggi per considerare finestra testabile
    verbose : bool, default=True
        Stampa info su finestre rimosse
        
    Returns
    -------
    pd.DataFrame
        WFO summary filtrato con sole finestre testabili
        
    Esempio
    -------
    >>> wfo_full = load_wfo_summary("Alpha Euro_2026.wfo_summary")
    >>> wfo_testable = filter_testable_windows(wfo_full)
    >>> # wfo_testable esclude finestre 2026 se incomplete
    """
    
    today = pd.Timestamp.now().normalize()
    testable_indices = []
    skipped = []
    
    for idx in wfo_summary.index:
        try:
            # Parse date da index
            parts = [x.strip() for x in str(idx).split("→", 1)]
            if len(parts) != 2:
                skipped.append((idx, "invalid format"))
                continue
            
            test_start = pd.Timestamp(parts[0])
            test_end = pd.Timestamp(parts[1])
            
            # Check se testabile
            if test_start > today:
                skipped.append((idx, "future window"))
                continue
            
            if test_end > today:
                skipped.append((idx, "incomplete (ends in future)"))
                continue
            
            days_since_end = (today - test_end).days
            if days_since_end < min_days_ago:
                skipped.append((idx, f"too recent ({days_since_end} days ago)"))
                continue
            
            # Testabile
            testable_indices.append(idx)
            
        except Exception as e:
            skipped.append((idx, f"parse error: {e}"))
            continue
    
    # Filtra
    wfo_filtered = wfo_summary.loc[testable_indices]
    
    # Report
    if verbose:
        print(f"WFO Summary Filtering:")
        print(f"  Total windows: {len(wfo_summary)}")
        print(f"  Testable: {len(wfo_filtered)}")
        print(f"  Skipped: {len(skipped)}")
        
        if skipped:
            print(f"\nSkipped windows:")
            for window, reason in skipped:
                print(f"  - {window}: {reason}")
        print()
    
    return wfo_filtered


# =============================================================================
# ANALISI AGGREGATE
# =============================================================================

def analyze_wfo_mc_results(
    wfo_mc_df: pd.DataFrame,
    print_report: bool = True
) -> Dict:
    """
    Analizza risultati aggregati di MC su WFO multi-finestra.
    
    Parametri
    ----------
    wfo_mc_df : pd.DataFrame
        Output di monte_carlo_wfo_per_window()
    print_report : bool
        Stampa report
        
    Returns
    -------
    dict
        Analisi aggregate e raccomandazioni
    """
    
    n_windows = len(wfo_mc_df)
    
    # Aggregate statistics
    avg_overlap_all = wfo_mc_df['Avg_Overlap'].mean()
    min_overlap_all = wfo_mc_df['Avg_Overlap'].min()
    
    avg_degradation_all = wfo_mc_df['Sharpe_Degradation_Pct'].mean()
    max_degradation_all = wfo_mc_df['Sharpe_Degradation_Pct'].max()
    
    n_robust = (wfo_mc_df['Overall_Rating'] == 'ROBUST').sum()
    n_acceptable = (wfo_mc_df['Overall_Rating'] == 'ACCEPTABLE').sum()
    n_risky = (wfo_mc_df['Overall_Rating'] == 'RISKY').sum()
    
    # Overall assessment
    robust_pct = n_robust / n_windows
    
    if robust_pct >= 0.70:
        overall_rating = "ROBUST"
        overall_color = "🟢"
        overall_msg = "Maggioranza finestre robuste, strategia approvata"
    elif robust_pct >= 0.40:
        overall_rating = "ACCEPTABLE"
        overall_color = "🟡"
        overall_msg = "Mix di finestre robuste/fragili, cautela"
    else:
        overall_rating = "RISKY"
        overall_color = "🔴"
        overall_msg = "Maggioranza finestre fragili, rivedere strategia"
    
    # Identify worst window
    worst_idx = wfo_mc_df['Stability_Score'].idxmin()
    worst_window = wfo_mc_df.loc[worst_idx]
    
    # Identify best window
    best_idx = wfo_mc_df['Stability_Score'].idxmax()
    best_window = wfo_mc_df.loc[best_idx]
    
    analysis = {
        'n_windows': n_windows,
        'avg_overlap_all': avg_overlap_all,
        'min_overlap_all': min_overlap_all,
        'avg_degradation_all': avg_degradation_all,
        'max_degradation_all': max_degradation_all,
        'n_robust': n_robust,
        'n_acceptable': n_acceptable,
        'n_risky': n_risky,
        'robust_pct': robust_pct,
        'overall_rating': overall_rating,
        'overall_color': overall_color,
        'overall_msg': overall_msg,
        'worst_window': worst_window.to_dict(),
        'best_window': best_window.to_dict()
    }
    
    # =========================================================================
    # PRINT REPORT
    # =========================================================================
    
    if print_report:
        print("=" * 80)
        print("MONTE CARLO WFO - ANALISI AGGREGATE")
        print("=" * 80)
        print(f"Finestre analizzate: {n_windows}")
        print()
        
        print("─" * 80)
        print("STABILITÀ SELEZIONI (aggregate)")
        print("─" * 80)
        print(f"  Avg Overlap (tutte finestre):  {avg_overlap_all:.1%}")
        print(f"  Min Overlap (finestra peggiore): {min_overlap_all:.1%}")
        print()
        
        print("─" * 80)
        print("PERFORMANCE ROBUSTNESS (aggregate)")
        print("─" * 80)
        print(f"  Avg Sharpe Degradation:   {avg_degradation_all:+.1f}%")
        print(f"  Max Sharpe Degradation:   {max_degradation_all:+.1f}%")
        print()
        
        print("─" * 80)
        print("RATING PER FINESTRA")
        print("─" * 80)
        print(f"  ROBUST:     {n_robust}/{n_windows} ({robust_pct:.0%}) 🟢")
        print(f"  ACCEPTABLE: {n_acceptable}/{n_windows} ({n_acceptable/n_windows:.0%}) 🟡")
        print(f"  RISKY:      {n_risky}/{n_windows} ({n_risky/n_windows:.0%}) 🔴")
        print()
        
        print("=" * 80)
        print("OVERALL ASSESSMENT")
        print("=" * 80)
        print(f"{overall_color} {overall_rating}: {overall_msg}")
        print()
        
        print("─" * 80)
        print("FINESTRA PEGGIORE")
        print("─" * 80)
        print(f"  Window: {worst_window['Window']}")
        print(f"  Avg Overlap: {worst_window['Avg_Overlap']:.1%}")
        print(f"  Sharpe Degradation: {worst_window['Sharpe_Degradation_Pct']:+.1f}%")
        print(f"  Rating: {worst_window['Overall_Rating']}")
        print()
        
        print("─" * 80)
        print("FINESTRA MIGLIORE")
        print("─" * 80)
        print(f"  Window: {best_window['Window']}")
        print(f"  Avg Overlap: {best_window['Avg_Overlap']:.1%}")
        print(f"  Sharpe Degradation: {best_window['Sharpe_Degradation_Pct']:+.1f}%")
        print(f"  Rating: {best_window['Overall_Rating']}")
        print()
        
        print("=" * 80)
        print("RACCOMANDAZIONI")
        print("=" * 80)
        
        if overall_rating == "ROBUST":
            print("✅ STRATEGIA APPROVATA:")
            print("   → Maggioranza finestre mostra parametri robusti")
            print("   → Safe to deploy con confidence")
            print()
        elif overall_rating == "RISKY":
            print("🔴 STRATEGIA A RISCHIO:")
            print("   → Maggioranza finestre fragili")
            print("   → Considera:")
            print("     • Lookback più lunghi (60→120 giorni)")
            print("     • n_top più alto (3→8)")
            print("     • Rimuovere filtri aggressivi")
            print()
        else:
            print("🟡 STRATEGIA ACCETTABILE CON CAUTELA:")
            print("   → Mix di finestre robuste e fragili")
            print("   → Suggerimenti:")
            print("     • Deploy con position sizing ridotto (50-70%)")
            print("     • Monitoring intensivo prime rebalances")
            print("     • Analizza pattern finestre fragili (regime-specific?)")
            print()
        
        # Analisi temporale
        if 'TestStart' in wfo_mc_df.columns:
            wfo_mc_df_sorted = wfo_mc_df.sort_values('TestStart')
            recent_windows = wfo_mc_df_sorted.tail(3)
            
            recent_robust = (recent_windows['Overall_Rating'] == 'ROBUST').sum()
            
            print("─" * 80)
            print("TREND TEMPORALE (ultime 3 finestre)")
            print("─" * 80)
            
            for _, row in recent_windows.iterrows():
                status = "✅" if row['Overall_Rating'] == "ROBUST" else "🟡" if row['Overall_Rating'] == "ACCEPTABLE" else "🔴"
                print(f"  {status} {row['Window']}: Overlap={row['Avg_Overlap']:.1%}, Rating={row['Overall_Rating']}")
            
            if recent_robust >= 2:
                print("\n  → Trend positivo: finestre recenti robuste")
            elif recent_robust == 0:
                print("\n  ⚠️  Trend negativo: nessuna finestra recente robusta")
            
            print()
        
        print("=" * 80)
    
    return analysis

# =============================================================================
# VISUALIZZAZIONI
# =============================================================================

def plot_wfo_mc_results(
    wfo_mc_df: pd.DataFrame,
    save_path: Optional[str] = None,
    figsize=(16, 10)
):
    """
    Visualizza risultati MC per WFO multi-finestra.
    
    4 grafici:
    1. Overlap per finestra (timeline)
    2. Sharpe Degradation per finestra
    3. Rating distribution
    4. Stability Score trend
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    sns.set_style("whitegrid")
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    
    # Sort by TestStart
    if 'TestStart' in wfo_mc_df.columns:
        wfo_mc_df = wfo_mc_df.sort_values('TestStart')
    
    windows = wfo_mc_df['Window'].values
    x_pos = np.arange(len(windows))
    
    # =========================================================================
    # PLOT 1: Avg Overlap per Finestra
    # =========================================================================
    ax = axes[0, 0]
    
    colors = [
        'green' if r == 'ROBUST' else 'orange' if r == 'ACCEPTABLE' else 'red'
        for r in wfo_mc_df['Overall_Rating']
    ]
    
    ax.bar(x_pos, wfo_mc_df['Avg_Overlap'] * 100, color=colors, alpha=0.7, edgecolor='black')
    ax.axhline(80, color='green', linestyle='--', alpha=0.5, label='80% target')
    ax.axhline(60, color='orange', linestyle='--', alpha=0.5, label='60% min')
    
    ax.set_title('Selection Overlap per Finestra WFO', fontsize=14, fontweight='bold')
    ax.set_xlabel('Finestra')
    ax.set_ylabel('Avg Overlap (%)')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(windows, rotation=45, ha='right')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # =========================================================================
    # PLOT 2: Sharpe Degradation per Finestra
    # =========================================================================
    ax = axes[0, 1]
    
    ax.bar(x_pos, wfo_mc_df['Sharpe_Degradation_Pct'], color=colors, alpha=0.7, edgecolor='black')
    ax.axhline(0, color='black', linestyle='-', linewidth=1)
    ax.axhline(15, color='orange', linestyle='--', alpha=0.5, label='15% threshold')
    ax.axhline(30, color='red', linestyle='--', alpha=0.5, label='30% critical')
    
    ax.set_title('Sharpe Degradation per Finestra', fontsize=14, fontweight='bold')
    ax.set_xlabel('Finestra')
    ax.set_ylabel('Degradation (%)')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(windows, rotation=45, ha='right')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # =========================================================================
    # PLOT 3: Rating Distribution
    # =========================================================================
    ax = axes[1, 0]
    
    rating_counts = wfo_mc_df['Overall_Rating'].value_counts()
    colors_pie = []
    for rating in rating_counts.index:
        if rating == 'ROBUST':
            colors_pie.append('green')
        elif rating == 'ACCEPTABLE':
            colors_pie.append('orange')
        else:
            colors_pie.append('red')
    
    ax.pie(rating_counts.values, labels=rating_counts.index, autopct='%1.0f%%',
           colors=colors_pie, startangle=90)
    ax.set_title('Rating Distribution', fontsize=14, fontweight='bold')
    
    # =========================================================================
    # PLOT 4: Stability Score Trend
    # =========================================================================
    ax = axes[1, 1]
    
    ax.plot(x_pos, wfo_mc_df['Stability_Score'], 'o-', linewidth=2, markersize=8)
    ax.axhline(0.75, color='green', linestyle='--', alpha=0.5, label='Good (0.75)')
    ax.axhline(0.60, color='orange', linestyle='--', alpha=0.5, label='Acceptable (0.60)')
    
    ax.set_title('Stability Score Trend', fontsize=14, fontweight='bold')
    ax.set_xlabel('Finestra')
    ax.set_ylabel('Composite Stability Score')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(windows, rotation=45, ha='right')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Plot saved: {save_path}")
    
    plt.show()
    
    return fig



In [ ]:
# Engine Sanity Check

def build_engine_health_check(
    pf,
    sel_tickers: "pd.DataFrame",
    *,
    prices: "pd.DataFrame | None" = None,
    start_date: "pd.Timestamp | str | None" = None,
    end_date: "pd.Timestamp | str | None" = None,
    include_prev: bool = True,
    annual_trading_days: int = 252,
    # --- warning thresholds (tuning) ---
    warn_pct_never_selected: float = 0.40,
    warn_top1_sel_share: float = 0.35,
    warn_avg_churn: float = 0.70,
    warn_concentration_hhi: float = 0.18,
    return_details: bool = True,
):
    """
    Health-check del motore rotazionale (per sviluppatori).

    Input richiesti:
    - pf: vbt.Portfolio (rotational)
    - sel_tickers: DataFrame con indice datetime-like e colonna 'tickers' (liste o stringhe)

    Input opzionali (consigliati):
    - prices: DataFrame prezzi (Date x Ticker) per:
        1) definire l'universo reale (tickers disponibili)
        2) calcolare "held return" per ticker (proxy: compounding sui giorni in cui il ticker risulta selezionato)
    - start_date/end_date: finestra analisi; se None usa range di pf (se disponibile) o sel_tickers
    - include_prev: include ultima selezione prima di start_date come “set iniziale”

    Output:
    - health_df: tabella sintetica health-check (metriche + warning + sintesi finale)
    - ticker_df: diagnostica per-ticker (selezioni, share, held-return, ecc.)
    - selection_log: DataFrame long con colonne [Date, Ticker] di tutte le selezioni
    - details: dict con oggetti intermedi (se return_details=True)
    """
    import numpy as np
    import pandas as pd

    # --- helpers ---
    def _to_ts(x):
        if x is None:
            return None
        ts = pd.to_datetime(x)
        try:
            if getattr(ts, "tzinfo", None) is not None:
                ts = ts.tz_localize(None)
        except Exception:
            pass
        return pd.Timestamp(ts).normalize()

    def _norm_index(idx):
        idx = pd.to_datetime(idx)
        try:
            if getattr(idx, "tz", None) is not None:
                idx = idx.tz_localize(None)
        except Exception:
            pass
        return pd.DatetimeIndex(idx).normalize()

    def _ensure_list(x):
        if x is None:
            return []
        if isinstance(x, float) and pd.isna(x):
            return []
        if isinstance(x, (list, tuple, set)):
            return [str(t).strip() for t in x if str(t).strip()]
        if isinstance(x, str):
            s = x.strip()
            if not s:
                return []
            if "," in s:
                return [p.strip() for p in s.split(",") if p.strip()]
            return [s]
        return [str(x).strip()]

    # --- validate sel_tickers ---
    if sel_tickers is None or getattr(sel_tickers, "empty", True):
        raise ValueError("sel_tickers è vuoto.")
    if "tickers" not in sel_tickers.columns:
        raise KeyError("sel_tickers deve contenere la colonna 'tickers'.")

    df = sel_tickers.copy()
    df.index = _norm_index(df.index)
    df = df.sort_index()

    # --- resolve analysis window (pf preferred) ---
    pf_start = None
    pf_end = None
    try:
        st = pf.stats()
        if isinstance(st, dict) or hasattr(st, "__getitem__"):
            pf_start = _to_ts(pd.to_datetime(st["Start"]))
            pf_end = _to_ts(pd.to_datetime(st["End"]))
    except Exception:
        pass

    s = _to_ts(start_date) or pf_start or df.index.min()
    e = _to_ts(end_date) or pf_end or df.index.max()

    if s is None or pd.isna(s):
        s = df.index.min()
    if e is None or pd.isna(e):
        e = df.index.max()
    if s > e:
        s, e = e, s

    # --- filter window + include prev selection ---
    df_win = df.loc[s:e].copy()

    if include_prev and s is not None:
        df_prev = df.loc[:s]
        if not df_prev.empty:
            if df_prev.index.max() == s and len(df_prev) >= 2:
                df_prev_strict = df_prev.iloc[:-1]
            elif df_prev.index.max() < s:
                df_prev_strict = df_prev
            else:
                df_prev_strict = df_prev

            if not df_prev_strict.empty:
                prev_row = df_prev_strict.iloc[[-1]]
                df_win = pd.concat([prev_row, df_win], axis=0)
                df_win = df_win[~df_win.index.duplicated(keep="last")].sort_index()

    if df_win.empty:
        raise ValueError(f"Nessuna selezione nella finestra {s} → {e}.")

    # --- build selection_log (long): Date x Ticker ---
    rows = []
    for dt, row in df_win.iterrows():
        tick_list = _ensure_list(row["tickers"])
        for t in tick_list:
            rows.append((dt, t))

    selection_log = pd.DataFrame(rows, columns=["Date", "Ticker"])
    selection_log["Date"] = _norm_index(selection_log["Date"])
    selection_log = selection_log.sort_values(["Date", "Ticker"]).reset_index(drop=True)

    # --- counts & shares ---
    counts = selection_log["Ticker"].value_counts()
    total_picks = int(len(selection_log))
    n_dates = int(df_win.shape[0])

    ticker_df = pd.DataFrame({
        "Ticker": counts.index.astype(str),
        "Selections": counts.values,
    })
    ticker_df["Selection_Share"] = ticker_df["Selections"] / total_picks if total_picks else np.nan

    # --- per-date churn metrics ---
    sets_by_date = (
        df_win["tickers"]
        .apply(_ensure_list)
        .apply(lambda lst: tuple(sorted(set(lst))))
    )
    churns = []
    prev = None
    for _, cur_tuple in sets_by_date.items():
        cur = set(cur_tuple)
        if prev is None:
            prev = cur
            continue
        inter = len(prev.intersection(cur))
        union = len(prev.union(cur))
        jacc = (inter / union) if union else 1.0
        churns.append(1.0 - jacc)
        prev = cur
    avg_churn = float(np.nanmean(churns)) if len(churns) else np.nan

    # --- concentration (HHI over selection shares) ---
    shares = ticker_df["Selection_Share"].to_numpy(dtype=float) if not ticker_df.empty else np.array([])
    hhi = float(np.nansum(shares**2)) if shares.size else np.nan

    # --- never selected (universe from prices if provided) ---
    never_selected = []
    pct_never_selected = np.nan
    prices_universe = None
    prices_ok = bool(prices is not None and hasattr(prices, "columns") and len(getattr(prices, "columns", [])) > 0)

    if prices_ok:
        prices_universe = [str(c) for c in prices.columns]
        prices_universe_set = set(prices_universe)
        never_selected = sorted(prices_universe_set - set(counts.index.astype(str)))
        pct_never_selected = (
            len(never_selected) / len(prices_universe_set)
            if len(prices_universe_set) else np.nan
        )
        universe_note = f"Universe da prices: {len(prices_universe)} ticker. Never-selected: {len(never_selected)}."
    else:
        # fallback: within observed selections, "never selected" è 0% by construction
        pct_never_selected = 0.0
        universe_note = "Universe non disponibile (prices assente). % never-selected = 0% by construction."

    # --- top1 selection share ---
    top1_share = float(ticker_df["Selection_Share"].iloc[0]) if not ticker_df.empty else np.nan
    top1_ticker = str(ticker_df["Ticker"].iloc[0]) if not ticker_df.empty else None

    # --- held-return per ticker (optional, requires prices) ---
    held_ret_by_ticker = {}
    held_days_by_ticker = {}
    if prices_ok and hasattr(prices, "index"):
        px = prices.copy()
        px.index = _norm_index(px.index)
        px = px.sort_index()

        # restrict to analysis window
        px = px.loc[s:e]

        # daily returns for all tickers (NO pad fill)
        r_px = px.pct_change(fill_method=None)

        # selection sets per selection date -> daily with ffill
        sel_sets = pd.Series(
            index=df_win.index,
            data=df_win["tickers"].apply(_ensure_list).apply(lambda lst: tuple(sorted(set(lst))))
        ).sort_index()

        sel_sets_daily = sel_sets.reindex(px.index, method="ffill")
        valid_mask = sel_sets_daily.notna()
        sel_sets_daily = sel_sets_daily.loc[valid_mask.index[valid_mask]]
        r_px = r_px.loc[sel_sets_daily.index]

        # compute per ticker held returns
        for t in px.columns.astype(str):
            held = sel_sets_daily.apply(lambda tup: t in set(tup) if isinstance(tup, tuple) else False).astype(bool)
            rt = r_px[t].loc[held.index]
            rt_held = rt[held].dropna()
            held_days = int(rt_held.shape[0])
            held_ret = float((1.0 + rt_held).prod() - 1.0) if held_days > 0 else np.nan
            held_ret_by_ticker[t] = held_ret
            held_days_by_ticker[t] = held_days

        ticker_df["Held_Days"] = ticker_df["Ticker"].map(held_days_by_ticker).astype(float)
        ticker_df["Held_Return"] = ticker_df["Ticker"].map(held_ret_by_ticker).astype(float)
    else:
        ticker_df["Held_Days"] = np.nan
        ticker_df["Held_Return"] = np.nan

    # ------------------------------------------------------------
    # Storico: valutazione adeguatezza (NUOVO)
    # ------------------------------------------------------------
    history_days = int((e - s).days)
    history_years = history_days / 365.25

    if history_years >= 2 and n_dates >= 24:
        history_status = "ADEGUATO"
        history_note = "Storico sufficiente per valutazione completa del motore."
    elif history_years >= 1 and n_dates >= 12:
        history_status = "PARZIALE"
        history_note = (
            "Storico sufficiente solo per analisi preliminari. "
            "Metriche come churn/copertura vanno interpretate con cautela."
        )
    else:
        history_status = "INSUFFICIENTE"
        history_note = (
            "Storico troppo corto per una valutazione affidabile del motore. "
            "Consigliato ripetere con uno storico più ampio."
        )

    # --- build health table ---
    def _flag(cond: bool) -> str:
        return "⚠️" if cond else "OK"

    meta_rows = [
        {
            "Check": "Adeguatezza storico",
            "Value": f"{history_years:.2f} anni / {n_dates} selezioni",
            "Threshold": "≥ 2 anni & ≥ 24 selezioni",
            "Status": history_status,
            "Note": history_note,
        },
        {"Check": "Periodo analisi", "Value": f"{s.date()} → {e.date()}", "Threshold": "", "Status": "", "Note": ""},
        {"Check": "N. date selezione", "Value": n_dates, "Threshold": "", "Status": "", "Note": ""},
        {"Check": "N. selezioni totali (date×top)", "Value": total_picks, "Threshold": "", "Status": "", "Note": ""},
        {"Check": "N. ticker selezionati almeno 1 volta", "Value": int(counts.shape[0]), "Threshold": "", "Status": "", "Note": ""},
        {"Check": "prices_ok", "Value": prices_ok, "Threshold": "", "Status": "", "Note": ""},
        {"Check": "prices_shape", "Value": (getattr(prices, "shape", None) if prices_ok else None), "Threshold": "", "Status": "", "Note": ""},
        {"Check": "Universe (da prices) size", "Value": (len(prices_universe) if prices_universe is not None else np.nan), "Threshold": "", "Status": "", "Note": universe_note},
    ]

    warn_rows = [
        {
            "Check": "% titoli mai selezionati (su universe)",
            "Value": pct_never_selected,
            "Threshold": f">{warn_pct_never_selected:.0%}",
            "Status": _flag(pd.notna(pct_never_selected) and pct_never_selected > warn_pct_never_selected),
            "Note": universe_note,
        },
        {
            "Check": "Concentrazione Top-1 (share selezioni)",
            "Value": top1_share,
            "Threshold": f">{warn_top1_sel_share:.0%}",
            "Status": _flag(pd.notna(top1_share) and top1_share > warn_top1_sel_share),
            "Note": f"Top1: {top1_ticker}" if top1_ticker else "",
        },
        {
            "Check": "Churn medio (1 - Jaccard)",
            "Value": avg_churn,
            "Threshold": f">{warn_avg_churn:.2f}",
            "Status": _flag(pd.notna(avg_churn) and avg_churn > warn_avg_churn),
            "Note": "Se troppo alto: turnover eccessivo/instabile.",
        },
        {
            "Check": "Concentrazione HHI (share^2)",
            "Value": hhi,
            "Threshold": f">{warn_concentration_hhi:.2f}",
            "Status": _flag(pd.notna(hhi) and hhi > warn_concentration_hhi),
            "Note": "Più alto = selezioni concentrate su pochi titoli.",
        },
    ]

    health_df = pd.concat([pd.DataFrame(meta_rows), pd.DataFrame(warn_rows)], ignore_index=True)

    # ------------------------------------------------------------
    # Sintesi finale stato di salute (NUOVO)
    # ------------------------------------------------------------
    n_warn = int((health_df["Status"] == "⚠️").sum())

    if history_status == "INSUFFICIENTE":
        engine_status = "🔴 CRITICO"
        engine_note = (
            "Valutazione non affidabile: storico insufficiente. "
            "Estendere lo storico prima di trarre conclusioni."
        )
    elif history_status == "PARZIALE":
        engine_status = "🟡 DA MONITORARE"
        engine_note = (
            "Motore funzionante ma valutazione parziale. "
            "Ripetere l’analisi su uno storico più esteso."
        )
    else:
        if n_warn >= 3:
            engine_status = "🟡 DA MONITORARE"
            engine_note = (
                "Motore attivo ma con segnali di instabilità strutturale "
                "(churn/concentrazione). Valutare revisione griglia WFO."
            )
        else:
            engine_status = "🟢 SANO"
            engine_note = (
                "Motore coerente: buona copertura universo e nessun segnale "
                "di concentrazione eccessiva o instabilità marcata."
            )

    summary_row = {
        "Check": "Stato di salute del motore",
        "Value": engine_status,
        "Threshold": "",
        "Status": "",
        "Note": engine_note,
    }
    health_df = pd.concat([health_df, pd.DataFrame([summary_row])], ignore_index=True)

    # --- details ---
    details = None
    if return_details:
        details = {
            "analysis_start": s,
            "analysis_end": e,
            "history_days": history_days,
            "history_years": history_years,
            "history_status": history_status,
            "sets_by_date": sets_by_date,
            "churn_series": pd.Series(churns, name="churn") if len(churns) else pd.Series(dtype=float, name="churn"),
            "never_selected": never_selected,
            "prices_universe": prices_universe,
            "universe_note": universe_note,
        }

    # --- sort ticker_df ---
    ticker_df = ticker_df.sort_values(["Selections", "Selection_Share"], ascending=[False, False]).reset_index(drop=True)

    return health_df, ticker_df, selection_log, details


### WFO CLustered

In [ ]:
# ============================================================
# CELL 0 — Import
# ============================================================
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from typing import Optional

# ============================================================
# CELL 1 — Funzioni (incolla tutto, non modificare)
# ============================================================

def filter_stocks_data(stocks_data: pd.DataFrame, tickers: list) -> pd.DataFrame:
    available = [t for t in tickers if t in stocks_data.columns]
    if len(available) < len(tickers):
        missing = set(tickers) - set(available)
        print(f"⚠️  Ticker mancanti: {missing}")
    return stocks_data[available]


def plot_dendrogram_colored(Z, labels, n_clusters, ax, palette=None):
    if palette is None:
        palette = ['#1D9E75', '#378ADD', '#BA7517', '#9F77DD', '#D85A30']

    cluster_ids     = fcluster(Z, t=n_clusters, criterion='maxclust')
    label_to_cluster= dict(zip(labels, cluster_ids))
    cluster_colors  = {cid: palette[i % len(palette)]
                       for i, cid in enumerate(sorted(set(cluster_ids)))}

    n_leaves = len(labels)

    def get_leaves(node_idx):
        if node_idx < n_leaves:
            return {node_idx}
        left  = int(Z[node_idx - n_leaves, 0])
        right = int(Z[node_idx - n_leaves, 1])
        return get_leaves(left) | get_leaves(right)

    def link_color_func(node_idx):
        leaves   = get_leaves(node_idx)
        clusters = {cluster_ids[i] for i in leaves}
        if len(clusters) == 1:
            return cluster_colors[next(iter(clusters))]
        return '#AAAAAA'

    dendrogram(
        Z,
        labels          = labels,
        leaf_rotation   = 90,
        link_color_func = link_color_func,
        ax              = ax,
    )

    # Linea di taglio — UNA SOLA, calcolata come media tra ultima fusione
    # intra-cluster e prima fusione inter-cluster
    cut_height = (Z[-(n_clusters-1), 2] + Z[-n_clusters, 2]) / 2
    ax.axhline(y=cut_height, color='red', linestyle='--',
               alpha=0.7, label=f'{n_clusters} cluster')
    ax.legend()

    return cluster_colors  # restituisce la palette per allineare lo scatter


def analyze_and_cluster_universe(
    prices       : pd.DataFrame,
    n_clusters   : int  = 3,
    lookback_days: int  = 252,
    plot         : bool = True,
) -> dict:
    px  = prices.dropna(axis=1, how='all').ffill().iloc[-lookback_days:]
    ret = px.pct_change().dropna()

    metrics = {}
    for ticker in ret.columns:
        r = ret[ticker].dropna()
        if len(r) < 60:
            continue
        cagr     = (1 + r.mean()) ** 252 - 1
        vol      = r.std() * np.sqrt(252)
        sharpe   = cagr / vol if vol > 0 else 0
        mom_3m   = px[ticker].iloc[-1] / px[ticker].iloc[-63]  - 1
        mom_6m   = px[ticker].iloc[-1] / px[ticker].iloc[-126] - 1
        mom_12m  = px[ticker].iloc[-1] / px[ticker].iloc[0]    - 1
        autocorr = r.autocorr(lag=5)
        cum      = (1 + r).cumprod()
        dd       = ((cum - cum.cummax()) / cum.cummax()).min()
        metrics[ticker] = dict(cagr=cagr, vol=vol, sharpe=sharpe,
                               mom_3m=mom_3m, mom_6m=mom_6m,
                               mom_12m=mom_12m, autocorr=autocorr, max_dd=dd)

    metrics_df  = pd.DataFrame(metrics).T.dropna()
    corr_matrix = ret[metrics_df.index].corr()
    dist_corr   = np.sqrt(0.5 * (1 - corr_matrix))

    scaler    = StandardScaler()
    feat      = scaler.fit_transform(
                    metrics_df[['vol','mom_6m','autocorr','max_dd']]
                )
    feat_dist = pd.DataFrame(
        np.linalg.norm(feat[:,None] - feat[None,:], axis=2) / feat.shape[1],
        index=metrics_df.index, columns=metrics_df.index
    )
    combined  = 0.6 * dist_corr + 0.4 * feat_dist
    condensed = squareform(combined.values, checks=False)
    Z         = linkage(condensed, method='ward')
    labels_cl = fcluster(Z, t=n_clusters, criterion='maxclust')
    cluster_map = dict(zip(metrics_df.index, labels_cl))

    cluster_groups = {}
    for ticker, cid in cluster_map.items():
        cluster_groups.setdefault(cid, []).append(ticker)

    # Soglie adattive
    vol_series = metrics_df['vol']
    vol_p33    = vol_series.quantile(0.33)
    vol_p66    = vol_series.quantile(0.66)
    mom_median = metrics_df['mom_6m'].median()
    sharpe_med = metrics_df['sharpe'].median()

    print(f"\nSoglie adattive universo:")
    print(f"  Vol p33={vol_p33:.1%}  p66={vol_p66:.1%}")
    print(f"  Mom6m mediana={mom_median:.1%}")
    print(f"  Sharpe mediana={sharpe_med:.2f}")
        
    cluster_labels = {}
    for cid, tickers in cluster_groups.items():
        sub     = metrics_df.loc[tickers]
        avg_vol = sub['vol'].mean()
        avg_mom = sub['mom_6m'].mean()
        avg_sh  = sub['sharpe'].mean()
        avg_dd  = sub['max_dd'].mean()

        # 1° AVOID: momentum negativo + drawdown pesante + Sharpe negativo
        if avg_dd <= -0.40 or (avg_sh < 0 and avg_mom < 0):
            label = "AVOID"

        # 2° HIGH_MOMENTUM: alta vol E momentum positivo
        # (valutato PRIMA di DEFENSIVE per non confondere Sharpe alto con difensività)
        elif avg_vol >= vol_p66 and avg_mom >= mom_median:
            label = "HIGH_MOMENTUM"

        # 3° DEFENSIVE: bassa vol O (Sharpe alto E bassa vol relativa)
        elif avg_vol <= vol_p33:
            label = "DEFENSIVE"

        # 4° BALANCED: tutto il resto
        else:
            label = "BALANCED"

        cluster_labels[cid] = label

        print(f"\nCluster {cid} [{label}] — {len(tickers)} asset")
        print(f"  Tickers  : {tickers}")
        print(f"  Avg Vol  : {avg_vol:.1%}")
        print(f"  Avg Mom6m: {avg_mom:.1%}")
        print(f"  Avg MaxDD: {avg_dd:.1%}")
        print(f"  Sharpe   : {avg_sh:.2f}")

    # Fallback: se tutti i cluster hanno la stessa label forza differenziazione
    unique_labels = set(cluster_labels.values())
    if len(unique_labels) == 1:
        print(f"\n⚠️  Tutti i cluster hanno label '{list(unique_labels)[0]}'"
              f" — forzo differenziazione per Sharpe")
        sharpe_by_cluster = {
            cid: metrics_df.loc[tickers, 'sharpe'].mean()
            for cid, tickers in cluster_groups.items()
        }
        sorted_cids   = sorted(sharpe_by_cluster,
                               key=sharpe_by_cluster.get, reverse=True)
        forced_labels = ["DEFENSIVE", "BALANCED", "HIGH_MOMENTUM"]
        for i, cid in enumerate(sorted_cids):
            idx = min(i, len(forced_labels) - 1)
            cluster_labels[cid] = forced_labels[idx]
            print(f"  Cluster {cid} → {forced_labels[idx]} "
                  f"(Sharpe={sharpe_by_cluster[cid]:.2f})")

    if plot:
        palette = ['#1D9E75', '#378ADD', '#BA7517', '#9F77DD', '#D85A30']

        fig, axes = plt.subplots(1, 2, figsize=(16, 5))

        cluster_colors = plot_dendrogram_colored(
            Z          = Z,
            labels     = metrics_df.index.tolist(),
            n_clusters = n_clusters,
            ax         = axes[0],
            palette    = palette,
        )
        axes[0].set_title("Dendrogramma — Clustering Universo")

        offsets = [(6,6), (-6,6), (6,-10), (-6,-10), (10,0), (-10,0)]
        for cid, tickers in cluster_groups.items():
            sub   = metrics_df.loc[tickers]
            color = cluster_colors.get(cid, '#888888')
            axes[1].scatter(
                sub['vol'], sub['mom_6m'],
                label  = f"C{cid}: {cluster_labels[cid]}",
                color  = color,
                s      = 100,
                zorder = 5,
            )
            for i, t in enumerate(tickers):
                ox, oy = offsets[i % len(offsets)]
                axes[1].annotate(
                    t.replace('.MI','').replace('.DE','')
                     .replace('.PA','').replace('.MC',''),
                    xy         = (metrics_df.loc[t,'vol'],
                                  metrics_df.loc[t,'mom_6m']),
                    xytext     = (ox, oy),
                    textcoords = 'offset points',
                    fontsize   = 7,
                    ha         = 'left' if ox >= 0 else 'right',
                    va         = 'bottom' if oy >= 0 else 'top',
                )

        axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)
        axes[1].set_xlabel("Volatilità annualizzata")
        axes[1].set_ylabel("Momentum 6 mesi")
        axes[1].set_title("Scatter Vol vs Momentum — per Cluster")
        axes[1].legend()
        plt.tight_layout()
        plt.show()

    return dict(
        cluster_map    = cluster_map,
        cluster_groups = cluster_groups,
        metrics_df     = metrics_df,
        cluster_labels = cluster_labels,
    )


def build_cluster_grids(
    cluster_labels : dict,
    cluster_groups : dict,
    n_top_min      : int   = 2,
    n_top_fraction : tuple = (0.10, 0.20, 0.30),
) -> dict:

    def _build_n_top(n_assets: int) -> list[int]:
        candidates = sorted(set(
            max(n_top_min, round(n_assets * f))
            for f in n_top_fraction
        ))
        candidates = [v for v in candidates
                      if v <= max(n_top_min, int(n_assets * 0.40))]
        if len(candidates) < 2:
            candidates = [n_top_min, max(n_top_min + 1, candidates[-1])]
        return candidates

    grids = {}

    for cid, label in cluster_labels.items():
        tickers  = cluster_groups.get(cid, [])
        n_assets = len(tickers)
        n_top    = _build_n_top(n_assets)

        print(f"\nCluster {cid} [{label}] — {n_assets} asset → n_top: {n_top}")

        if label == "HIGH_MOMENTUM":
            grid = {
                "rebalance_frequency"      : ["ME"],
                "momentum_lookback_days"   : [20, 40, 60],
                "riskparity_lookback_days" : [20, 40],
                "n_top"                    : n_top,
                "use_acceleration"         : [True, False],
                "momentum_weight"          : [0.7, 1.0],
                "filter_ema"               : [True],
                "filter_volatility"        : [True, False],
                "filter_min_momentum"      : [True],
            }

        elif label == "DEFENSIVE":
            grid = {
                "rebalance_frequency"      : ["ME", "QE"],
                "momentum_lookback_days"   : [60, 120, 180],
                "riskparity_lookback_days" : [60, 120],
                "n_top"                    : n_top,
                "use_acceleration"         : [False],
                "momentum_weight"          : [0.5, 0.7],
                "filter_ema"               : [True, False],
                "filter_volatility"        : [True],
                "filter_min_momentum"      : [True, False],
            }

        elif label == "AVOID":
            # ✅ Griglia molto selettiva — entra solo con momentum genuino
            grid = {
                "rebalance_frequency"      : ["ME"],
                "momentum_lookback_days"   : [60, 120],
                "riskparity_lookback_days" : [60, 120],
                "n_top"                    : n_top,
                "use_acceleration"         : [False],
                "momentum_weight"          : [1.0],       # puro momentum
                "filter_ema"               : [True],      # tutti i filtri ON
                "filter_volatility"        : [True],
                "filter_min_momentum"      : [True],
            }

        else:  # BALANCED
            grid = {
                "rebalance_frequency"      : ["ME", "QE"],
                "momentum_lookback_days"   : [40, 60, 120],
                "riskparity_lookback_days" : [40, 60],
                "n_top"                    : n_top,
                "use_acceleration"         : [True, False],
                "momentum_weight"          : [0.7, 1.0],
                "filter_ema"               : [True, False],
                "filter_volatility"        : [True, False],
                "filter_min_momentum"      : [True, False],
            }

        grids[cid] = grid
        n_comb = int(np.prod([len(v) for v in grid.values()]))
        print(f"  Combinazioni totali: {n_comb}")

    return grids


def run_clustered_wfo(
    cluster_groups  : dict,
    cluster_grids   : dict,
    cluster_labels  : dict,
    stocks_data_raw : pd.DataFrame,
    wfo_kwargs      : dict,
) -> dict:
    results = {}
    for cid, tickers in cluster_groups.items():
        label = cluster_labels[cid]
        grid  = cluster_grids[cid]

        if label == "DEFENSIVE" and "XEON.MI" not in tickers:
            tickers = tickers + ["XEON.MI"]

        cluster_data = filter_stocks_data(stocks_data_raw, tickers)

        print(f"\n{'='*55}")
        print(f"WFO Cluster {cid} [{label}] — {len(tickers)} asset")
        print(f"Tickers: {tickers}")
        print(f"{'='*55}")

        try:
            summary_df = walk_forward_rotational(
                stocks_data            = cluster_data,
                param_grid             = grid,
                ratio                  = wfo_kwargs.get('ratio', 'sharpe'),
                metric                 = wfo_kwargs.get('metric', 'TestScore'),
                start_date             = wfo_kwargs.get('start_date'),
                end_date               = wfo_kwargs.get('end_date'),
                benchmark_data         = wfo_kwargs.get('benchmark_data'),
                n_jobs                 = wfo_kwargs.get('n_jobs', 1),
                backend                = wfo_kwargs.get('backend', 'loky'),
                plot                   = False,
                verbose                = wfo_kwargs.get('verbose', False),
                debug                  = False,
                force_next_year_params = wfo_kwargs.get('force_next_year_params', True),
            )
            results[cid] = dict(summary_df=summary_df, label=label, universe=tickers)
            print(f"✅ Cluster {cid} — WFO completata")
            print(f"   TrainScore medio: {summary_df['TrainScore'].mean():.4f}")
            print(f"   TestScore medio : {summary_df['TestScore'].mean():.4f}")

            # Display con short_map
            df_disp = summary_df.rename(columns=short_map)
            my_display(
                title=f"WFO Results Cluster {cid} [{label}]",
                data=df_disp
            )

        except Exception as e:
            print(f"❌ Cluster {cid} fallito: {e}")
            results[cid] = dict(summary_df=None, label=label, universe=tickers)

    return results


def compute_market_regime(
    prices        : pd.DataFrame,
    equity_tickers: list,
    ema_fast      : int   = 50,
    ema_slow      : int   = 200,
    vol_window    : int   = 20,
    vol_threshold : float = 0.25,
) -> pd.Series:
    eq_px    = prices[equity_tickers].dropna(axis=1, how='all').ffill()
    eq_index = eq_px.mean(axis=1)
    ema_f    = eq_index.ewm(span=ema_fast,  adjust=False).mean()
    ema_s    = eq_index.ewm(span=ema_slow,  adjust=False).mean()
    vol_roll = eq_index.pct_change().rolling(vol_window).std() * np.sqrt(252)
    regime   = ((ema_f > ema_s) & (vol_roll < vol_threshold)).astype(int)
    regime.name = "regime"
    print(f"Regime ON : {regime.mean():.1%} del tempo")
    print(f"Regime OFF: {(1 - regime.mean()):.1%} del tempo")
    return regime


def aggregate_cluster_portfolios(
    wfo_results    : dict,
    stocks_data    : pd.DataFrame,
    benchmark_data,
    regime         : pd.Series,
    weight_on      : dict  = None,
    weight_off     : dict  = None,
    start_date     : str   = None,
    end_date       : str   = None,
    init_cash      : float = 100_000,
    plot           : bool  = True,
) -> dict:
    if weight_on is None:
        weight_on  = {"HIGH_MOMENTUM": 0.60, "BALANCED": 0.30, "DEFENSIVE": 0.10}
    if weight_off is None:
        weight_off = {"HIGH_MOMENTUM": 0.10, "BALANCED": 0.20, "DEFENSIVE": 0.70}

    cluster_returns = {}
    cluster_pf      = {}

    for cid, res in wfo_results.items():
        if res['summary_df'] is None:
            print(f"⚠️  Cluster {cid} senza summary_df, skip")
            continue
        label        = res['label']
        cluster_data = filter_stocks_data(stocks_data, res['universe'])

        print(f"\nCostruzione portafoglio Cluster {cid} [{label}]...")
        try:
            pf_rot, pf_bh, selections = build_portfolio_from_wfo_summary(
                summary_df      = res['summary_df'],
                stocks_data     = cluster_data,
                benchmark_data  = benchmark_data,
                benchmark_title = f"Cluster {cid} BM",
                init_cash       = init_cash,
                start_date      = start_date,
                end_date        = end_date,
                plot            = False,
                show_report     = False,
                portfolio_name  = f"Cluster {cid} [{label}]",
            )
            eq = pf_rot.value()
            if isinstance(eq, pd.DataFrame):
                eq = eq.sum(axis=1)
            eq_norm = eq / eq.iloc[0]
            ret     = eq_norm.pct_change().fillna(0)

            cluster_returns[label] = ret
            cluster_pf[cid]        = pf_rot
            print(f"✅ Cluster {cid} [{label}] costruito")

        except Exception as e:
            print(f"❌ Cluster {cid} fallito: {e}")

    if not cluster_returns:
        raise ValueError("Nessun cluster costruito correttamente")

    ret_df   = pd.DataFrame(cluster_returns).ffill().dropna()
    regime_a = regime.reindex(ret_df.index).ffill().fillna(0)

    w_rows = []
    for date in ret_df.index:
        r   = int(regime_a.loc[date])
        w   = weight_on if r == 1 else weight_off
        w_rows.append({label: w.get(label, 0.0) for label in ret_df.columns})

    w_df     = pd.DataFrame(w_rows, index=ret_df.index)
    w_df     = w_df.div(w_df.sum(axis=1), axis=0)
    agg_ret  = (ret_df * w_df).sum(axis=1)
    agg_eq   = (1 + agg_ret).cumprod() * init_cash

    n_years  = len(agg_ret) / 252
    cagr     = (agg_eq.iloc[-1] / agg_eq.iloc[0]) ** (1 / n_years) - 1
    vol      = agg_ret.std() * np.sqrt(252)
    sharpe   = cagr / vol if vol > 0 else 0
    dd       = ((agg_eq / agg_eq.cummax()) - 1).min()
    calmar   = cagr / abs(dd) if dd != 0 else 0

    print(f"\n{'='*55}")
    print(f"PORTAFOGLIO AGGREGATO CLUSTERED")
    print(f"{'='*55}")
    print(f"CAGR      : {cagr:.2%}")
    print(f"Volatilità: {vol:.2%}")
    print(f"Sharpe    : {sharpe:.2f}")
    print(f"Max DD    : {dd:.2%}")
    print(f"Calmar    : {calmar:.2f}")
    print(f"Regime ON : {regime_a.mean():.1%} del tempo")

    if plot:
        fig, axes = plt.subplots(2, 1, figsize=(14, 8),
                                  gridspec_kw={'height_ratios': [3, 1]})
        for label, ret_s in cluster_returns.items():
            eq_c = (1 + ret_s.reindex(ret_df.index).fillna(0)).cumprod() * 100
            axes[0].plot(eq_c, alpha=0.5, linestyle='--', label=f"Cluster {label}")
        agg_plot = (1 + agg_ret).cumprod() * 100
        axes[0].plot(agg_plot, color='navy', linewidth=2.5, label="Aggregato")
        axes[0].set_title("Portafoglio Clustered Aggregato vs Cluster singoli")
        axes[0].legend()
        axes[0].grid(alpha=0.3)

        axes[1].fill_between(regime_a.index, regime_a,
                              alpha=0.4, color='green', label='Risk ON')
        axes[1].fill_between(regime_a.index, 1 - regime_a,
                              alpha=0.4, color='red',   label='Risk OFF')
        axes[1].set_title("Regime Risk ON / OFF")
        axes[1].legend()
        axes[1].grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

    return dict(
        equity          = agg_eq,
        returns         = agg_ret,
        weights         = w_df,
        regime          = regime_a,
        cluster_pf      = cluster_pf,
        cluster_returns = cluster_returns,
        metrics         = dict(cagr=cagr, vol=vol, sharpe=sharpe,
                               max_dd=dd, calmar=calmar),
    )


# cluster_grids = build_cluster_grids(cluster_result['cluster_labels'])


def merge_cluster_summary_dfs(
    wfo_results    : dict,
    cluster_labels : dict,
    regime         : pd.Series,
    dominant_on    : str = "HIGH_MOMENTUM",
    dominant_off   : str = "DEFENSIVE",
) -> pd.DataFrame:
    """
    Produce un summary_df aggregato compatibile con
    build_rotational_portfolios_from_wfo_result.

    Per ogni finestra WFO:
      - se il regime medio nella finestra è ON  → parametri da dominant_on
      - se il regime medio nella finestra è OFF → parametri da dominant_off

    Parameters
    ----------
    wfo_results    : output di run_clustered_wfo
    cluster_labels : {cluster_id: label}
    regime         : pd.Series 0/1 da compute_market_regime
    dominant_on    : label cluster da usare in Risk ON
    dominant_off   : label cluster da usare in Risk OFF

    Returns
    -------
    summary_df compatibile con build_rotational_portfolios_from_wfo_result
    """

    # Mappa label → summary_df
    label_to_summary = {}
    for cid, res in wfo_results.items():
        if res['summary_df'] is not None:
            label = cluster_labels[cid]
            label_to_summary[label] = res['summary_df']

    # Verifica che i cluster dominanti esistano
    for dominant in [dominant_on, dominant_off]:
        if dominant not in label_to_summary:
            available = list(label_to_summary.keys())
            print(f"⚠️  Cluster '{dominant}' non trovato, "
                  f"disponibili: {available}")
            # Fallback: usa BALANCED se disponibile, altrimenti il primo
            fallback = "BALANCED" if "BALANCED" in available else available[0]
            if dominant == dominant_on:
                dominant_on  = fallback
            else:
                dominant_off = fallback
            print(f"   → Fallback su '{fallback}'")

    # Prendi tutte le finestre disponibili (usa il summary più lungo)
    all_windows = set()
    for df in label_to_summary.values():
        all_windows.update(df.index.tolist())
    all_windows = sorted(all_windows)

    rows = []
    for window in all_windows:
        # Estrai date dalla finestra (es. "2022-01-01 →2022-12-31")
        # Il regime viene valutato sulla data di fine finestra
        try:
            # Parsing della data di fine finestra dall'index
            if isinstance(window, tuple):
                end_date_win = pd.Timestamp(window[1])
            elif isinstance(window, str) and '→' in window:
                end_date_win = pd.Timestamp(window.split('→')[1].strip())
            else:
                end_date_win = pd.Timestamp(window)
        except Exception:
            end_date_win = None

        # Determina regime dominante nella finestra
        if end_date_win is not None and end_date_win in regime.index:
            regime_val = int(regime.loc[end_date_win])
        elif end_date_win is not None:
            # Prendi il valore più vicino
            idx_pos    = regime.index.get_indexer([end_date_win],
                                                   method='nearest')[0]
            regime_val = int(regime.iloc[idx_pos])
        else:
            regime_val = 1  # default ON

        # Seleziona cluster dominante
        dominant = dominant_on if regime_val == 1 else dominant_off
        src_df   = label_to_summary[dominant]

        # Prendi riga corrispondente alla finestra
        if window in src_df.index:
            row = src_df.loc[window].copy()
        else:
            # Finestra non presente nel cluster dominante → usa l'altro
            fallback_label = dominant_off if dominant == dominant_on \
                             else dominant_on
            fallback_df    = label_to_summary.get(fallback_label)
            if fallback_df is not None and window in fallback_df.index:
                row = fallback_df.loc[window].copy()
            else:
                # Ultima riga disponibile come fallback finale
                row = src_df.iloc[-1].copy()

        row['_cluster_used'] = dominant   # colonna debug, non impatta il motore
        row['_regime']       = regime_val
        rows.append((window, row))

    # Ricostruisce DataFrame con stesso formato di summary_df originale
    merged = pd.DataFrame(
        [r for _, r in rows],
        index=[w for w, _ in rows]
    )
    merged.index.name = "Window"

    # Stampa riepilogo
    n_on  = (merged['_regime'] == 1).sum()
    n_off = (merged['_regime'] == 0).sum()
    print(f"\n{'='*50}")
    print(f"SUMMARY DF AGGREGATO")
    print(f"{'='*50}")
    print(f"Finestre totali : {len(merged)}")
    print(f"Risk ON  → {dominant_on:<16} : {n_on} finestre")
    print(f"Risk OFF → {dominant_off:<16} : {n_off} finestre")

    by_cluster = merged['_cluster_used'].value_counts()
    for label, count in by_cluster.items():
        print(f"  {label}: {count} finestre ({count/len(merged):.0%})")

    return merged


def get_clean_summary_df(merged: pd.DataFrame) -> pd.DataFrame:
    """
    Rimuove le colonne di debug prima di passare a
    build_rotational_portfolios_from_wfo_result.
    """
    return merged.drop(
        columns=[c for c in ['_cluster_used', '_regime']
                 if c in merged.columns]
    )

    

In [ ]:
print("Libreria r_functions importata.")